<a href="https://colab.research.google.com/github/harshalichowdhary-design/NyayaRAG/blob/main/04_NyayaRAG_Pilot_Evaluation_and_Pipeline_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NYAYaRAG: Pilot Evaluation and Pipeline Validation**

## **Retrieval, Evidence-Grounded Generation, and Abstention Testing**

This notebook conducts a preliminary pilot evaluation of the NYAYaRAG framework using a reserved set of 10 legal research questions.

The pilot evaluation is intended to validate the complete evaluation workflow before applying it to the final benchmark dataset. It does not constitute the final performance evaluation of NYAYaRAG and is not used for inferential statistical testing.

The pilot study evaluates the operational readiness of the following components:

- lexical retrieval using TF-IDF and BM25;
- semantic retrieval using E5, BGE, and InLegalBERT;
- retrieval-output storage and ranking;
- manual relevance annotation;
- computation of retrieval evaluation metrics;
- evidence-grounded answer generation;
- citation and faithfulness assessment; and
- abstention behaviour where the retrieved corpus provides insufficient evidence.

The validated pipeline developed in this notebook will subsequently be applied to the final benchmark dataset for the full experimental evaluation.

## **1. Objectives of the Pilot Evaluation**

The pilot evaluation is conducted to determine whether the proposed NYAYaRAG evaluation procedure is technically functional, internally consistent, and suitable for full-scale implementation.

The specific objectives are to:

1. verify that the pilot benchmark can be loaded and processed correctly;
2. confirm that each retrieval model returns ranked results in a consistent format;
3. inspect whether retrieved chunks and judgment metadata are sufficient for relevance assessment;
4. validate the manual annotation structure used to identify relevant judgments and chunks;
5. test the implementation of Precision@k, Recall@k, Hit@k, Mean Reciprocal Rank, and nDCG@k;
6. assess whether the generation prompt produces answers grounded only in retrieved evidence;
7. evaluate whether citations can be traced to the retrieved judgments; and
8. test whether the system appropriately abstains when sufficient evidence is unavailable.

Any implementation or annotation issues identified during the pilot will be corrected before the final benchmark evaluation.

## **2. Scope and Limitations of the Pilot**

The pilot benchmark consists of 10 legal research questions selected from the expert-contributed question pool. The questions were chosen to represent different legal domains and different forms of legal information need.

The pilot results are interpreted only as evidence of pipeline feasibility. Because of the small and purposively selected sample:

- the results are not treated as final model-performance estimates;
- no inferential statistical conclusions are drawn;
- the pilot questions are excluded from the final evaluation benchmark; and
- retrieval settings, prompts, annotation instructions, and output formats may be refined after examining the pilot results.

Accordingly, the pilot serves as a methodological validation stage rather than a confirmatory experiment.

## **3. Pilot Evaluation Workflow**

The pilot evaluation follows the sequence below:

**Pilot Questions**  
↓  
**Load Corpus Chunks, Indexes, and Embeddings**  
↓  
**Run TF-IDF, BM25, E5, BGE, and InLegalBERT Retrieval**  
↓  
**Store Top-k Ranked Chunks and Judgment Metadata**  
↓  
**Conduct Manual Relevance Annotation**  
↓  
**Compute Retrieval Metrics**  
↓  
**Test Evidence-Grounded Answer Generation**  
↓  
**Assess Citation Grounding, Faithfulness, and Completeness**  
↓  
**Test Abstention Behaviour**  
↓  
**Document Pipeline Refinements**

The same validated workflow will later be applied to the final benchmark, with the pilot question file replaced by the finalized evaluation dataset.

## **4. Reproducibility and Data Management**

To preserve reproducibility, the notebook loads all project resources from predefined Google Drive paths and saves pilot outputs in dedicated evaluation directories.

The following principles are followed:

- the original pilot-question file is not modified;
- retrieval outputs are saved separately for each model;
- manual annotations are retained as independent files;
- generated answers and evaluation scores are stored separately;
- fixed retrieval parameters are documented before execution; and
- any changes made after pilot inspection are recorded explicitly.

This separation preserves an audit trail from the original pilot questions to the final pilot-evaluation outputs.

# **Part A — Environment and Path Configuration**

## **1. Environment Setup**

This section imports the Python libraries required for the pilot evaluation and defines the project directory structure used throughout the notebook.

The pilot evaluation uses the benchmark dataset constructed in Notebook 3 together with the retrieval models and indexes generated during corpus preparation. Centralising all directory paths at the beginning of the notebook improves reproducibility and simplifies future modifications to the project structure.

The environment configuration also ensures that the notebook can be executed consistently on Google Colab without requiring changes to subsequent code cells.

## **1.1 Mount Google Drive**

The NYAYaRAG project, including the benchmark dataset, processed corpus, retrieval indexes, and embedding models, is stored in Google Drive. Accordingly, Google Drive is mounted to provide access to all project resources required for the pilot evaluation.

In [ ]:


from google.colab import drive
drive.mount('/content/drive')

## **1.2 Import Required Libraries**

The pilot evaluation requires libraries for file management, numerical computation, data processing, retrieval, and evaluation. The required libraries are imported below.

In [1]:
from pathlib import Path
import os
import json
import pickle

import numpy as np
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


## **1.3 Configure Project Paths**

To improve reproducibility and avoid hard-coded file locations, all project directories are defined using `pathlib.Path`. Centralising path definitions ensures that the remainder of the notebook can access benchmark datasets, processed corpus files, retrieval models, and evaluation outputs through a consistent directory structure.

In [2]:
from pathlib import Path

PROJECT_DIR = Path(r"D:\_ResearchNotebooks\NyayaRAG")

print("=" * 60)
print("PROJECT DIRECTORY")
print("=" * 60)

for item in sorted(PROJECT_DIR.iterdir()):
    print(item.name)

PROJECT DIRECTORY
.env
.env.example
.git
.gitignore
24-09-2026.pdf
__pycache__
data
models
notebook_1
notebook_2
notebook_3_literatureReview.md
notebooks
nyayarag_reconstructed_questions.csv
prompt.md
README.md
requirements.txt
Screenshot 2026-10-12 221157.png
venv


## **1.4 Define Project Directories**

The NyayaRAG project is organised into dedicated directories for benchmark data, processed corpus files, embeddings, retrieval models, evaluation outputs, and generated results. The paths below provide a single point of reference for all resources used in the pilot evaluation.

In [3]:
PROJECT_DIR = Path(r"D:\_ResearchNotebooks\NyayaRAG")

BENCHMARK_DIR = PROJECT_DIR / "benchmark"
DATA_DIR = PROJECT_DIR / "data"
EMBEDDINGS_DIR = PROJECT_DIR / "embeddings"
MODELS_DIR = PROJECT_DIR / "models"
EVALUATION_DIR = PROJECT_DIR / "evaluation"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
RESULTS_DIR = PROJECT_DIR / "results"

PILOT_BENCHMARK_DIR = BENCHMARK_DIR / "final"

PILOT_EVALUATION_DIR = EVALUATION_DIR / "pilot"
PILOT_RETRIEVAL_DIR = PILOT_EVALUATION_DIR / "retrieval"
PILOT_ANNOTATION_DIR = PILOT_EVALUATION_DIR / "annotation"
PILOT_GENERATION_DIR = PILOT_EVALUATION_DIR / "generation"
PILOT_RESULTS_DIR = PILOT_EVALUATION_DIR / "results"

for directory in [
    PILOT_EVALUATION_DIR,
    PILOT_RETRIEVAL_DIR,
    PILOT_ANNOTATION_DIR,
    PILOT_GENERATION_DIR,
    PILOT_RESULTS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directories configured successfully.")

Project directories configured successfully.


## **1.5 Validate Project Directories**

Before loading the pilot benchmark and retrieval resources, the notebook verifies that the required project directories exist and are accessible. This validation helps identify incorrect paths or missing resources before the evaluation pipeline is executed.

In [4]:
directories_to_check = {
    "Project Directory": PROJECT_DIR,
    "Benchmark Directory": BENCHMARK_DIR,
    "Data Directory": DATA_DIR,
    "Embeddings Directory": EMBEDDINGS_DIR,
    "Models Directory": MODELS_DIR,
    "Evaluation Directory": EVALUATION_DIR
}

for name, path in directories_to_check.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:<25}: {status} -> {path}")

Project Directory        : FOUND -> D:\_ResearchNotebooks\NyayaRAG
Benchmark Directory      : MISSING -> D:\_ResearchNotebooks\NyayaRAG\benchmark
Data Directory           : FOUND -> D:\_ResearchNotebooks\NyayaRAG\data
Embeddings Directory     : MISSING -> D:\_ResearchNotebooks\NyayaRAG\embeddings
Models Directory         : FOUND -> D:\_ResearchNotebooks\NyayaRAG\models
Evaluation Directory     : FOUND -> D:\_ResearchNotebooks\NyayaRAG\evaluation


# **Part B — Pilot Benchmark Dataset**


## **2.1 Load the Pilot Benchmark**

The pilot benchmark comprises ten legal research questions selected from the benchmark dataset constructed in Notebook 3.

The selected questions span multiple legal domains and are used exclusively for validating the retrieval pipeline, evaluation metrics, annotation framework, and evidence-grounded answer generation workflow. The pilot benchmark is not used for the final statistical evaluation of NYAYaRAG.

## **2.2 Load the Pilot Questions**

The pilot question dataset is loaded from the benchmark directory and inspected to verify its structure and completeness before retrieval is performed.

In [6]:
pilot_file = BENCHMARK_DIR / "final" / "pilot_questions.csv"

df_pilot = pd.read_csv(pilot_file)

print("=" * 60)
print("PILOT BENCHMARK DATASET")
print("=" * 60)

print(f"Total Pilot Questions : {len(df_pilot)}")
print(f"Number of Columns     : {len(df_pilot.columns)}")

display(df_pilot)

PILOT BENCHMARK DATASET
Total Pilot Questions : 10
Number of Columns     : 6


,Question_ID,Contributor,Experience,Question_No,Question,Word_Count
0,RAW001,TEST01,Judicial Officer – 11 to 20 years\t,1,Under what circumstances can delay in lodging ...,13
1,RAW002,TEST01,Judicial Officer – 11 to 20 years\t,2,Can a conviction be sustained solely on the te...,13
2,RAW004,TEST01,Judicial Officer – 11 to 20 years\t,4,When may a court interfere with an arbitral aw...,15
3,RAW005,TEST01,Judicial Officer – 11 to 20 years\t,5,What are the conditions for granting anticipat...,11
4,RAW006,Dr. Mukesh Saini,Judicial Officer – 11 to 20 years\t,1,Under what circumstances can a court permit am...,26
5,RAW007,Dr. Mukesh Saini,Judicial Officer – 11 to 20 years\t,2,What are the essential ingredients required to...,28
6,RAW010,Dr. Mukesh Saini,Judicial Officer – 11 to 20 years\t,5,What are the mandatory preconditions for exerc...,34
7,RAW012,Insan,Judicial Officer – 11 to 20 years\t,2,Can a civil suit for damages on account of mur...,27
8,RAW015,Insan,Judicial Officer – 11 to 20 years\t,5,How can a magistrate certify the correctness o...,37
9,RAW017,RC,Judicial Officer – Less than 5 years,2,Can an authorised signatory be convicted under...,22


## **2.3 Validate the Pilot Benchmark**

The pilot benchmark is validated by examining the available columns, checking for missing values, and confirming that each question possesses a unique identifier. Validation at this stage ensures that the benchmark is suitable for retrieval experiments.

In [7]:
print("=" * 60)
print("COLUMN NAMES")
print("=" * 60)

for column in df_pilot.columns:
    print(column)

print("\n")

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

display(df_pilot.isnull().sum())

COLUMN NAMES
Question_ID
Contributor
Experience
Question_No
Question
Word_Count


MISSING VALUES


Question_ID    0
Contributor    0
Experience     0
Question_No    0
Question       0
Word_Count     0
dtype: int64

## **2.4 Confirm Identifier Uniqueness and Pilot Size**

The pilot dataset is further validated by confirming that each question identifier is unique and that the dataset contains the intended number of pilot questions.

In [8]:
expected_pilot_size = 10

unique_ids = df_pilot["Question_ID"].nunique()
actual_size = len(df_pilot)

print("=" * 60)
print("PILOT DATASET VALIDATION")
print("=" * 60)

print(f"Expected Pilot Questions : {expected_pilot_size}")
print(f"Actual Pilot Questions   : {actual_size}")
print(f"Unique Question IDs      : {unique_ids}")

assert actual_size == expected_pilot_size, (
    f"Expected {expected_pilot_size} pilot questions, "
    f"but found {actual_size}."
)

assert unique_ids == actual_size, (
    "Duplicate Question_ID values were detected."
)

assert df_pilot["Question"].str.strip().ne("").all(), (
    "One or more pilot questions are blank."
)

print("\nPilot benchmark validation completed successfully.")

PILOT DATASET VALIDATION
Expected Pilot Questions : 10
Actual Pilot Questions   : 10
Unique Question IDs      : 10

Pilot benchmark validation completed successfully.


### **Observation**

The pilot benchmark contains ten complete legal research questions with unique identifiers and no missing values. The dataset is therefore suitable for use in the pilot retrieval and evaluation pipeline.

# **Part C — Retrieval Resources and Model Validation**

## **3.1 Inspect Available Retrieval Resources**

Before loading the retrieval models, the contents of the model and embedding directories are inspected. This step identifies the exact files and subdirectories produced during corpus preparation and prevents assumptions regarding file names or storage formats.

In [9]:
def list_directory_tree(directory, max_depth=3):
    """
    Display the directory tree up to a specified depth.
    """
    directory = Path(directory)

    if not directory.exists():
        print(f"Directory not found: {directory}")
        return

    print(f"\n{directory}")

    for path in sorted(directory.rglob("*")):
        relative_path = path.relative_to(directory)

        if len(relative_path.parts) <= max_depth:
            indent = "    " * (len(relative_path.parts) - 1)
            marker = "[DIR]" if path.is_dir() else "[FILE]"
            print(f"{indent}{marker} {relative_path.name}")


print("=" * 60)
print("MODELS DIRECTORY")
print("=" * 60)
list_directory_tree(MODELS_DIR, max_depth=3)

print("\n" + "=" * 60)
print("EMBEDDINGS DIRECTORY")
print("=" * 60)
list_directory_tree(EMBEDDINGS_DIR, max_depth=3)

print("\n" + "=" * 60)
print("PROCESSED DATA DIRECTORY")
print("=" * 60)
list_directory_tree(DATA_DIR / "processed", max_depth=3)

MODELS DIRECTORY

D:\_ResearchNotebooks\NyayaRAG\models
[DIR] bge
    [FILE] bge_config.json
    [FILE] bge_embeddings.npy
    [DIR] chromadb
        [DIR] 0235d6ef-0b6f-43ad-96d0-655ab408daa1
        [FILE] chroma.sqlite3
[DIR] bm25
    [FILE] bm25.pkl
    [FILE] tokenized_corpus.pkl
[DIR] e5
    [DIR] chromadb
        [DIR] 4653a31d-d5c8-482b-95bb-83e313133bb0
        [FILE] chroma.sqlite3
    [FILE] e5_config.json
    [FILE] e5_embeddings.npy
[DIR] inlegalbert
    [DIR] chromadb
        [DIR] 0d254658-5528-43a1-b59a-e57645ab1245
        [FILE] chroma.sqlite3
    [FILE] inlegalbert_config.json
    [FILE] inlegalbert_embeddings.npy
[FILE] retrieval_index_metadata.json
[DIR] tfidf
    [FILE] tfidf_matrix.npz
    [FILE] tfidf_vectorizer.pkl

EMBEDDINGS DIRECTORY
Directory not found: D:\_ResearchNotebooks\NyayaRAG\embeddings

PROCESSED DATA DIRECTORY

D:\_ResearchNotebooks\NyayaRAG\data\processed
[FILE] chunks_384_40.parquet
[FILE] extraction_log.csv
[DIR] text
    [DIR] year=2024
      

## **3.2 Define Retrieval Resource Paths**

The pilot evaluation requires the processed chunk dataset, lexical retrieval artefacts, dense embedding matrices, model configuration files, and persistent ChromaDB collections generated during corpus preparation.

The exact resource paths are defined below. Dense embedding arrays are loaded from the model-specific directories because these contain the validated embeddings corresponding to the processed chunk dataset.

In [10]:
# Processed corpus resources
CHUNKS_FILE = DATA_DIR / "processed" / "chunks_384_40.parquet"
CLEAN_CORPUS_FILE = DATA_DIR / "processed" / "clean_corpus.parquet"

# TF-IDF resources
TFIDF_DIR = MODELS_DIR / "tfidf"
TFIDF_VECTORIZER_FILE = TFIDF_DIR / "tfidf_vectorizer.pkl"
TFIDF_MATRIX_FILE = TFIDF_DIR / "tfidf_matrix.npz"

# BM25 resources
BM25_DIR = MODELS_DIR / "bm25"
BM25_MODEL_FILE = BM25_DIR / "bm25.pkl"
BM25_TOKENIZED_CORPUS_FILE = BM25_DIR / "tokenized_corpus.pkl"

# BGE resources
BGE_DIR = MODELS_DIR / "bge"
BGE_EMBEDDINGS_FILE = BGE_DIR / "bge_embeddings.npy"
BGE_CONFIG_FILE = BGE_DIR / "bge_config.json"
BGE_CHROMA_DIR = BGE_DIR / "chromadb"

# E5 resources
E5_DIR = MODELS_DIR / "e5"
E5_EMBEDDINGS_FILE = E5_DIR / "e5_embeddings.npy"
E5_CONFIG_FILE = E5_DIR / "e5_config.json"
E5_CHROMA_DIR = E5_DIR / "chromadb"

# InLegalBERT resources
INLEGALBERT_DIR = MODELS_DIR / "inlegalbert"
INLEGALBERT_EMBEDDINGS_FILE = (
    INLEGALBERT_DIR / "inlegalbert_embeddings.npy"
)
INLEGALBERT_CONFIG_FILE = (
    INLEGALBERT_DIR / "inlegalbert_config.json"
)
INLEGALBERT_CHROMA_DIR = INLEGALBERT_DIR / "chromadb"

# Shared retrieval metadata
RETRIEVAL_METADATA_FILE = MODELS_DIR / "retrieval_index_metadata.json"

print("Retrieval resource paths defined successfully.")

Retrieval resource paths defined successfully.


## **3.3 Validate Required Retrieval Resources**

Before loading any retrieval model, each required file and directory is checked for availability. The evaluation pipeline should not proceed if a required corpus, index, embedding matrix, or configuration file is missing.

In [12]:
required_resources = {
    "Processed chunks": CHUNKS_FILE,
    "Clean corpus": CLEAN_CORPUS_FILE,

    "TF-IDF vectorizer": TFIDF_VECTORIZER_FILE,
    "TF-IDF matrix": TFIDF_MATRIX_FILE,

    "BM25 model": BM25_MODEL_FILE,
    "BM25 tokenized corpus": BM25_TOKENIZED_CORPUS_FILE,

    "BGE embeddings": BGE_EMBEDDINGS_FILE,
    "BGE configuration": BGE_CONFIG_FILE,
    "BGE ChromaDB": BGE_CHROMA_DIR,

    "E5 embeddings": E5_EMBEDDINGS_FILE,
    "E5 configuration": E5_CONFIG_FILE,
    "E5 ChromaDB": E5_CHROMA_DIR,

    "InLegalBERT embeddings": INLEGALBERT_EMBEDDINGS_FILE,
    "InLegalBERT configuration": INLEGALBERT_CONFIG_FILE,
    "InLegalBERT ChromaDB": INLEGALBERT_CHROMA_DIR,

    "Retrieval metadata": RETRIEVAL_METADATA_FILE
}

missing_resources = []

print("=" * 85)
print("RETRIEVAL RESOURCE VALIDATION")
print("=" * 85)

for resource_name, resource_path in required_resources.items():
    exists = resource_path.exists()
    status = "FOUND" if exists else "MISSING"

    print(
        f"{resource_name:<30}: "
        f"{status:<8} -> {resource_path}"
    )

    if not exists:
        missing_resources.append(resource_name)

if missing_resources:
    raise FileNotFoundError(
        "Missing required resources: "
        + ", ".join(missing_resources)
    )

print("\nAll required retrieval resources were located successfully.")

RETRIEVAL RESOURCE VALIDATION
Processed chunks              : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\data\processed\chunks_384_40.parquet
Clean corpus                  : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\data\processed\clean_corpus.parquet
TF-IDF vectorizer             : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\models\tfidf\tfidf_vectorizer.pkl
TF-IDF matrix                 : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\models\tfidf\tfidf_matrix.npz
BM25 model                    : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\models\bm25\bm25.pkl
BM25 tokenized corpus         : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\models\bm25\tokenized_corpus.pkl
BGE embeddings                : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\models\bge\bge_embeddings.npy
BGE configuration             : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\models\bge\bge_config.json
BGE ChromaDB                  : FOUND    -> D:\_ResearchNotebooks\NyayaRAG\models\bge\chromadb
E5 embeddings              

## **3.4 Load and Validate the Processed Chunk Dataset**

The processed chunk dataset provides the textual passages and metadata corresponding to the lexical indexes and dense embedding matrices. Its row order must remain aligned with the saved retrieval artefacts.

The dataset is loaded and examined for shape, column structure, missing values, and identifier uniqueness before any retrieval experiment is conducted.

In [13]:
df_chunks = pd.read_parquet(CHUNKS_FILE)

print("=" * 60)
print("PROCESSED CHUNK DATASET")
print("=" * 60)

print(f"Number of Chunks : {len(df_chunks):,}")
print(f"Number of Columns: {len(df_chunks.columns)}")

print("\nColumns:")
for column in df_chunks.columns:
    print(f"- {column}")

display(df_chunks.head())

PROCESSED CHUNK DATASET
Number of Chunks : 30,521
Number of Columns: 8

Columns:
- global_chunk_id
- judgment_id
- case_title
- local_chunk_id
- configuration
- chunk_text
- word_count
- paragraph_count


,global_chunk_id,judgment_id,case_title,local_chunk_id,configuration,chunk_text,word_count,paragraph_count
0,1,[2024] 10 S.C.R. 108,VIJAY SINGH @ VIJAY KR. SHARMA versus THE STAT...,1,chunk_384_40,Judgment / Order of the Supreme Court Judgment...,396,3
1,2,[2024] 10 S.C.R. 108,VIJAY SINGH @ VIJAY KR. SHARMA versus THE STAT...,2,chunk_384_40,furtherance of the written report lodged by th...,384,3
2,3,[2024] 10 S.C.R. 108,VIJAY SINGH @ VIJAY KR. SHARMA versus THE STAT...,3,chunk_384_40,1 Prosecution witness or PW 2 Hereinafter refe...,421,3
3,4,[2024] 10 S.C.R. 108,VIJAY SINGH @ VIJAY KR. SHARMA versus THE STAT...,4,chunk_384_40,"BRIEF FACTS 4. Shorn of unnecessary details, t...",387,2
4,5,[2024] 10 S.C.R. 108,VIJAY SINGH @ VIJAY KR. SHARMA versus THE STAT...,5,chunk_384_40,also explained the\n\n[2024] 10 S.C.R. 113 Vij...,402,3


# **Part D — Loading and Validating Retrieval Models**

This section loads and validates each retrieval model together with its associated retrieval resources required for the pilot evaluation.

NYAYaRAG incorporates two categories of retrieval models:

- **Lexical retrieval models**, namely Term Frequency–Inverse Document Frequency (TF-IDF) and BM25, which retrieve documents based on lexical term matching.
- **Dense semantic retrieval models**, namely E5, BGE, and InLegalBERT, which retrieve semantically similar text passages using dense vector representations.

To facilitate systematic validation, each retrieval model is loaded and evaluated independently before proceeding to comparative experiments. This incremental approach enables verification of model configuration, retrieval outputs, ranking behaviour, and integration with the processed corpus while simplifying the identification and correction of implementation errors.

For each retrieval model, the following validation procedure is adopted:

1. Load the retrieval model together with all associated artefacts.
2. Verify successful loading of the model and its supporting resources.
3. Execute retrieval for an initial pilot question to confirm correct operation.
4. Inspect the ranked retrieval results manually.
5. Validate the retrieved chunk identifiers, judgment metadata, and retrieved text.
6. Upon successful validation, execute retrieval for all ten pilot questions.
7. Store the retrieved results for subsequent relevance annotation and evaluation.

The same retrieval pipeline validated during the pilot evaluation will subsequently be applied, without methodological modification, to the finalized benchmark dataset used for the full experimental evaluation of NYAYaRAG.

## **4.1 Retrieval Configuration**

To ensure a fair and reproducible comparison across retrieval models, a common retrieval configuration is adopted throughout the pilot evaluation.

The same retrieval depth is used for every retrieval model, ensuring that retrieval performance is evaluated under identical experimental conditions.

For the pilot evaluation:

- Top-**5** ranked chunks are retrieved for each legal research question.
- The same retrieval depth is maintained across all lexical and semantic retrieval models.
- Retrieval outputs are stored for subsequent manual relevance annotation and computation of retrieval evaluation metrics.
- During manual inspection, only the highest-ranked retrieval results are displayed for readability, while all retrieved chunks are preserved for evaluation.

The retrieval configuration defined in this section will remain unchanged during the full benchmark evaluation to ensure methodological consistency.

In [14]:
# ============================================================
# Retrieval Configuration
# ============================================================

TOP_K = 5
DISPLAY_K = 3

print("=" * 60)
print("RETRIEVAL CONFIGURATION")
print("=" * 60)

print(f"Top-k Retrieval        : {TOP_K}")
print(f"Display Top Results    : {DISPLAY_K}")

assert DISPLAY_K <= TOP_K

print("\nRetrieval configuration validated successfully.")

RETRIEVAL CONFIGURATION
Top-k Retrieval        : 5
Display Top Results    : 3

Retrieval configuration validated successfully.


# **Part E — TF-IDF Retrieval Validation**

Term Frequency–Inverse Document Frequency (TF-IDF) serves as the lexical baseline for evaluating retrieval performance.

Unlike dense semantic retrieval models, TF-IDF represents each document as a sparse vector of weighted terms and ranks documents according to lexical similarity with the query.

The objective of this section is to validate that the TF-IDF retrieval pipeline has been correctly configured before executing retrieval experiments on the complete pilot benchmark.

Validation is performed in four stages:

1. Load the TF-IDF vectorizer and sparse document-term matrix.
2. Verify that the model dimensions correspond to the processed corpus.
3. Execute retrieval for an initial pilot question.
4. Inspect the retrieved chunks and associated judgment metadata before processing the remaining pilot questions.

## **5.1 Load the TF-IDF Retrieval Model**

The TF-IDF vectorizer and sparse document-term matrix generated during corpus preparation are loaded from disk. The loaded artefacts are validated to ensure consistency with the processed chunk dataset before retrieval experiments are performed.

Only previously generated retrieval artefacts are used during evaluation to ensure complete reproducibility of the retrieval pipeline.

In [15]:
# ============================================================
# Load TF-IDF Retrieval Artefacts
# ============================================================

import joblib
from scipy import sparse

tfidf_vectorizer = joblib.load(TFIDF_VECTORIZER_FILE)

tfidf_matrix = sparse.load_npz(TFIDF_MATRIX_FILE)

print("="*70)
print("TF-IDF RETRIEVAL ARTEFACTS")
print("="*70)

print(f"Vocabulary Size : {len(tfidf_vectorizer.vocabulary_):,}")
print(f"Matrix Shape    : {tfidf_matrix.shape}")
print(f"Non-zero Values : {tfidf_matrix.nnz:,}")

assert tfidf_matrix.shape[0] == len(df_chunks)

print("\nTF-IDF artefacts loaded successfully.")

c:\Users\SPAR-HYD-049\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\SPAR-HYD-049\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


TF-IDF RETRIEVAL ARTEFACTS
Vocabulary Size : 230,196
Matrix Shape    : (30521, 230196)
Non-zero Values : 12,406,627

TF-IDF artefacts loaded successfully.


## **5.2 Validate Corpus Alignment**

The TF-IDF document-term matrix must contain one row for every processed text chunk. Validation of matrix dimensions confirms that the retrieval model remains correctly aligned with the processed corpus used throughout the evaluation.

In [16]:
print("="*70)
print("CORPUS ALIGNMENT")
print("="*70)

print(f"Chunks in Corpus : {len(df_chunks):,}")
print(f"Rows in Matrix   : {tfidf_matrix.shape[0]:,}")

print(
    "\nAlignment:",
    tfidf_matrix.shape[0] == len(df_chunks)
)

CORPUS ALIGNMENT
Chunks in Corpus : 30,521
Rows in Matrix   : 30,521

Alignment: True


## **5.3 Execute Retrieval for a Single Pilot Question**

Before evaluating the complete pilot benchmark, retrieval is first performed for a single legal research question. This preliminary validation confirms that the TF-IDF retrieval pipeline functions correctly and returns ranked legal text chunks together with the associated judgment metadata.

The retrieved results are manually inspected to verify their relevance and correctness before processing the remaining pilot questions.

In [17]:
# ============================================================
# Select First Pilot Question
# ============================================================

sample_question = df_pilot.iloc[0]["Question"]

print("=" * 70)
print("PILOT QUESTION")
print("=" * 70)

print(sample_question)

PILOT QUESTION
Under what circumstances can delay in lodging an FIR prove fatal to prosecution?


## **5.4 Retrieve Relevant Chunks Using TF-IDF**

The pilot question is transformed into a TF-IDF query vector using the previously trained vectorizer. Cosine similarity is then computed between the query vector and every document chunk in the corpus.

The document chunks are ranked according to their similarity scores, and the highest-ranked chunks are retrieved for manual inspection. The retrieved results include the chunk identifier, judgment identifier, case title, chunk text, and similarity score.

In [18]:
# ============================================================
# TF-IDF Retrieval Function
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

def retrieve_tfidf(query, top_k=5):
    """
    Retrieve the Top-k most relevant chunks using TF-IDF.
    """

    # Convert query into TF-IDF vector
    query_vector = tfidf_vectorizer.transform([query])

    # Compute cosine similarity
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Top-k indices
    top_indices = np.argsort(similarity_scores)[::-1][:top_k]

    # Build results dataframe
    results = df_chunks.iloc[top_indices].copy()

    results["Similarity_Score"] = similarity_scores[top_indices]

    return results[
        [
            "global_chunk_id",
            "judgment_id",
            "case_title",
            "local_chunk_id",
            "Similarity_Score",
            "chunk_text"
        ]
    ].reset_index(drop=True)

## **5.5 Validate TF-IDF Retrieval**

The TF-IDF retrieval function is validated using the first pilot question. The retrieved chunks are manually inspected to determine whether the retrieved judgments appear relevant to the legal issue posed by the query.

In [19]:
tfidf_results = retrieve_tfidf(
    sample_question,
    top_k=TOP_K
)

print("=" * 70)
print("TF-IDF RETRIEVAL RESULTS")
print("=" * 70)

display(tfidf_results)

TF-IDF RETRIEVAL RESULTS


,global_chunk_id,judgment_id,case_title,local_chunk_id,Similarity_Score,chunk_text
0,19490,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,7,0.125399,the contention in that regard on behalf of A-1...
1,19489,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,6,0.121470,8 Hereinafter ‘PW-17’ 9 (2002) 7 SCC 91\n\n[20...
2,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.118694,"provides that ‘if, from information received o..."
3,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,0.115738,15 Nevada Properties (P) Ltd. Vs. State of Mah...
4,14761,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,43,0.095032,358 [2024] 7 S.C.R. Digital Supreme Court Repo...


In [20]:
# ============================================================
# Display Retrieved Chunks for Manual Inspection
# ============================================================

for i, row in tfidf_results.iterrows():

    print("=" * 100)
    print(f"Rank             : {i+1}")
    print(f"Similarity Score : {row['Similarity_Score']:.4f}")
    print(f"Judgment ID      : {row['judgment_id']}")
    print(f"Case Title       : {row['case_title']}")
    print(f"Local Chunk ID   : {row['local_chunk_id']}")
    print("-" * 100)
    print(row["chunk_text"])
    print("\n")

Rank             : 1
Similarity Score : 0.1254
Judgment ID      : [2024] 3 S.C.R. 747
Case Title       : PERIYASAMY versus THE STATE REPRESENTED BY THE INSPECTOR OF POLICE
Local Chunk ID   : 7
----------------------------------------------------------------------------------------------------
the contention in that regard on behalf of A-1 has to be negatived. Given that the presence of the owner of the shop has nowhere been mentioned, his non-examination cannot be termed fatal to the prosecution case. 11. In such terms, the High Court confirmed the conviction and sentence handed down by the Trial Court as regards A1 and A2. Submissions 12. The present appeals are a challenge to the judgments of the Trial Court and High Court. We have heard Mr. S Nagamuthu, learned senior counsel, Mr. S. Arun Prakash for A-1, and Mr. Vipin Jai for A-2. Dr. Joseph Aristotle S. was heard for the State. Contentions on behalf of A-1 13. The primary ground urged on behalf of A-1 was that nearly all witnesses

## **5.7 Preliminary Relevance Assessment**

The retrieved chunks were manually examined to determine whether they addressed the legal effect of delay in lodging an FIR.

Relevance was assessed using a graded scheme:

- **2 — Highly relevant:** The chunk states or applies the legal principle required to answer the question.
- **1 — Partially relevant:** The chunk discusses the issue or presents related factual or argumentative context but does not independently provide the governing legal principle.
- **0 — Not relevant:** The chunk does not materially assist in answering the question.

The pilot inspection also identified repeated retrieval of overlapping chunks from the same judgment. Accordingly, both chunk-level and judgment-level retrieval performance will be examined during the pilot evaluation.

In [21]:
# ============================================================
# Preliminary Manual Relevance Labels for Sample Question
# ============================================================

sample_relevance = {
    4254: 2,     # Direct legal principle on FIR delay
    4253: 2,     # Relevant but overlapping with Rank 1
    19490: 1,    # Issue raised, but governing reasoning not shown
    19489: 1,    # Overlapping contextual passage
    18642: 0     # Does not address effect of FIR delay
}

tfidf_results["Relevance_Grade"] = (
    tfidf_results["global_chunk_id"]
    .map(sample_relevance)
)

display(
    tfidf_results[
        [
            "global_chunk_id",
            "judgment_id",
            "case_title",
            "Similarity_Score",
            "Relevance_Grade"
        ]
    ]
)

,global_chunk_id,judgment_id,case_title,Similarity_Score,Relevance_Grade
0,19490,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,0.125399,1.0
1,19489,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,0.121470,1.0
2,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,0.118694,2.0
3,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,0.115738,2.0
4,14761,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,0.095032,NaN


# **Part F — TF-IDF Retrieval for the Pilot Benchmark**

Following successful validation using a single pilot question, the TF-IDF retrieval pipeline is executed for the complete pilot benchmark.

For each pilot question, the top five ranked document chunks are retrieved together with their similarity scores and judgment metadata. The retrieved results are stored for subsequent manual relevance annotation and computation of retrieval evaluation metrics.

## **6.1 Retrieve Top-k Results for All Pilot Questions**

The TF-IDF retrieval function is applied to each pilot question using a fixed retrieval depth of five. The retrieval outputs are consolidated into a single dataframe for subsequent inspection and evaluation.

In [22]:
# ============================================================
# Retrieve TF-IDF Results for Entire Pilot Benchmark
# ============================================================

all_tfidf_results = []

for _, row in df_pilot.iterrows():

    question_id = row["Question_ID"]
    question = row["Question"]

    retrieved = retrieve_tfidf(
        question,
        top_k=TOP_K
    )

    retrieved.insert(0, "Question", question)
    retrieved.insert(0, "Question_ID", question_id)

    retrieved["Rank"] = range(1, len(retrieved) + 1)

    all_tfidf_results.append(retrieved)

df_tfidf_results = pd.concat(
    all_tfidf_results,
    ignore_index=True
)

print("=" * 70)
print("TF-IDF PILOT RETRIEVAL")
print("=" * 70)

print(f"Pilot Questions : {len(df_pilot)}")
print(f"Retrieved Chunks: {len(df_tfidf_results)}")

display(df_tfidf_results.head(15))

TF-IDF PILOT RETRIEVAL
Pilot Questions : 10
Retrieved Chunks: 50


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,Similarity_Score,chunk_text,Rank
0,RAW001,Under what circumstances can delay in lodging ...,19490,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,7,0.125399,the contention in that regard on behalf of A-1...,1
1,RAW001,Under what circumstances can delay in lodging ...,19489,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,6,0.121470,8 Hereinafter ‘PW-17’ 9 (2002) 7 SCC 91\n\n[20...,2
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.118694,"provides that ‘if, from information received o...",3
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,0.115738,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4
4,RAW001,Under what circumstances can delay in lodging ...,14761,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,43,0.095032,358 [2024] 7 S.C.R. Digital Supreme Court Repo...,5
5,RAW002,Can a conviction be sustained solely on the te...,9132,[2024] 12 S.C.R. 769,GEORGE versus THE STATE OF TAMIL NADU AND OTHERS,10,0.183937,[2024] 12 S.C.R. 775 George v. The State of Ta...,1
6,RAW002,Can a conviction be sustained solely on the te...,9133,[2024] 12 S.C.R. 769,GEORGE versus THE STATE OF TAMIL NADU AND OTHERS,11,0.177264,caused the injury to the deceased. In our cons...,2
7,RAW002,Can a conviction be sustained solely on the te...,4824,[2024] 4 S.C.R. 48,BALLU @ BALRAM @ BALMUKUND AND ANOTHER versus ...,21,0.154247,14. This elaborate exercise of the learned tri...,3
8,RAW002,Can a conviction be sustained solely on the te...,4823,[2024] 4 S.C.R. 48,BALLU @ BALRAM @ BALMUKUND AND ANOTHER versus ...,20,0.153968,of the material placed on record.\n\n60 [2024]...,4
9,RAW002,Can a conviction be sustained solely on the te...,13734,[2024] 8 S.C.R. 345,ALLARAKHA HABIB MEMON ETC. versus STATE OF GUJ...,46,0.148023,372 [2024] 8 S.C.R. Digital Supreme Court Repo...,5


## **6.3 Save TF-IDF Pilot Retrieval Results**

The TF-IDF retrieval outputs are saved to the pilot evaluation directory. Preserving retrieval results ensures reproducibility of the pilot experiment and provides the basis for subsequent manual relevance annotation and retrieval metric computation.

Each retrieved record contains the originating pilot question, retrieval rank, similarity score, judgment metadata, and retrieved chunk text.

In [23]:
# ============================================================
# Save TF-IDF Pilot Retrieval Results
# ============================================================

tfidf_output_file = (
    PILOT_RETRIEVAL_DIR /
    "tfidf_pilot_retrieval.csv"
)

df_tfidf_results.to_csv(
    tfidf_output_file,
    index=False
)

print("=" * 60)
print("TF-IDF PILOT RESULTS SAVED")
print("=" * 60)
print(f"Location : {tfidf_output_file}")
print(f"Rows     : {len(df_tfidf_results):,}")

TF-IDF PILOT RESULTS SAVED
Location : D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\retrieval\tfidf_pilot_retrieval.csv
Rows     : 50


# **Part G — Pilot Relevance Annotation**

The retrieval outputs generated during the pilot evaluation are manually annotated to establish a provisional relevance benchmark for retrieval evaluation.

Each retrieved chunk is assessed to determine whether it contributes to answering the legal research question posed. The resulting annotations provide the reference labels required for computing retrieval evaluation metrics, including Precision@k, Recall@k, Hit@k, Mean Reciprocal Rank (MRR), and Normalized Discounted Cumulative Gain (nDCG@k).

The pilot annotations are intended solely to validate the annotation workflow and evaluation pipeline. They are not used as the final benchmark annotations for the full experimental evaluation.

## **7.1 Create the Annotation Template**

An annotation template is generated from the TF-IDF retrieval outputs. The template contains the pilot questions together with the retrieved judgments and chunk metadata.

Additional columns are provided for manually recording the relevance of each retrieved chunk and any reviewer observations.

In [24]:
# ============================================================
# Create Annotation Template
# ============================================================

annotation_df = df_tfidf_results.copy()

annotation_df["Relevance"] = ""
annotation_df["Relevance_Level"] = ""
annotation_df["Comments"] = ""

display(annotation_df.head())

,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,Similarity_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,19490,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,7,0.125399,the contention in that regard on behalf of A-1...,1,,,
1,RAW001,Under what circumstances can delay in lodging ...,19489,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,6,0.121470,8 Hereinafter ‘PW-17’ 9 (2002) 7 SCC 91\n\n[20...,2,,,
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.118694,"provides that ‘if, from information received o...",3,,,
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,0.115738,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4,,,
4,RAW001,Under what circumstances can delay in lodging ...,14761,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,43,0.095032,358 [2024] 7 S.C.R. Digital Supreme Court Repo...,5,,,


## **7.2 Save the Annotation Template**

The annotation template is saved as a CSV file in the pilot annotation directory. The file serves as the working document for manual relevance assessment during the pilot evaluation.

In [25]:
annotation_file = (
    PILOT_ANNOTATION_DIR /
    "tfidf_annotation_template.csv"
)

annotation_df.to_csv(annotation_file, index=False)

print("=" * 60)
print("ANNOTATION TEMPLATE CREATED")
print("=" * 60)

print(annotation_file)

ANNOTATION TEMPLATE CREATED
D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\annotation\tfidf_annotation_template.csv


## **7.3 Annotation Guidelines**

Each retrieved chunk is assigned one of the following relevance labels:

| Relevance | Interpretation |
|------------|----------------|
| **2** | Highly relevant. The chunk directly answers or contains the governing legal principle required to answer the question. |
| **1** | Partially relevant. The chunk discusses the issue or provides useful contextual information but is insufficient by itself to answer the question completely. |
| **0** | Not relevant. The chunk does not materially assist in answering the legal research question. |

Reviewer comments may be recorded where clarification is required, including duplicate chunks, overlapping passages, or retrieval anomalies.

# **Part H — Manual Relevance Annotation**

The TF-IDF retrieval results obtained for the pilot benchmark are manually reviewed to determine their relevance to each legal research question.

Each retrieved chunk is examined in the context of the corresponding legal question and assigned a relevance label. The resulting annotations constitute the provisional reference standard used to validate the retrieval evaluation pipeline during the pilot study.

The pilot annotation exercise also provides an opportunity to refine the annotation guidelines, identify duplicate retrievals, and assess the practicality of the evaluation framework before constructing the final benchmark.

## **8.1 Manual Annotation Procedure**

For each pilot question:

1. Read the legal research question.
2. Examine each of the five retrieved chunks.
3. Assign an appropriate relevance label.
4. Record observations where necessary.

The completed annotation file will subsequently be used to validate the retrieval metrics implemented in this notebook.

# **Part I — Retrieval Evaluation Metrics**

The manually annotated pilot retrieval results are used to compute standard information retrieval evaluation metrics.

These metrics quantify the ability of the retrieval model to identify legally relevant Supreme Court judgment chunks for each benchmark question. The pilot evaluation validates both the implementation of the evaluation metrics and the annotation framework before applying the methodology to the final benchmark dataset.

The retrieval metrics computed in this section are:

- Precision@5

- Hit@5

- Mean Reciprocal Rank (MRR)

- Normalized Discounted Cumulative Gain (nDCG@5)

The same evaluation procedure will subsequently be applied to every retrieval model evaluated within the NYAYaRAG framework.

## **9.1 Load the Annotated Retrieval Results**

The manually annotated retrieval file is loaded from the pilot annotation directory. The annotated relevance labels constitute the reference standard for computing retrieval performance metrics.

In [78]:
# ============================================================
# Load Annotated TF-IDF Pilot Results
# ============================================================

ANNOTATION_FILE = (
    PILOT_ANNOTATION_DIR /
    "tfidf_gemini_judged.csv"
)

annotated_df = pd.read_csv(ANNOTATION_FILE)

print("=" * 70)
print("ANNOTATED PILOT RETRIEVAL RESULTS")
print("=" * 70)

print(f"Rows    : {len(annotated_df):,}")
print(f"Columns : {len(annotated_df.columns)}")

display(annotated_df.head())

ANNOTATED PILOT RETRIEVAL RESULTS
Rows    : 50
Columns : 12


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,Similarity_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,19490,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,7,0.125399,the contention in that regard on behalf of A-1...,1,0,Not relevant,Submissions. The chunk only summarizes the def...
1,RAW001,Under what circumstances can delay in lodging ...,19489,[2024] 3 S.C.R. 747,PERIYASAMY versus THE STATE REPRESENTED BY THE...,6,0.121470,8 Hereinafter ‘PW-17’ 9 (2002) 7 SCC 91\n\n[20...,2,0,Not relevant,Submissions. The chunk only summarizes the def...
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.118694,"provides that ‘if, from information received o...",3,1,Partially relevant,Ratio/holding. It provides a partial test for ...
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,0.115738,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4,1,Partially relevant,Ratio/holding. It provides a partial test for ...
4,RAW001,Under what circumstances can delay in lodging ...,14761,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,43,0.095032,358 [2024] 7 S.C.R. Digital Supreme Court Repo...,5,0,Not relevant,Facts. The passage only discusses the trial co...


## **9.2 Validate the Annotation Dataset**

Before computing retrieval metrics, the annotated dataset is validated to ensure that all retrieved chunks possess completed relevance annotations and that the expected number of pilot retrieval records is present.

In [79]:
print("=" * 70)
print("ANNOTATION VALIDATION")
print("=" * 70)

print(f"Total Records : {len(annotated_df)}")

print("\nMissing Values\n")
print(annotated_df.isnull().sum())

print("\nRelevance Labels\n")
print(annotated_df["Relevance"].value_counts())

print("\nRelevance Levels\n")
print(annotated_df["Relevance_Level"].value_counts())

ANNOTATION VALIDATION
Total Records : 50

Missing Values

Question_ID         0
Question            0
global_chunk_id     0
judgment_id         0
case_title          0
local_chunk_id      0
Similarity_Score    0
chunk_text          0
Rank                0
Relevance           0
Relevance_Level     0
Comments            0
dtype: int64

Relevance Labels

Relevance
1    26
0    18
2     6
Name: count, dtype: int64

Relevance Levels

Relevance_Level
Partially relevant    26
Not relevant          18
Highly relevant        6
Name: count, dtype: int64


## **9.3 Prepare Relevance Labels for Metric Computation**

The manual annotations collected during the pilot evaluation employ three relevance categories:

- **2 – Highly Relevant**
- **1 – Partially Relevant**
- **0 – Not Relevant**

Different retrieval evaluation metrics require different interpretations of these labels.

For **Precision@5, Recall@5, Hit@5, and Mean Reciprocal Rank (MRR)**, relevance is treated as a binary concept. Accordingly, both **Highly Relevant (2)** and **Partially Relevant (1)** chunks are regarded as relevant, while **Not Relevant (0)** chunks are regarded as non-relevant.

For **Normalized Discounted Cumulative Gain (nDCG@5)**, the original graded relevance labels are retained because nDCG explicitly rewards highly relevant documents more than partially relevant documents.

This preprocessing step creates a binary relevance variable while preserving the original graded annotations for subsequent evaluation.

In [80]:
# ============================================================
# Create Binary Relevance Labels
# ============================================================

annotated_df["Binary_Relevance"] = (
    annotated_df["Relevance"] > 0
).astype(int)

print("=" * 70)
print("BINARY RELEVANCE DISTRIBUTION")
print("=" * 70)

print(annotated_df["Binary_Relevance"].value_counts().sort_index())

display(
    annotated_df[
        [
            "Question_ID",
            "Rank",
            "Relevance",
            "Binary_Relevance",
            "Relevance_Level"
        ]
    ].head(10)
)

BINARY RELEVANCE DISTRIBUTION
Binary_Relevance
0    18
1    32
Name: count, dtype: int64


,Question_ID,Rank,Relevance,Binary_Relevance,Relevance_Level
0,RAW001,1,0,0,Not relevant
1,RAW001,2,0,0,Not relevant
2,RAW001,3,1,1,Partially relevant
3,RAW001,4,1,1,Partially relevant
4,RAW001,5,0,0,Not relevant
5,RAW002,1,2,1,Highly relevant
6,RAW002,2,2,1,Highly relevant
7,RAW002,3,0,0,Not relevant
8,RAW002,4,0,0,Not relevant
9,RAW002,5,0,0,Not relevant


## **9.4 Precision@5**



Precision@5 measures the proportion of the top five retrieved chunks that are relevant to the legal research question.

For the pilot evaluation, a retrieved chunk is considered relevant if it has been annotated as either **Highly Relevant** or **Partially Relevant**.

Precision@5 is computed independently for each pilot question and then averaged across all questions to obtain the overall retrieval precision of the TF-IDF model.

The mathematical definition is:

$$
\text{Precision@5}
=
\frac{\text{Number of Relevant Chunks Retrieved in Top 5}}{5}
$$

Since five chunks are retrieved for every query during the pilot evaluation, the denominator remains constant.

In [81]:
# ============================================================
# Compute Precision@5 for TF-IDF
# ============================================================

precision_at_5 = (
    annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .mean()
    .reset_index(name="Precision@5")
)

mean_precision_at_5 = precision_at_5["Precision@5"].mean()

print("=" * 70)
print("TF-IDF PRECISION@5")
print("=" * 70)

display(precision_at_5)

print(f"\nMean Precision@5 : {mean_precision_at_5:.4f}")

TF-IDF PRECISION@5


,Question_ID,Precision@5
0,RAW001,0.4
1,RAW002,0.4
2,RAW004,0.8
3,RAW005,1.0
4,RAW006,0.8
5,RAW007,0.6
6,RAW010,1.0
7,RAW012,0.0
8,RAW015,0.8
9,RAW017,0.6



Mean Precision@5 : 0.6400


In [82]:
# Confirm that each pilot question has exactly five retrieved chunks
retrieval_counts = (
    annotated_df
    .groupby("Question_ID")
    .size()
)

assert (retrieval_counts == TOP_K).all(), (
    "At least one question does not have exactly TOP_K retrieved chunks."
)

assert precision_at_5["Precision@5"].between(0, 1).all(), (
    "Precision@5 values must lie between 0 and 1."
)

print("Precision@5 validation completed successfully.")

Precision@5 validation completed successfully.


## **9.5 Interpretation of Precision@5**

The TF-IDF retrieval model achieved a mean **Precision@5 of 0.58** across the ten pilot questions.

This indicates that, on average, **58% of the five highest-ranked retrieved chunks were relevant** to the corresponding legal research question.

The pilot results demonstrate that the retrieval pipeline functions correctly and is capable of identifying relevant legal passages for a substantial proportion of benchmark queries. However, the variation observed across individual questions suggests that retrieval effectiveness is dependent on the nature of the legal issue and the lexical overlap between the query and the corpus.

The pilot Precision@5 value is interpreted solely as a validation of the retrieval and evaluation pipeline. Final retrieval performance will be determined using the complete benchmark dataset and compared across all retrieval models.

## **9.6 Hit@5**

Hit@5 measures whether the retrieval model successfully retrieves **at least one relevant chunk** within the top five ranked results for a given legal research question.

Unlike Precision@5, which measures the proportion of retrieved chunks that are relevant, Hit@5 evaluates whether the retrieval model retrieves any relevant evidence for the query.

For the pilot evaluation, a retrieved chunk is considered relevant if it has been annotated as either **Highly Relevant (2)** or **Partially Relevant (1)**.

The overall Hit@5 score is calculated as:

$$
\mathrm{Hit@5}
=
\frac{\text{Number of Questions with at Least One Relevant Chunk in the Top 5}}
{\text{Total Number of Questions}}
$$

For an individual question:

- **Hit@5 = 1** if at least one relevant chunk is retrieved within the top five results.
- **Hit@5 = 0** if no relevant chunk is retrieved within the top five results.

The overall Hit@5 value is obtained by averaging the individual Hit@5 scores across all pilot questions.

In [83]:
# ============================================================
# Compute Hit@5 for TF-IDF
# ============================================================

hit_at_5 = (
    annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .max()
    .reset_index(name="Hit@5")
)

mean_hit_at_5 = hit_at_5["Hit@5"].mean()

print("=" * 70)
print("TF-IDF HIT@5")
print("=" * 70)

display(hit_at_5)

print(f"\nMean Hit@5 : {mean_hit_at_5:.4f}")

TF-IDF HIT@5


,Question_ID,Hit@5
0,RAW001,1
1,RAW002,1
2,RAW004,1
3,RAW005,1
4,RAW006,1
5,RAW007,1
6,RAW010,1
7,RAW012,0
8,RAW015,1
9,RAW017,1



Mean Hit@5 : 0.9000


In [84]:
# ============================================================
# Validate Hit@5
# ============================================================

assert hit_at_5["Hit@5"].isin([0, 1]).all(), (
    "Hit@5 values must be either 0 or 1."
)

print("Hit@5 validation completed successfully.")

Hit@5 validation completed successfully.


## **9.7 Interpretation of Hit@5**

The TF-IDF retrieval model achieved a mean **Hit@5 of 0.70** across the ten pilot questions.

This indicates that the model successfully retrieved at least one relevant legal text chunk within the top five ranked results for **70% of the pilot questions**.

The pilot Hit@5 result confirms that the retrieval pipeline is capable of identifying relevant legal authorities for the majority of benchmark queries. However, three pilot questions did not retrieve any relevant chunk within the top five results, indicating potential limitations of lexical retrieval for certain legal queries.

As with Precision@5, the pilot Hit@5 value is used solely to validate the retrieval and evaluation pipeline. Final performance comparisons will be conducted using the complete benchmark dataset across all retrieval models.

## **9.8 Mean Reciprocal Rank**

Mean Reciprocal Rank (MRR) evaluates how highly the first relevant chunk is ranked for each legal research question.

For an individual question, the reciprocal rank is calculated as:

$$
\mathrm{Reciprocal\ Rank}
=
\frac{1}{\text{Rank of the First Relevant Chunk}}
$$

If no relevant chunk is retrieved within the top five results, the reciprocal rank is assigned a value of **0**.

The overall MRR is obtained by averaging the reciprocal-rank values across all pilot questions.

MRR is particularly useful in legal retrieval because it rewards models that place the first useful legal authority near the top of the ranked results.

In [85]:
# ============================================================
# Compute Mean Reciprocal Rank for TF-IDF
# ============================================================

def reciprocal_rank(group):
    relevant_rows = group[group["Binary_Relevance"] == 1]

    if relevant_rows.empty:
        return 0.0

    first_relevant_rank = relevant_rows["Rank"].min()
    return 1.0 / first_relevant_rank


mrr_per_question = (
    annotated_df
    .groupby("Question_ID")
    .apply(reciprocal_rank, include_groups=False)
    .reset_index(name="Reciprocal_Rank")
)

mean_mrr = mrr_per_question["Reciprocal_Rank"].mean()

print("=" * 70)
print("TF-IDF MEAN RECIPROCAL RANK")
print("=" * 70)

display(mrr_per_question)

print(f"\nMean Reciprocal Rank : {mean_mrr:.4f}")

TF-IDF MEAN RECIPROCAL RANK


,Question_ID,Reciprocal_Rank
0,RAW001,0.333333
1,RAW002,1.000000
2,RAW004,0.500000
3,RAW005,1.000000
4,RAW006,1.000000
5,RAW007,1.000000
6,RAW010,1.000000
7,RAW012,0.000000
8,RAW015,1.000000
9,RAW017,0.333333



Mean Reciprocal Rank : 0.7167


In [86]:
# ============================================================
# Validate MRR
# ============================================================

assert mrr_per_question["Reciprocal_Rank"].between(0, 1).all(), (
    "Reciprocal-rank values must lie between 0 and 1."
)

assert len(mrr_per_question) == df_pilot["Question_ID"].nunique(), (
    "MRR was not computed for every pilot question."
)

print("MRR validation completed successfully.")

MRR validation completed successfully.


## **9.9 Interpretation of Mean Reciprocal Rank**

The TF-IDF retrieval model achieved a **Mean Reciprocal Rank (MRR) of 0.6333** across the ten pilot questions.

This result indicates that, on average, the first relevant legal text chunk was retrieved relatively high in the ranked list. For six of the ten pilot questions, the first retrieved chunk was already relevant, yielding a reciprocal rank of **1.0**. One question retrieved the first relevant chunk at Rank 3, while three questions failed to retrieve any relevant chunk within the top five results.

The pilot MRR demonstrates that the retrieval model is capable of presenting useful legal authorities early in the ranked output for a majority of benchmark queries. As with the other pilot metrics, this value serves to validate the retrieval evaluation pipeline and will be compared with the corresponding values obtained from the remaining retrieval models during the final evaluation.

## **9.10 Normalized Discounted Cumulative Gain (nDCG@5)**

Normalized Discounted Cumulative Gain (nDCG@5) evaluates the quality of the ranking produced by the retrieval model by considering both the relevance of the retrieved chunks and their positions within the ranked list.

Unlike Precision@5, Hit@5, and Mean Reciprocal Rank, which treat relevance as a binary concept, nDCG@5 utilises the graded relevance annotations assigned during manual review:

- **2 – Highly Relevant**
- **1 – Partially Relevant**
- **0 – Not Relevant**

Highly relevant chunks contribute more to the score than partially relevant chunks, and relevant chunks appearing at higher ranks receive greater weight than those appearing lower in the ranking.

The Discounted Cumulative Gain (DCG) for a query is computed as:

$$
DCG@5=
\sum_{i=1}^{5}
\frac{2^{rel_i}-1}
{\log_2(i+1)}
$$

The DCG is then normalised by the Ideal Discounted Cumulative Gain (IDCG), obtained by ranking the same retrieved chunks in descending order of relevance:

$$
nDCG@5=
\frac{DCG@5}{IDCG@5}
$$

The overall nDCG@5 is calculated as the mean of the individual nDCG values across all pilot questions.

In [87]:
# ============================================================
# Compute nDCG@5 for TF-IDF
# ============================================================

from sklearn.metrics import ndcg_score
import numpy as np

ndcg_scores = []

for question_id, group in annotated_df.groupby("Question_ID"):

    # True graded relevance
    true_relevance = group["Relevance"].to_numpy().reshape(1, -1)

    # Predicted ranking scores (similarity)
    predicted_scores = group["Similarity_Score"].to_numpy().reshape(1, -1)

    score = ndcg_score(
        true_relevance,
        predicted_scores,
        k=TOP_K
    )

    ndcg_scores.append(
        {
            "Question_ID": question_id,
            "nDCG@5": score
        }
    )

ndcg_results = pd.DataFrame(ndcg_scores)

mean_ndcg = ndcg_results["nDCG@5"].mean()

print("=" * 70)
print("TF-IDF nDCG@5")
print("=" * 70)

display(ndcg_results)

print(f"\nMean nDCG@5 : {mean_ndcg:.4f}")

TF-IDF nDCG@5


,Question_ID,nDCG@5
0,RAW001,0.570642
1,RAW002,1.000000
2,RAW004,0.760640
3,RAW005,0.847086
4,RAW006,1.000000
5,RAW007,0.946902
6,RAW010,1.000000
7,RAW012,0.000000
8,RAW015,1.000000
9,RAW017,0.618289



Mean nDCG@5 : 0.7744


In [88]:
# ============================================================
# Validate nDCG@5
# ============================================================

assert ndcg_results["nDCG@5"].between(0, 1).all(), (
    "nDCG values must lie between 0 and 1."
)

print("nDCG@5 validation completed successfully.")

nDCG@5 validation completed successfully.


## **9.11 Interpretation of nDCG@5**

The TF-IDF retrieval model achieved a mean **nDCG@5 of 0.6200** across the ten pilot questions.

Unlike the preceding retrieval metrics, nDCG@5 considers both the graded relevance of the retrieved chunks and their positions within the ranked list. Consequently, highly relevant chunks retrieved near the top of the ranking contribute more to the overall score than partially relevant chunks retrieved at lower ranks.

The pilot nDCG@5 result indicates that the TF-IDF retrieval model generally ranked the most relevant legal evidence higher than less relevant evidence for a majority of the pilot questions. Nevertheless, several questions yielded no relevant retrieved chunks, resulting in nDCG values of zero and reducing the overall average.

The pilot nDCG@5 score validates the implementation of graded relevance evaluation within the NYAYaRAG framework and will subsequently be compared with the corresponding values obtained from BM25, E5, BGE, InLegalBERT, and the hybrid retrieval model during the final evaluation.

In [89]:
# ============================================================
# TF-IDF Pilot Evaluation Summary
# ============================================================

tfidf_summary = pd.DataFrame({
    "Retriever": ["TF-IDF"],
    "Precision@5": [mean_precision_at_5],
    "Hit@5": [mean_hit_at_5],
    "MRR": [mean_mrr],
    "nDCG@5": [mean_ndcg]
})

display(tfidf_summary.round(4))

,Retriever,Precision@5,Hit@5,MRR,nDCG@5
0,TF-IDF,0.64,0.9,0.7167,0.7744


# **Part J — BM25 Retrieval Evaluation**

Following the successful validation of the TF-IDF retrieval pipeline, the same pilot benchmark is evaluated using the BM25 lexical retrieval model.

BM25 is a probabilistic lexical retrieval algorithm that ranks document chunks according to term frequency, inverse document frequency, and document length normalization. Unlike TF-IDF, BM25 employs a probabilistic scoring function that generally provides improved retrieval effectiveness for keyword-based search.

The retrieval methodology, manual annotation process, and evaluation metrics remain identical to those used for the TF-IDF evaluation, thereby ensuring a fair comparison between the two lexical retrieval approaches.

## **10.1 Load the BM25 Retrieval Model**

The previously constructed BM25 index is loaded from disk and validated before executing the pilot retrieval experiment.

In [53]:
import os

print(os.listdir(models_dir))

NameError: name 'models_dir' is not defined

In [ ]:
bm25_dir = models_dir / "bm25"

print(bm25_dir)

In [ ]:
import os

print(os.listdir(bm25_dir))

In [ ]:
!pip install rank_bm25

In [55]:
import joblib
from pathlib import Path

BM25_DIR = Path(r"D:\_ResearchNotebooks\NyayaRAG\models\bm25")

bm25 = joblib.load(BM25_DIR / "bm25.pkl")
tokenized_corpus = joblib.load(BM25_DIR / "tokenized_corpus.pkl")

## **10.2 Validate Corpus Alignment**

Before executing retrieval, the BM25 retrieval artefacts are validated to ensure consistency with the processed corpus used throughout the NYAYaRAG framework.

The validation confirms that the saved BM25 corpus contains the same number of document chunks as the processed chunk dataset generated during corpus preparation. This step ensures that every BM25 retrieval score corresponds to the correct legal text chunk and associated judgment metadata.

In [56]:
# ============================================================
# Validate BM25 Corpus Alignment
# ============================================================

print("=" * 70)
print("BM25 CORPUS VALIDATION")
print("=" * 70)

print(f"BM25 Corpus Size      : {len(tokenized_corpus):,}")
print(f"Processed Chunks      : {len(df_chunks):,}")

alignment = len(tokenized_corpus) == len(df_chunks)

print(f"Corpus Alignment      : {alignment}")

assert alignment, (
    "BM25 corpus and processed chunk dataset are not aligned."
)

print("\nBM25 corpus validation completed successfully.")

BM25 CORPUS VALIDATION
BM25 Corpus Size      : 30,521
Processed Chunks      : 30,521
Corpus Alignment      : True

BM25 corpus validation completed successfully.


## **10.3 Execute BM25 Retrieval for a Single Pilot Question**

Before applying BM25 to the complete pilot benchmark, retrieval is first performed for a single legal research question.

The query is tokenised using the same preprocessing approach employed during construction of the saved BM25 index. BM25 scores are then computed for all corpus chunks, and the top five ranked chunks are retrieved together with their judgment metadata.

The retrieved results are manually inspected to verify that the saved BM25 model remains aligned with the processed chunk dataset and produces meaningful legal retrieval outputs.

## **10.3 Select the First Pilot Question**

Before applying BM25 to the complete pilot benchmark, retrieval is first tested using the same initial pilot question previously used for TF-IDF validation.

Using the same question across retrieval models ensures a consistent qualitative comparison of ranking behaviour and retrieved legal content.

In [57]:
# ============================================================
# Select First Pilot Question for BM25 Validation
# ============================================================

bm25_sample_question = df_pilot.iloc[0]["Question"]

print("=" * 70)
print("BM25 PILOT QUESTION")
print("=" * 70)

print(bm25_sample_question)

assert bm25_sample_question == sample_question, (
    "BM25 validation must use the same first pilot question as TF-IDF."
)

print("\nPilot question consistency validated successfully.")

BM25 PILOT QUESTION
Under what circumstances can delay in lodging an FIR prove fatal to prosecution?

Pilot question consistency validated successfully.


## **10.4 Define the BM25 Retrieval Function**

The BM25 retrieval function tokenises the legal research question using the same query preprocessing approach applied during construction of the BM25 index.

The saved BM25 model then assigns a relevance score to every corpus chunk. The chunks are ranked in descending order of BM25 score, and the top five results are returned together with their judgment metadata.

To ensure consistency across retrieval models, the BM25 function returns the same core output structure used for TF-IDF retrieval.

In [58]:
# ============================================================
# BM25 Retrieval Function
# ============================================================

import re
import numpy as np


def tokenize_bm25_query(query):
    """
    Tokenise a legal research query for BM25 retrieval.

    The query is converted to lowercase and split into
    alphanumeric word tokens.
    """
    return re.findall(r"\b\w+\b", str(query).lower())


def retrieve_bm25(query, top_k=5):
    """
    Retrieve the top-k most relevant chunks using BM25.

    Parameters
    ----------
    query : str
        Legal research question.
    top_k : int
        Number of ranked chunks to retrieve.

    Returns
    -------
    pandas.DataFrame
        Ranked BM25 retrieval results with chunk and judgment metadata.
    """

    query_tokens = tokenize_bm25_query(query)

    bm25_scores = np.asarray(
        bm25.get_scores(query_tokens),
        dtype=float
    )

    top_indices = np.argsort(bm25_scores)[::-1][:top_k]

    results = df_chunks.iloc[top_indices].copy()

    results["BM25_Score"] = bm25_scores[top_indices]

    return results[
        [
            "global_chunk_id",
            "judgment_id",
            "case_title",
            "local_chunk_id",
            "BM25_Score",
            "chunk_text"
        ]
    ].reset_index(drop=True)

In [59]:
# ============================================================
# Validate BM25 Retrieval Function Structure
# ============================================================

assert callable(retrieve_bm25), (
    "The BM25 retrieval function was not created successfully."
)

assert TOP_K > 0, (
    "TOP_K must be a positive integer."
)

print("BM25 retrieval function defined successfully.")

BM25 retrieval function defined successfully.


## **10.5 Validate BM25 Retrieval**

The BM25 retrieval function is validated using the first pilot question.

The top five ranked chunks are examined to confirm that BM25 scores, chunk identifiers, judgment metadata, and retrieved text are returned correctly before the model is applied to the complete pilot benchmark.

In [60]:
# ============================================================
# Validate BM25 Retrieval on the First Pilot Question
# ============================================================

bm25_results = retrieve_bm25(
    bm25_sample_question,
    top_k=TOP_K
)

print("=" * 70)
print("BM25 RETRIEVAL RESULTS")
print("=" * 70)

print(f"Retrieved Chunks : {len(bm25_results)}")

display(bm25_results)

BM25 RETRIEVAL RESULTS
Retrieved Chunks : 5


,global_chunk_id,judgment_id,case_title,local_chunk_id,BM25_Score,chunk_text
0,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,37.246092,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare..."
1,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,36.415445,20 See Yogesh Singh v. Mahabeer Singh and Othe...
2,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,36.094698,"provides that ‘if, from information received o..."
3,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,35.723494,15 Nevada Properties (P) Ltd. Vs. State of Mah...
4,14760,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,42,34.446598,Mobile Ltd. Mohali. Letter Ex. DG is a letter ...


In [61]:
# ============================================================
# Validate BM25 Retrieval Output
# ============================================================

expected_columns = [
    "global_chunk_id",
    "judgment_id",
    "case_title",
    "local_chunk_id",
    "BM25_Score",
    "chunk_text"
]

assert len(bm25_results) == TOP_K, (
    f"Expected {TOP_K} retrieved chunks, "
    f"but found {len(bm25_results)}."
)

assert bm25_results.columns.tolist() == expected_columns, (
    "BM25 retrieval output columns do not match the expected structure."
)

assert bm25_results["BM25_Score"].notna().all(), (
    "One or more BM25 scores are missing."
)

assert (
    bm25_results["BM25_Score"].to_numpy()[:-1]
    >= bm25_results["BM25_Score"].to_numpy()[1:]
).all(), (
    "BM25 results are not arranged in descending score order."
)

print("BM25 retrieval output validation completed successfully.")

BM25 retrieval output validation completed successfully.


## **10.6 Display Retrieved Chunks for Manual Inspection**

The retrieved BM25 chunks are displayed in full to facilitate manual inspection of the retrieval quality.

This qualitative review enables verification that the retrieved legal passages are relevant to the pilot question and provides an initial assessment of the effectiveness of BM25 prior to large-scale evaluation.

In [62]:
# ============================================================
# Display BM25 Retrieved Chunks
# ============================================================

for rank, row in bm25_results.iterrows():

    print("=" * 100)
    print(f"Rank             : {rank + 1}")
    print(f"BM25 Score       : {row['BM25_Score']:.4f}")
    print(f"Judgment ID      : {row['judgment_id']}")
    print(f"Case Title       : {row['case_title']}")
    print(f"Local Chunk ID   : {row['local_chunk_id']}")
    print("-" * 100)
    print(row["chunk_text"])
    print("\n")

Rank             : 1
BM25 Score       : 37.2461
Judgment ID      : [2024] 10 S.C.R. 1313
Case Title       : RAMA DEVI versus THE STATE OF BIHAR AND OTHERS
Local Chunk ID   : 41
----------------------------------------------------------------------------------------------------
(PW-1), Mahanth Ashwani Das (PW-25), and Amarendra Kumar Sinha (PW-10). The inquest reports of Brij Bihari Prasad and Lakshmeshwar Sahu (Exhibits 42/1 and 52) were prepared on the same night and, thereafter, the post-mortem was conducted by Dr. Arvind Kumar Singh (PW-7) at 12:30 a.m. and 1:30 a.m. respectively on 14.06.1998. 30. The occurrence having taken place at night on 13.06.1998, normally the FIR should have been sent to the jurisdictional magistrate on 14.06.1998. However, 14.06.1998 being a Sunday was a holiday. The FIR was forwarded to the jurisdictional magistrate on 15.06.1998. There is, therefore, an explanation for the delay in forwarding a copy of the FIR to the jurisdictional magistrate in terms of

## **10.7 Preliminary Relevance Assessment**

The retrieved BM25 results are manually reviewed to determine whether the highest-ranked chunks address the legal issue raised in the pilot question.

This preliminary assessment serves only as a qualitative validation of the retrieval pipeline. Formal relevance annotation and computation of retrieval evaluation metrics will be performed after retrieval has been completed for all pilot questions.

## **10.8 Retrieve Top-k Results for All Pilot Questions**

Following successful validation on the first pilot question, the BM25 retrieval function is applied to all ten pilot questions.

For each question, the top five ranked chunks are retrieved and combined into a single dataframe containing the question identifier, question text, retrieval rank, BM25 score, chunk identifier, judgment identifier, case title, and retrieved chunk text.

In [63]:
# ============================================================
# Retrieve BM25 Results for the Complete Pilot Benchmark
# ============================================================

all_bm25_results = []

for _, row in df_pilot.iterrows():

    question_id = row["Question_ID"]
    question = row["Question"]

    retrieved = retrieve_bm25(
        question,
        top_k=TOP_K
    )

    retrieved.insert(0, "Question", question)
    retrieved.insert(0, "Question_ID", question_id)

    retrieved["Rank"] = range(1, len(retrieved) + 1)

    all_bm25_results.append(retrieved)

df_bm25_results = pd.concat(
    all_bm25_results,
    ignore_index=True
)

print("=" * 70)
print("BM25 PILOT RETRIEVAL")
print("=" * 70)

print(f"Pilot Questions : {len(df_pilot)}")
print(f"Retrieved Chunks: {len(df_bm25_results)}")

display(df_bm25_results.head(15))

BM25 PILOT RETRIEVAL
Pilot Questions : 10
Retrieved Chunks: 50


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BM25_Score,chunk_text,Rank
0,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,37.246092,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",1
1,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,36.415445,20 See Yogesh Singh v. Mahabeer Singh and Othe...,2
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,36.094698,"provides that ‘if, from information received o...",3
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,35.723494,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4
4,RAW001,Under what circumstances can delay in lodging ...,14760,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,42,34.446598,Mobile Ltd. Mohali. Letter Ex. DG is a letter ...,5
5,RAW002,Can a conviction be sustained solely on the te...,13909,[2024] 7 S.C.R. 365,VINOD JASWANTRAY VYAS (DEAD) THROUGH LRS. vers...,39,46.736489,"serious doubt. In other words, the conduct of ...",1
6,RAW002,Can a conviction be sustained solely on the te...,18798,[2024] 10 S.C.R. 753,VISHWAJEET KERBA MASALKAR versus STATE OF MAH...,16,46.080293,go to the hospital. That injured was not in a ...,2
7,RAW002,Can a conviction be sustained solely on the te...,13908,[2024] 7 S.C.R. 365,VINOD JASWANTRAY VYAS (DEAD) THROUGH LRS. vers...,38,45.920973,3 [2023] 2 SCR 746 : 2023(4) SCALE 478 4 [2023...,3
8,RAW002,Can a conviction be sustained solely on the te...,18797,[2024] 10 S.C.R. 753,VISHWAJEET KERBA MASALKAR versus STATE OF MAH...,15,45.444837,on 5th October 2012 at 09:05 PM. It will also ...,4
9,RAW002,Can a conviction be sustained solely on the te...,19304,[2024] 12 S.C.R. 392,KUNHIMUHAMMED@KUNHEETHU versus THE STATE OF KE...,8,45.107159,also the injured to save him. He has drawn att...,5


In [64]:
# ============================================================
# Validate Complete BM25 Pilot Retrieval
# ============================================================

expected_rows = len(df_pilot) * TOP_K

assert len(df_bm25_results) == expected_rows, (
    f"Expected {expected_rows} BM25 retrieval rows, "
    f"but found {len(df_bm25_results)}."
)

results_per_question = (
    df_bm25_results
    .groupby("Question_ID")
    .size()
)

assert (results_per_question == TOP_K).all(), (
    "At least one pilot question does not have exactly TOP_K BM25 results."
)

assert df_bm25_results["Question_ID"].nunique() == len(df_pilot), (
    "BM25 results do not cover all pilot questions."
)

print("Complete BM25 pilot retrieval validation passed.")

Complete BM25 pilot retrieval validation passed.


## **10.9 Save BM25 Pilot Retrieval Results**

The BM25 retrieval outputs are saved to the pilot retrieval directory to preserve the ranked results for subsequent manual relevance annotation and metric computation.

Each saved record contains the originating pilot question, retrieval rank, BM25 score, chunk identifier, judgment metadata, and retrieved chunk text.

In [65]:
# ============================================================
# Save BM25 Pilot Retrieval Results
# ============================================================

bm25_output_file = (
    PILOT_RETRIEVAL_DIR /
    "bm25_pilot_retrieval.csv"
)

df_bm25_results.to_csv(
    bm25_output_file,
    index=False
)

print("=" * 60)
print("BM25 PILOT RESULTS SAVED")
print("=" * 60)

print(f"Location : {bm25_output_file}")
print(f"Rows     : {len(df_bm25_results):,}")

BM25 PILOT RESULTS SAVED
Location : D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\retrieval\bm25_pilot_retrieval.csv
Rows     : 50


## **10.10 Create the BM25 Annotation Template**

A manual relevance annotation template is generated from the BM25 retrieval results.

The annotation structure is kept identical to the TF-IDF annotation template to ensure methodological consistency across lexical retrieval models.

In [66]:
# ============================================================
# Create BM25 Annotation Template
# ============================================================

bm25_annotation_df = df_bm25_results.copy()

bm25_annotation_df["Relevance"] = ""
bm25_annotation_df["Relevance_Level"] = ""
bm25_annotation_df["Comments"] = ""

display(bm25_annotation_df.head())

,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BM25_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,37.246092,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",1,,,
1,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,36.415445,20 See Yogesh Singh v. Mahabeer Singh and Othe...,2,,,
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,36.094698,"provides that ‘if, from information received o...",3,,,
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,35.723494,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4,,,
4,RAW001,Under what circumstances can delay in lodging ...,14760,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,42,34.446598,Mobile Ltd. Mohali. Letter Ex. DG is a letter ...,5,,,


## **10.11 Save the BM25 Annotation Template**

The BM25 annotation template is saved as a CSV file in the pilot annotation directory. The retrieved chunks will subsequently be reviewed using the same three-level relevance scheme applied to TF-IDF.

In [67]:
# ============================================================
# Save BM25 Annotation Template
# ============================================================

bm25_annotation_file = (
    PILOT_ANNOTATION_DIR /
    "bm25_annotation_template.csv"
)

bm25_annotation_df.to_csv(
    bm25_annotation_file,
    index=False
)

print("=" * 60)
print("BM25 ANNOTATION TEMPLATE CREATED")
print("=" * 60)

print(bm25_annotation_file)

BM25 ANNOTATION TEMPLATE CREATED
D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\annotation\bm25_annotation_template.csv


# **Part K — BM25 Retrieval Evaluation Metrics**

The manually annotated BM25 pilot retrieval results are used to compute the same retrieval evaluation metrics previously applied to TF-IDF.

Using an identical metric definition, relevance interpretation, retrieval depth, and pilot benchmark ensures a fair and methodologically consistent comparison between the two lexical retrieval models.

The BM25 pilot evaluation includes:

- Precision@5
- Hit@5
- Mean Reciprocal Rank (MRR)
- Normalized Discounted Cumulative Gain (nDCG@5)

Recall@5 is not computed at this stage because the complete set of relevant chunks or judgments for each pilot question has not yet been established. Recall will be calculated after pooling the retrieval results obtained from all evaluated retrievers.

The BM25 results generated in this section are preliminary pilot findings intended to validate the evaluation workflow. They are not treated as final answers to the research questions.

## **11.1 Load the Annotated BM25 Retrieval Results**

The manually annotated BM25 retrieval results are loaded from the pilot annotation directory.

The annotation file contains the relevance judgments assigned during manual review and serves as the reference standard for computing the retrieval evaluation metrics. These annotations enable quantitative assessment of the BM25 retrieval model using the same evaluation methodology previously applied to TF-IDF.

In [68]:
import os

print(os.listdir(PILOT_ANNOTATION_DIR))

['bm25_annotation_template.csv', 'tfidf_annotation_completed.csv', 'tfidf_annotation_template.csv', 'tfidf_gemini_judged.csv']


In [71]:
# ============================================================
# Load Annotated BM25 Pilot Results
# ============================================================

BM25_ANNOTATION_FILE = (
    PILOT_ANNOTATION_DIR /
    "bm25_pilot_annotations.csv"
)

bm25_annotated_df = pd.read_csv(BM25_ANNOTATION_FILE)

print("=" * 70)
print("ANNOTATED BM25 PILOT RETRIEVAL RESULTS")
print("=" * 70)

print(f"Rows    : {len(bm25_annotated_df):,}")
print(f"Columns : {len(bm25_annotated_df.columns)}")

display(bm25_annotated_df.head())

ANNOTATED BM25 PILOT RETRIEVAL RESULTS
Rows    : 50
Columns : 12


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BM25_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,37.246092,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",1,2,Highly relevant,Ratio/holding on FIR delay. The chunk expressl...
1,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,36.415445,20 See Yogesh Singh v. Mahabeer Singh and Othe...,2,2,Highly relevant,Ratio/holding on FIR delay. The chunk expressl...
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,36.094698,"provides that ‘if, from information received o...",3,2,Highly relevant,Ratio/holding on FIR delay. The chunk directly...
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,35.723494,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4,2,Highly relevant,Ratio/holding on FIR delay. The chunk directly...
4,RAW001,Under what circumstances can delay in lodging ...,14760,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,42,34.446598,Mobile Ltd. Mohali. Letter Ex. DG is a letter ...,5,0,Not relevant,Facts only. The chunk discusses the specific f...


## **11.2 Validate the Annotation Dataset**

Before computing the retrieval evaluation metrics, the annotated BM25 dataset is validated to ensure that all retrieved chunks have been assigned relevance labels and that the expected number of pilot retrieval records is present.

The validation also summarises the distribution of relevance scores assigned during manual annotation.

In [72]:
# ============================================================
# Validate BM25 Annotation Dataset
# ============================================================

print("=" * 70)
print("BM25 ANNOTATION VALIDATION")
print("=" * 70)

print(f"Total Records : {len(bm25_annotated_df)}")

print("\nMissing Values\n")
print(bm25_annotated_df.isnull().sum())

print("\nRelevance Labels\n")
print(bm25_annotated_df["Relevance"].value_counts())

print("\nRelevance Levels\n")
print(bm25_annotated_df["Relevance_Level"].value_counts())

BM25 ANNOTATION VALIDATION
Total Records : 50

Missing Values

Question_ID        0
Question           0
global_chunk_id    0
judgment_id        0
case_title         0
local_chunk_id     0
BM25_Score         0
chunk_text         0
Rank               0
Relevance          0
Relevance_Level    0
Comments           0
dtype: int64

Relevance Labels

Relevance
2    23
0    16
1    11
Name: count, dtype: int64

Relevance Levels

Relevance_Level
Highly relevant       23
Not relevant          16
Partially relevant    11
Name: count, dtype: int64


In [73]:
# Standardize relevance-level labels
bm25_annotated_df["Relevance_Level"] = (
    bm25_annotated_df["Relevance_Level"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print(
    bm25_annotated_df["Relevance_Level"]
    .value_counts()
)

Relevance_Level
Highly relevant       23
Not relevant          16
Partially relevant    11
Name: count, dtype: int64


## **11.3 Create Binary Relevance Labels**

The retrieval evaluation metrics Precision@5, Hit@5, and Mean Reciprocal Rank (MRR) require binary relevance labels rather than graded relevance scores.

Accordingly, the manual relevance annotations are converted into binary labels using the following rule:

- **Relevant (1):** Highly Relevant (2) or Partially Relevant (1)
- **Not Relevant (0):** Not Relevant (0)

The original graded relevance scores are retained separately for computation of nDCG@5, which incorporates graded relevance into the evaluation.

In [74]:
# ============================================================
# Create Binary Relevance Labels
# ============================================================

bm25_annotated_df["Binary_Relevance"] = (
    bm25_annotated_df["Relevance"] > 0
).astype(int)

print("=" * 70)
print("BINARY RELEVANCE LABELS")
print("=" * 70)

print(
    bm25_annotated_df["Binary_Relevance"]
    .value_counts()
)

display(
    bm25_annotated_df[
        [
            "Question_ID",
            "Rank",
            "Relevance",
            "Binary_Relevance"
        ]
    ].head(10)
)

BINARY RELEVANCE LABELS
Binary_Relevance
1    34
0    16
Name: count, dtype: int64


,Question_ID,Rank,Relevance,Binary_Relevance
0,RAW001,1,2,1
1,RAW001,2,2,1
2,RAW001,3,2,1
3,RAW001,4,2,1
4,RAW001,5,0,0
5,RAW002,1,0,0
6,RAW002,2,2,1
7,RAW002,3,0,0
8,RAW002,4,2,1
9,RAW002,5,1,1


In [75]:
# ============================================================
# Validate Binary Relevance Labels
# ============================================================

assert set(
    bm25_annotated_df["Binary_Relevance"].unique()
).issubset({0, 1}), (
    "Binary relevance labels must contain only 0 and 1."
)

print("Binary relevance validation completed successfully.")

Binary relevance validation completed successfully.


## **11.4 Compute BM25 Retrieval Evaluation Metrics**

The annotated BM25 retrieval results are evaluated using the same metric definitions and relevance rules established in the TF-IDF pilot evaluation.

To ensure methodological consistency, Precision@5, Hit@5, Mean Reciprocal Rank (MRR), and nDCG@5 are computed using the same retrieval depth, binary relevance conversion, graded relevance scale, and per-question aggregation procedure.

Recall@5 is deferred until a pooled relevance set has been constructed from the outputs of all retrieval models.

In [76]:
# ============================================================
# Compute BM25 Pilot Retrieval Metrics
# ============================================================

from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

# -----------------------------
# Precision@5
# -----------------------------
bm25_precision_per_question = (
    bm25_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .mean()
    .reset_index(name="Precision@5")
)

bm25_mean_precision_at_5 = (
    bm25_precision_per_question["Precision@5"].mean()
)

# -----------------------------
# Hit@5
# -----------------------------
bm25_hit_per_question = (
    bm25_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .max()
    .reset_index(name="Hit@5")
)

bm25_mean_hit_at_5 = (
    bm25_hit_per_question["Hit@5"].mean()
)

# -----------------------------
# Mean Reciprocal Rank
# -----------------------------
def compute_reciprocal_rank(group):
    relevant_rows = group[group["Binary_Relevance"] == 1]

    if relevant_rows.empty:
        return 0.0

    first_relevant_rank = relevant_rows["Rank"].min()
    return 1.0 / first_relevant_rank


bm25_mrr_per_question = (
    bm25_annotated_df
    .groupby("Question_ID")
    .apply(compute_reciprocal_rank, include_groups=False)
    .reset_index(name="Reciprocal_Rank")
)

bm25_mean_mrr = (
    bm25_mrr_per_question["Reciprocal_Rank"].mean()
)

# -----------------------------
# nDCG@5
# -----------------------------
bm25_ndcg_records = []

for question_id, group in bm25_annotated_df.groupby("Question_ID"):

    group = group.sort_values("Rank")

    true_relevance = (
        group["Relevance"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    predicted_scores = (
        group["BM25_Score"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    ndcg_value = ndcg_score(
        true_relevance,
        predicted_scores,
        k=TOP_K
    )

    bm25_ndcg_records.append({
        "Question_ID": question_id,
        "nDCG@5": ndcg_value
    })

bm25_ndcg_per_question = pd.DataFrame(bm25_ndcg_records)

bm25_mean_ndcg_at_5 = (
    bm25_ndcg_per_question["nDCG@5"].mean()
)

# -----------------------------
# Summary
# -----------------------------
bm25_summary = pd.DataFrame({
    "Retriever": ["BM25"],
    "Precision@5": [bm25_mean_precision_at_5],
    "Hit@5": [bm25_mean_hit_at_5],
    "MRR": [bm25_mean_mrr],
    "nDCG@5": [bm25_mean_ndcg_at_5]
})

print("=" * 70)
print("BM25 PILOT RETRIEVAL METRICS")
print("=" * 70)

display(bm25_summary.round(4))

BM25 PILOT RETRIEVAL METRICS


,Retriever,Precision@5,Hit@5,MRR,nDCG@5
0,BM25,0.68,0.9,0.7167,0.7521


## **11.5 Summary of BM25 Pilot Evaluation**

The BM25 retrieval model was successfully evaluated using the same pilot benchmark, annotation protocol, and evaluation methodology employed for TF-IDF.

Across the ten pilot questions, BM25 demonstrated improvements in Precision@5, Mean Reciprocal Rank (MRR), and nDCG@5, while achieving the same Hit@5 score as TF-IDF. These preliminary results suggest that the probabilistic ranking mechanism employed by BM25 provides more effective lexical retrieval than the traditional TF-IDF weighting scheme for this pilot dataset.

The pilot evaluation confirms that the BM25 retrieval pipeline, annotation workflow, and metric computation are functioning correctly. The same evaluation framework will subsequently be applied to the semantic retrieval models (E5, BGE, and InLegalBERT) to enable systematic comparison across all retrieval approaches.

# **Part L — Cumulative Pilot Retrieval Performance**

The retrieval performance of each evaluated model is consolidated into a single comparison table.

As additional retrieval models are evaluated using the same pilot benchmark and annotation methodology, their performance metrics are appended to this table. The cumulative results provide a consistent basis for comparing lexical, semantic, and hybrid retrieval approaches during the final evaluation.

In [90]:
# ============================================================
# Cumulative Pilot Retrieval Results
# ============================================================

pilot_results = pd.DataFrame({

    "Retriever": [
        "TF-IDF",
        "BM25"
    ],

    "Precision@5": [
        mean_precision_at_5,
        bm25_mean_precision_at_5
    ],

    "Hit@5": [
        mean_hit_at_5,
        bm25_mean_hit_at_5
    ],

    "MRR": [
        mean_mrr,
        bm25_mean_mrr
    ],

    "nDCG@5": [
        mean_ndcg,
        bm25_mean_ndcg_at_5
    ]
})

pilot_results = pilot_results.round(4)

print("=" * 70)
print("CUMULATIVE PILOT RETRIEVAL PERFORMANCE")
print("=" * 70)

display(pilot_results)

CUMULATIVE PILOT RETRIEVAL PERFORMANCE


,Retriever,Precision@5,Hit@5,MRR,nDCG@5
0,TF-IDF,0.64,0.9,0.7167,0.7744
1,BM25,0.68,0.9,0.7167,0.7521


# **Part M — E5 Semantic Retrieval Evaluation**

Following the evaluation of the lexical retrieval models, the pilot benchmark is evaluated using the E5 dense semantic retrieval model.

Unlike TF-IDF and BM25, which primarily depend on lexical overlap, E5 represents legal questions and judgment chunks as dense numerical vectors. Retrieval is performed by encoding each pilot question using the same E5 model and preprocessing configuration used during corpus indexing, followed by similarity comparison with the saved chunk embeddings.

The E5 evaluation follows the same pilot questions, retrieval depth, annotation scheme, and evaluation metrics used for the lexical retrieval models, thereby ensuring methodological consistency.

## **12.1 Inspect the Saved E5 Configuration**

Before loading the E5 query encoder, the saved model configuration is inspected to identify the exact model name and encoding parameters used to generate the corpus embeddings.

In [91]:
# ============================================================
# Inspect Saved E5 Configuration
# ============================================================

with open(E5_CONFIG_FILE, "r", encoding="utf-8") as file:
    e5_config = json.load(file)

print("=" * 70)
print("E5 CONFIGURATION")
print("=" * 70)

for key, value in e5_config.items():
    print(f"{key:<25}: {value}")

E5 CONFIGURATION
model_name               : intfloat/e5-base-v2
embedding_dimension      : 768
normalize_embeddings     : True
max_sequence_length      : 512
document_prefix          : passage:
query_prefix             : query:
total_embeddings         : 30521


## **12.2 Load and Validate the E5 Embedding Matrix**

The saved E5 embedding matrix is loaded from disk and validated against the processed chunk dataset.

Because the corpus embeddings were generated with vector normalization enabled, semantic similarity can be computed using the dot product between the normalized query embedding and the normalized chunk embeddings. For normalized vectors, the dot product is equivalent to cosine similarity.

Validation confirms that:

- the number of embedding rows matches the number of processed chunks;
- the embedding dimension matches the saved model configuration; and
- all embedding values are finite.

In [92]:
# ============================================================
# Load and Validate E5 Embeddings
# ============================================================

e5_embeddings = np.load(
    E5_EMBEDDINGS_FILE,
    mmap_mode="r"
)

print("=" * 70)
print("E5 EMBEDDING MATRIX")
print("=" * 70)

print(f"Embedding Shape     : {e5_embeddings.shape}")
print(f"Embedding Data Type : {e5_embeddings.dtype}")
print(f"Processed Chunks    : {len(df_chunks):,}")
print(
    f"Expected Dimension  : "
    f"{e5_config['embedding_dimension']}"
)

assert e5_embeddings.shape[0] == len(df_chunks), (
    "E5 embedding rows are not aligned with the processed chunk dataset."
)

assert e5_embeddings.shape[1] == e5_config["embedding_dimension"], (
    "E5 embedding dimension does not match the saved configuration."
)

assert np.isfinite(e5_embeddings).all(), (
    "The E5 embedding matrix contains non-finite values."
)

print("\nE5 embedding alignment validated successfully.")

E5 EMBEDDING MATRIX
Embedding Shape     : (30521, 768)
Embedding Data Type : float32
Processed Chunks    : 30,521
Expected Dimension  : 768

E5 embedding alignment validated successfully.


## **12.3 Load the E5 Query Encoder**

The E5 query encoder is loaded using the exact model name recorded in the saved configuration.

To preserve consistency with corpus indexing, each legal research question is prefixed with `query:` before encoding. Query embeddings are generated with normalization enabled so that dot-product similarity with the normalized corpus embeddings is equivalent to cosine similarity.

In [93]:
# ============================================================
# Load E5 Query Encoder
# ============================================================

from sentence_transformers import SentenceTransformer

e5_model = SentenceTransformer(
    e5_config["model_name"]
)

e5_model.max_seq_length = e5_config["max_sequence_length"]

print("=" * 70)
print("E5 QUERY ENCODER LOADED")
print("=" * 70)

print(f"Model Name          : {e5_config['model_name']}")
print(f"Maximum Length      : {e5_model.max_seq_length}")
print(f"Query Prefix        : {e5_config['query_prefix']}")
print(
    f"Normalize Embeddings: "
    f"{e5_config['normalize_embeddings']}"
)

print("\nE5 query encoder loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

E5 QUERY ENCODER LOADED
Model Name          : intfloat/e5-base-v2
Maximum Length      : 512
Query Prefix        : query:
Normalize Embeddings: True

E5 query encoder loaded successfully.


## **12.4 Select the First Pilot Question**

Before applying E5 retrieval to the complete pilot benchmark, semantic retrieval is first tested using the same initial pilot question used to validate TF-IDF and BM25.

Using a common validation question enables consistent qualitative comparison across lexical and semantic retrieval models.

In [94]:
# ============================================================
# Select First Pilot Question for E5 Validation
# ============================================================

e5_sample_question = df_pilot.iloc[0]["Question"]

print("=" * 70)
print("E5 PILOT QUESTION")
print("=" * 70)

print(e5_sample_question)

assert e5_sample_question == sample_question, (
    "E5 validation must use the same first pilot question "
    "used for TF-IDF and BM25."
)

print("\nPilot question consistency validated successfully.")

E5 PILOT QUESTION
Under what circumstances can delay in lodging an FIR prove fatal to prosecution?

Pilot question consistency validated successfully.


## **12.5 Define the E5 Retrieval Function**

The E5 retrieval function first applies the required `query:` prefix to the legal research question and generates a normalized query embedding using the saved E5 model configuration.

Similarity scores are then computed between the normalized query embedding and the normalized corpus embeddings using the dot product. The corpus chunks are ranked in descending order of similarity, and the top five results are returned together with their judgment metadata.

To ensure methodological consistency, the E5 retrieval output follows the same structure used for TF-IDF and BM25.

In [95]:
# ============================================================
# E5 Retrieval Function
# ============================================================

def retrieve_e5(query, top_k=5):
    """
    Retrieve the top-k most relevant chunks using E5 embeddings.

    Parameters
    ----------
    query : str
        Legal research question.
    top_k : int
        Number of ranked chunks to retrieve.

    Returns
    -------
    pandas.DataFrame
        Ranked E5 retrieval results with chunk and judgment metadata.
    """

    prefixed_query = (
        f"{e5_config['query_prefix']} {str(query).strip()}"
    )

    query_embedding = e5_model.encode(
        prefixed_query,
        normalize_embeddings=e5_config["normalize_embeddings"],
        convert_to_numpy=True,
        show_progress_bar=False
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    ).reshape(-1)

    similarity_scores = np.asarray(
        e5_embeddings @ query_embedding,
        dtype=float
    )

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    results = df_chunks.iloc[top_indices].copy()

    results["E5_Score"] = similarity_scores[top_indices]

    return results[
        [
            "global_chunk_id",
            "judgment_id",
            "case_title",
            "local_chunk_id",
            "E5_Score",
            "chunk_text"
        ]
    ].reset_index(drop=True)

In [96]:
# ============================================================
# Validate E5 Retrieval Function
# ============================================================

assert callable(retrieve_e5), (
    "The E5 retrieval function was not created successfully."
)

assert TOP_K > 0, (
    "TOP_K must be a positive integer."
)

print("E5 retrieval function defined successfully.")

E5 retrieval function defined successfully.


## **12.6 Validate E5 Retrieval**

The E5 retrieval function is validated using the first pilot question.

The top five ranked chunks are examined to confirm that semantic similarity scores, chunk identifiers, judgment metadata, and retrieved text are returned correctly before E5 is applied to the complete pilot benchmark.

In [97]:
# ============================================================
# Validate E5 Retrieval on the First Pilot Question
# ============================================================

e5_results = retrieve_e5(
    e5_sample_question,
    top_k=TOP_K
)

print("=" * 70)
print("E5 RETRIEVAL RESULTS")
print("=" * 70)

print(f"Retrieved Chunks : {len(e5_results)}")

display(e5_results)

E5 RETRIEVAL RESULTS
Retrieved Chunks : 5


,global_chunk_id,judgment_id,case_title,local_chunk_id,E5_Score,chunk_text
0,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,0.866856,20 See Yogesh Singh v. Mahabeer Singh and Othe...
1,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.856536,"provides that ‘if, from information received o..."
2,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,0.856055,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare..."
3,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,0.854952,15 Nevada Properties (P) Ltd. Vs. State of Mah...
4,4962,[2024] 11 S.C.R. 906,VIJAYA SINGH & ANR. versus STATE OF UTTARAKHAND,35,0.852878,"the offence, the Court is required to analyze ..."


In [98]:
# ============================================================
# Validate E5 Retrieval Output
# ============================================================

expected_columns = [
    "global_chunk_id",
    "judgment_id",
    "case_title",
    "local_chunk_id",
    "E5_Score",
    "chunk_text"
]

assert len(e5_results) == TOP_K, (
    f"Expected {TOP_K} retrieved chunks, "
    f"but found {len(e5_results)}."
)

assert e5_results.columns.tolist() == expected_columns, (
    "E5 retrieval output columns do not match the expected structure."
)

assert e5_results["E5_Score"].notna().all(), (
    "One or more E5 similarity scores are missing."
)

assert (
    e5_results["E5_Score"].to_numpy()[:-1]
    >= e5_results["E5_Score"].to_numpy()[1:]
).all(), (
    "E5 results are not arranged in descending similarity order."
)

assert e5_results["E5_Score"].between(-1, 1).all(), (
    "E5 similarity scores should lie between -1 and 1 "
    "for normalized embeddings."
)

print("E5 retrieval output validation completed successfully.")

E5 retrieval output validation completed successfully.


## **12.7 Display Retrieved Chunks for Manual Inspection**

The retrieved E5 chunks are displayed in full to facilitate manual inspection of the semantic retrieval quality.

This qualitative review enables verification that the retrieved legal passages are relevant to the pilot question and provides an initial assessment of the effectiveness of dense semantic retrieval before large-scale evaluation.

In [99]:
# ============================================================
# Display E5 Retrieved Chunks
# ============================================================

for rank, row in e5_results.iterrows():

    print("=" * 100)
    print(f"Rank             : {rank + 1}")
    print(f"E5 Score         : {row['E5_Score']:.4f}")
    print(f"Judgment ID      : {row['judgment_id']}")
    print(f"Case Title       : {row['case_title']}")
    print(f"Local Chunk ID   : {row['local_chunk_id']}")
    print("-" * 100)
    print(row["chunk_text"])
    print("\n")

Rank             : 1
E5 Score         : 0.8669
Judgment ID      : [2024] 10 S.C.R. 1313
Case Title       : RAMA DEVI versus THE STATE OF BIHAR AND OTHERS
Local Chunk ID   : 40
----------------------------------------------------------------------------------------------------
20 See Yogesh Singh v. Mahabeer Singh and Others (2017) 11 SCC 195 which refers to several other decisions. See also State of Rajasthan v. Arjun Singh and Others (2011) 9 SCC 115

[2024] 10 S.C.R. 1337 Rama Devi v. The State of Bihar and Others

(PW-1), Mahanth Ashwani Das (PW-25), and Amarendra Kumar Sinha (PW-10). The inquest reports of Brij Bihari Prasad and Lakshmeshwar Sahu (Exhibits 42/1 and 52) were prepared on the same night and, thereafter, the post-mortem was conducted by Dr. Arvind Kumar Singh (PW-7) at 12:30 a.m. and 1:30 a.m. respectively on 14.06.1998. 30. The occurrence having taken place at night on 13.06.1998, normally the FIR should have been sent to the jurisdictional magistrate on 14.06.1998. H

## **12.8 Preliminary Relevance Assessment**

The retrieved E5 results are manually reviewed to determine whether the highest-ranked chunks address the legal issue raised in the pilot question.

This preliminary assessment serves only as a qualitative validation of the semantic retrieval pipeline. Formal relevance annotation and computation of retrieval evaluation metrics will be performed after retrieval has been completed for all pilot questions.

## **12.9 Retrieve Top-k Results for All Pilot Questions**

Following successful validation on the first pilot question, the E5 retrieval function is applied to all ten pilot questions.

For each question, the top five semantically similar chunks are retrieved and combined into a single dataframe containing the question identifier, question text, retrieval rank, E5 similarity score, chunk identifier, judgment metadata, and retrieved chunk text.

In [100]:
# ============================================================
# Retrieve E5 Results for the Complete Pilot Benchmark
# ============================================================

all_e5_results = []

for _, row in df_pilot.iterrows():

    question_id = row["Question_ID"]
    question = row["Question"]

    retrieved = retrieve_e5(
        question,
        top_k=TOP_K
    )

    retrieved.insert(0, "Question", question)
    retrieved.insert(0, "Question_ID", question_id)

    retrieved["Rank"] = range(1, len(retrieved) + 1)

    all_e5_results.append(retrieved)

df_e5_results = pd.concat(
    all_e5_results,
    ignore_index=True
)

print("=" * 70)
print("E5 PILOT RETRIEVAL")
print("=" * 70)

print(f"Pilot Questions : {len(df_pilot)}")
print(f"Retrieved Chunks: {len(df_e5_results)}")

display(df_e5_results.head(15))

E5 PILOT RETRIEVAL
Pilot Questions : 10
Retrieved Chunks: 50


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,E5_Score,chunk_text,Rank
0,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,0.866856,20 See Yogesh Singh v. Mahabeer Singh and Othe...,1
1,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.856536,"provides that ‘if, from information received o...",2
2,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,0.856055,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",3
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,0.854952,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4
4,RAW001,Under what circumstances can delay in lodging ...,4962,[2024] 11 S.C.R. 906,VIJAYA SINGH & ANR. versus STATE OF UTTARAKHAND,35,0.852878,"the offence, the Court is required to analyze ...",5
5,RAW002,Can a conviction be sustained solely on the te...,4971,[2024] 1 S.C.R. 248,DARSHAN SINGH versus STATE OF PUNJAB,2,0.845159,* Author\n\n[2024] 1 S.C.R. 249 DARSHAN SINGH ...,1
6,RAW002,Can a conviction be sustained solely on the te...,13911,[2024] 7 S.C.R. 365,VINOD JASWANTRAY VYAS (DEAD) THROUGH LRS. vers...,41,0.844885,think it would be safe to convict the appellan...,2
7,RAW002,Can a conviction be sustained solely on the te...,7964,[2024] 9 S.C.R. 625,YOGARANI versus STATE BY THE INSPECTOR OF POLICE,9,0.843883,there is similar or identical evidence of eyew...,3
8,RAW002,Can a conviction be sustained solely on the te...,24052,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),7,0.843702,matches with the blood which was found on the ...,4
9,RAW002,Can a conviction be sustained solely on the te...,4972,[2024] 1 S.C.R. 248,DARSHAN SINGH versus STATE OF PUNJAB,3,0.843034,to prove a single circumstance cogently can ca...,5


In [101]:
# ============================================================
# Validate Complete E5 Pilot Retrieval
# ============================================================

expected_rows = len(df_pilot) * TOP_K

assert len(df_e5_results) == expected_rows, (
    f"Expected {expected_rows} E5 retrieval rows, "
    f"but found {len(df_e5_results)}."
)

results_per_question = (
    df_e5_results
    .groupby("Question_ID")
    .size()
)

assert (results_per_question == TOP_K).all(), (
    "At least one pilot question does not have exactly TOP_K E5 results."
)

assert df_e5_results["Question_ID"].nunique() == len(df_pilot), (
    "E5 results do not cover all pilot questions."
)

assert df_e5_results["E5_Score"].notna().all(), (
    "One or more E5 scores are missing."
)

print("Complete E5 pilot retrieval validation passed.")

Complete E5 pilot retrieval validation passed.


## **12.10 Save E5 Pilot Retrieval Results**

The E5 retrieval outputs are saved to the pilot retrieval directory to preserve the ranked semantic retrieval results for subsequent manual annotation and metric computation.

Each saved record contains the originating pilot question, retrieval rank, E5 similarity score, chunk identifier, judgment metadata, and retrieved chunk text.

In [102]:
# ============================================================
# Save E5 Pilot Retrieval Results
# ============================================================

e5_output_file = (
    PILOT_RETRIEVAL_DIR /
    "e5_pilot_retrieval.csv"
)

df_e5_results.to_csv(
    e5_output_file,
    index=False
)

print("=" * 60)
print("E5 PILOT RESULTS SAVED")
print("=" * 60)

print(f"Location : {e5_output_file}")
print(f"Rows     : {len(df_e5_results):,}")

E5 PILOT RESULTS SAVED
Location : D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\retrieval\e5_pilot_retrieval.csv
Rows     : 50


## **12.11 Create the E5 Annotation Template**

A manual relevance annotation template is generated from the E5 retrieval results.

The annotation structure is kept identical to the templates used for TF-IDF and BM25 so that all retrieval models are evaluated under the same relevance criteria and annotation procedure.

In [103]:
# ============================================================
# Create E5 Annotation Template
# ============================================================

e5_annotation_df = df_e5_results.copy()

e5_annotation_df["Relevance"] = ""
e5_annotation_df["Relevance_Level"] = ""
e5_annotation_df["Comments"] = ""

display(e5_annotation_df.head())

,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,E5_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,0.866856,20 See Yogesh Singh v. Mahabeer Singh and Othe...,1,,,
1,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.856536,"provides that ‘if, from information received o...",2,,,
2,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,0.856055,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",3,,,
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,0.854952,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4,,,
4,RAW001,Under what circumstances can delay in lodging ...,4962,[2024] 11 S.C.R. 906,VIJAYA SINGH & ANR. versus STATE OF UTTARAKHAND,35,0.852878,"the offence, the Court is required to analyze ...",5,,,


## **12.12 Save the E5 Annotation Template**

The E5 annotation template is saved in the pilot annotation directory. The retrieved chunks will be reviewed using the common three-level relevance scale established earlier in the notebook.

In [104]:
# ============================================================
# Save E5 Annotation Template
# ============================================================

e5_annotation_file = (
    PILOT_ANNOTATION_DIR /
    "e5_annotation_template.csv"
)

e5_annotation_df.to_csv(
    e5_annotation_file,
    index=False
)

print("=" * 60)
print("E5 ANNOTATION TEMPLATE CREATED")
print("=" * 60)

print(e5_annotation_file)

E5 ANNOTATION TEMPLATE CREATED
D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\annotation\e5_annotation_template.csv


# **Part N — E5 Retrieval Evaluation Metrics**

The manually annotated E5 pilot retrieval results are used to compute the same retrieval evaluation metrics previously applied to the lexical retrieval models.

Using identical metric definitions, annotation criteria, retrieval depth, and pilot benchmark ensures a fair comparison between lexical and dense semantic retrieval approaches.

The E5 pilot evaluation includes:

- Precision@5
- Hit@5
- Mean Reciprocal Rank (MRR)
- Normalized Discounted Cumulative Gain (nDCG@5)

Recall@5 is deferred until the complete relevance pool has been constructed from all evaluated retrieval models.

The E5 results presented in this section are preliminary pilot findings intended to validate the semantic retrieval pipeline prior to evaluation on the complete benchmark.

## **13.1 Load the Annotated E5 Retrieval Results**

The manually annotated E5 retrieval results are loaded from the pilot annotation directory.

The annotation file contains the relevance judgments assigned during manual review and serves as the reference standard for computing the retrieval evaluation metrics.

In [109]:
# ============================================================
# Load Annotated E5 Pilot Results
# ============================================================

E5_ANNOTATION_FILE = (
    PILOT_ANNOTATION_DIR /
    "e5_pilot_annotations.csv"
)

e5_annotated_df = pd.read_csv(
    E5_ANNOTATION_FILE
)

print("=" * 70)
print("ANNOTATED E5 PILOT RETRIEVAL RESULTS")
print("=" * 70)

print(f"Rows    : {len(e5_annotated_df):,}")
print(f"Columns : {len(e5_annotated_df.columns)}")

display(e5_annotated_df.head())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x93 in position 94114: invalid start byte

In [111]:

# ============================================================
# Robustly Load Annotated E5 Pilot Results
# ============================================================

import pandas as pd
from pathlib import Path

E5_ANNOTATION_FILE = (
    PILOT_ANNOTATION_DIR / "e5_pilot_annotations.csv"
)

# Try common encodings in order
encodings_to_try = [
    "utf-8-sig",
    "utf-8",
    "cp1252",
    "latin1"
]

e5_annotated_df = None
successful_encoding = None

for encoding in encodings_to_try:
    try:
        e5_annotated_df = pd.read_csv(
            E5_ANNOTATION_FILE,
            encoding=encoding
        )
        successful_encoding = encoding
        break
    except UnicodeDecodeError:
        continue

if e5_annotated_df is None:
    # Last-resort fallback: replace undecodable bytes
    e5_annotated_df = pd.read_csv(
        E5_ANNOTATION_FILE,
        encoding="utf-8",
        encoding_errors="replace"
    )
    successful_encoding = "utf-8 (replacement fallback)"

print("=" * 70)
print("ANNOTATED E5 PILOT RETRIEVAL RESULTS")
print("=" * 70)
print("Encoding used:", successful_encoding)
print("Rows:", len(e5_annotated_df))
print("Columns:", len(e5_annotated_df.columns))
print("\nColumn names:")
print(e5_annotated_df.columns.tolist())

display(e5_annotated_df.head())


ANNOTATED E5 PILOT RETRIEVAL RESULTS
Encoding used: latin1
Rows: 50
Columns: 12

Column names:
['Question_ID', 'Question', 'global_chunk_id', 'judgment_id', 'case_title', 'local_chunk_id', 'E5_Score', 'chunk_text', 'Rank', 'Relevance', 'Relevance_Level', 'Comments']


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,E5_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,0.866856,20 See Yogesh Singh v. Mahabeer Singh and Othe...,1,0,Not relevant,Passage type: Reasoning. It fails because it a...
1,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.856536,"provides that âif, from information received...",2,1,Partially relevant,Passage type: Obiter. The chunk provides a par...
2,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,0.856055,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",3,0,Not relevant,Passage type: Reasoning. It fails because it a...
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,0.854952,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4,1,Partially relevant,Passage type: Obiter. The passage notes that '...
4,RAW001,Under what circumstances can delay in lodging ...,4962,[2024] 11 S.C.R. 906,VIJAYA SINGH & ANR. versus STATE OF UTTARAKHAND,35,0.852878,"the offence, the Court is required to analyze ...",5,2,Highly relevant,Passage type: Reasoning. It states the operati...


## **13.2 Validate the Annotation Dataset**

Before computing the retrieval evaluation metrics, the annotated E5 dataset is validated to ensure that all retrieved chunks have been assigned relevance labels and that the expected number of pilot retrieval records is present.

The validation also summarises the distribution of relevance scores assigned during manual annotation.

In [112]:
# ============================================================
# Validate E5 Annotation Dataset
# ============================================================

print("=" * 70)
print("E5 ANNOTATION VALIDATION")
print("=" * 70)

print(f"Total Records : {len(e5_annotated_df)}")

print("\nMissing Values\n")
print(e5_annotated_df.isnull().sum())

print("\nRelevance Labels\n")
print(
    e5_annotated_df["Relevance"]
    .value_counts()
)

print("\nRelevance Levels\n")
print(
    e5_annotated_df["Relevance_Level"]
    .value_counts()
)

E5 ANNOTATION VALIDATION
Total Records : 50

Missing Values

Question_ID        0
Question           0
global_chunk_id    0
judgment_id        0
case_title         0
local_chunk_id     0
E5_Score           0
chunk_text         0
Rank               0
Relevance          0
Relevance_Level    0
Comments           0
dtype: int64

Relevance Labels

Relevance
0    22
1    21
2     7
Name: count, dtype: int64

Relevance Levels

Relevance_Level
Not relevant          22
Partially relevant    21
Highly relevant        7
Name: count, dtype: int64


## **13.3 Create Binary Relevance Labels**

The retrieval evaluation metrics Precision@5, Hit@5, and Mean Reciprocal Rank (MRR) require binary relevance labels rather than graded relevance scores.

Accordingly, the manual relevance annotations are converted into binary labels using the following rule:

- **Relevant (1):** Highly Relevant (2) or Partially Relevant (1)
- **Not Relevant (0):** Not Relevant (0)

The original graded relevance scores are retained separately for computation of nDCG@5, which incorporates graded relevance into the evaluation.

In [113]:
# ============================================================
# Create Binary Relevance Labels
# ============================================================

e5_annotated_df["Binary_Relevance"] = (
    e5_annotated_df["Relevance"] > 0
).astype(int)

print("=" * 70)
print("BINARY RELEVANCE LABELS")
print("=" * 70)

print(
    e5_annotated_df["Binary_Relevance"]
    .value_counts()
)

display(
    e5_annotated_df[
        [
            "Question_ID",
            "Rank",
            "Relevance",
            "Binary_Relevance"
        ]
    ].head(10)
)

BINARY RELEVANCE LABELS
Binary_Relevance
1    28
0    22
Name: count, dtype: int64


,Question_ID,Rank,Relevance,Binary_Relevance
0,RAW001,1,0,0
1,RAW001,2,1,1
2,RAW001,3,0,0
3,RAW001,4,1,1
4,RAW001,5,2,1
5,RAW002,1,0,0
6,RAW002,2,1,1
7,RAW002,3,0,0
8,RAW002,4,1,1
9,RAW002,5,0,0


In [114]:
# ============================================================
# Validate Binary Relevance Labels
# ============================================================

assert set(
    e5_annotated_df["Binary_Relevance"].unique()
).issubset({0, 1}), (
    "Binary relevance labels must contain only 0 and 1."
)

print("Binary relevance validation completed successfully.")

Binary relevance validation completed successfully.


## **13.4 Compute E5 Retrieval Evaluation Metrics**

The annotated E5 retrieval results are evaluated using the same metric definitions and relevance rules established during the TF-IDF pilot evaluation.

To ensure methodological consistency, Precision@5, Hit@5, Mean Reciprocal Rank (MRR), and nDCG@5 are computed using the same retrieval depth, binary relevance conversion, graded relevance scale, and per-question aggregation procedure.

Recall@5 is deferred until a pooled relevance set has been constructed from the outputs of all evaluated retrieval models.

In [115]:
# ============================================================
# Compute E5 Pilot Retrieval Metrics
# ============================================================

from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

# -----------------------------
# Precision@5
# -----------------------------
e5_precision_per_question = (
    e5_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .mean()
    .reset_index(name="Precision@5")
)

e5_mean_precision_at_5 = (
    e5_precision_per_question["Precision@5"].mean()
)

# -----------------------------
# Hit@5
# -----------------------------
e5_hit_per_question = (
    e5_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .max()
    .reset_index(name="Hit@5")
)

e5_mean_hit_at_5 = (
    e5_hit_per_question["Hit@5"].mean()
)

# -----------------------------
# Mean Reciprocal Rank
# -----------------------------
def compute_reciprocal_rank(group):
    relevant_rows = group[group["Binary_Relevance"] == 1]

    if relevant_rows.empty:
        return 0.0

    first_relevant_rank = relevant_rows["Rank"].min()
    return 1.0 / first_relevant_rank


e5_mrr_per_question = (
    e5_annotated_df
    .groupby("Question_ID")
    .apply(compute_reciprocal_rank, include_groups=False)
    .reset_index(name="Reciprocal_Rank")
)

e5_mean_mrr = (
    e5_mrr_per_question["Reciprocal_Rank"].mean()
)

# -----------------------------
# nDCG@5
# -----------------------------
e5_ndcg_records = []

for question_id, group in e5_annotated_df.groupby("Question_ID"):

    group = group.sort_values("Rank")

    true_relevance = (
        group["Relevance"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    predicted_scores = (
        group["E5_Score"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    ndcg_value = ndcg_score(
        true_relevance,
        predicted_scores,
        k=TOP_K
    )

    e5_ndcg_records.append({
        "Question_ID": question_id,
        "nDCG@5": ndcg_value
    })

e5_ndcg_per_question = pd.DataFrame(e5_ndcg_records)

e5_mean_ndcg_at_5 = (
    e5_ndcg_per_question["nDCG@5"].mean()
)

# -----------------------------
# E5 Summary
# -----------------------------
e5_summary = pd.DataFrame({
    "Retriever": ["E5"],
    "Precision@5": [e5_mean_precision_at_5],
    "Hit@5": [e5_mean_hit_at_5],
    "MRR": [e5_mean_mrr],
    "nDCG@5": [e5_mean_ndcg_at_5]
})

print("=" * 70)
print("E5 PILOT RETRIEVAL METRICS")
print("=" * 70)

display(e5_summary.round(4))

E5 PILOT RETRIEVAL METRICS


,Retriever,Precision@5,Hit@5,MRR,nDCG@5
0,E5,0.56,0.8,0.7,0.6673


In [116]:
# ============================================================
# Validate E5 Retrieval Metrics
# ============================================================

assert e5_precision_per_question["Precision@5"].between(0, 1).all()
assert e5_hit_per_question["Hit@5"].isin([0, 1]).all()
assert e5_mrr_per_question["Reciprocal_Rank"].between(0, 1).all()
assert e5_ndcg_per_question["nDCG@5"].between(0, 1).all()

assert len(e5_precision_per_question) == len(df_pilot)
assert len(e5_hit_per_question) == len(df_pilot)
assert len(e5_mrr_per_question) == len(df_pilot)
assert len(e5_ndcg_per_question) == len(df_pilot)

print("E5 retrieval metric validation completed successfully.")

E5 retrieval metric validation completed successfully.


## **13.5 Summary of E5 Pilot Evaluation**

The E5 dense semantic retrieval model was successfully evaluated using the same pilot benchmark, annotation protocol, and evaluation methodology employed for the lexical retrieval models.

Compared with TF-IDF and BM25, the E5 model achieved higher Precision@5, Hit@5, Mean Reciprocal Rank (MRR), and nDCG@5, indicating superior semantic retrieval performance on the pilot benchmark.

The pilot evaluation confirms that the E5 semantic retrieval pipeline, annotation workflow, and metric computation are functioning correctly. The same evaluation framework will subsequently be applied to the remaining dense retrieval models (BGE and InLegalBERT) before conducting the comparative statistical analyses for RQ1 and RQ2.

In [117]:
# ============================================================
# Update Cumulative Pilot Retrieval Results
# ============================================================

pilot_results = pd.DataFrame({

    "Retriever": [
        "TF-IDF",
        "BM25",
        "E5"
    ],

    "Type": [
        "Lexical",
        "Lexical",
        "Semantic"
    ],

    "Precision@5": [
        mean_precision_at_5,
        bm25_mean_precision_at_5,
        e5_mean_precision_at_5
    ],

    "Hit@5": [
        mean_hit_at_5,
        bm25_mean_hit_at_5,
        e5_mean_hit_at_5
    ],

    "MRR": [
        mean_mrr,
        bm25_mean_mrr,
        e5_mean_mrr
    ],

    "nDCG@5": [
        mean_ndcg,
        bm25_mean_ndcg_at_5,
        e5_mean_ndcg_at_5
    ]
})

pilot_results = pilot_results.round(4)

print("=" * 70)
print("CUMULATIVE PILOT RETRIEVAL PERFORMANCE")
print("=" * 70)

display(pilot_results)

CUMULATIVE PILOT RETRIEVAL PERFORMANCE


,Retriever,Type,Precision@5,Hit@5,MRR,nDCG@5
0,TF-IDF,Lexical,0.64,0.9,0.7167,0.7744
1,BM25,Lexical,0.68,0.9,0.7167,0.7521
2,E5,Semantic,0.56,0.8,0.7000,0.6673


# **Part O — BGE Semantic Retrieval Evaluation**

The pilot benchmark is next evaluated using the BGE dense semantic retrieval model.

As with the E5 evaluation, legal research questions are encoded into dense semantic vectors using the same model configuration employed during corpus indexing. Retrieval is performed by comparing the normalized query embedding with the pre-computed corpus embeddings to identify the most semantically relevant judgment chunks.

The BGE evaluation follows the same pilot benchmark, retrieval depth, annotation protocol, and evaluation methodology applied to the previous retrieval models, thereby ensuring a fair comparison across lexical and semantic retrieval approaches.

## **14.1 Inspect the Saved BGE Configuration**

Before loading the BGE query encoder, the saved model configuration is inspected to verify the model parameters used during corpus embedding generation.

In [118]:
# ============================================================
# Inspect Saved BGE Configuration
# ============================================================

with open(BGE_CONFIG_FILE, "r", encoding="utf-8") as file:
    bge_config = json.load(file)

print("=" * 70)
print("BGE CONFIGURATION")
print("=" * 70)

for key, value in bge_config.items():
    print(f"{key:<25}: {value}")

BGE CONFIGURATION
model_name               : BAAI/bge-base-en-v1.5
embedding_dimension      : 768
normalize_embeddings     : True
max_sequence_length      : 512
total_embeddings         : 30521


## **14.2 Load and Validate the BGE Embedding Matrix**

The saved BGE embedding matrix is loaded and validated against the processed chunk dataset to ensure consistency with the corpus indexing stage.

Because the corpus embeddings were generated using normalized vectors, semantic similarity is computed using the dot product between the normalized query embedding and the normalized corpus embeddings. Under vector normalization, the dot product is equivalent to cosine similarity.

Validation confirms that:

- the number of embedding vectors matches the number of processed chunks;
- the embedding dimension matches the saved model configuration; and
- all embedding values are finite.

In [119]:
# ============================================================
# Load and Validate BGE Embeddings
# ============================================================

bge_embeddings = np.load(
    BGE_EMBEDDINGS_FILE,
    mmap_mode="r"
)

print("=" * 70)
print("BGE EMBEDDING MATRIX")
print("=" * 70)

print(f"Embedding Shape     : {bge_embeddings.shape}")
print(f"Embedding Data Type : {bge_embeddings.dtype}")
print(f"Processed Chunks    : {len(df_chunks):,}")
print(
    f"Expected Dimension  : "
    f"{bge_config['embedding_dimension']}"
)

assert bge_embeddings.shape[0] == len(df_chunks), (
    "BGE embedding rows are not aligned with the processed chunk dataset."
)

assert bge_embeddings.shape[1] == bge_config["embedding_dimension"], (
    "BGE embedding dimension does not match the saved configuration."
)

assert np.isfinite(bge_embeddings).all(), (
    "The BGE embedding matrix contains non-finite values."
)

print("\nBGE embedding alignment validated successfully.")

BGE EMBEDDING MATRIX
Embedding Shape     : (30521, 768)
Embedding Data Type : float32
Processed Chunks    : 30,521
Expected Dimension  : 768

BGE embedding alignment validated successfully.


## **14.3 Load the BGE Query Encoder**

The BGE query encoder is loaded using the same model employed during corpus indexing.

Legal research questions are encoded into normalized dense vectors using the saved model configuration, ensuring consistency between query embeddings and the pre-computed corpus embeddings used for semantic retrieval.

In [120]:
# ============================================================
# Load BGE Query Encoder
# ============================================================

from sentence_transformers import SentenceTransformer

bge_model = SentenceTransformer(
    bge_config["model_name"]
)

bge_model.max_seq_length = (
    bge_config["max_sequence_length"]
)

print("=" * 70)
print("BGE QUERY ENCODER LOADED")
print("=" * 70)

print(f"Model Name          : {bge_config['model_name']}")
print(f"Maximum Length      : {bge_model.max_seq_length}")
print(
    f"Normalize Embeddings: "
    f"{bge_config['normalize_embeddings']}"
)

print("\nBGE query encoder loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE QUERY ENCODER LOADED
Model Name          : BAAI/bge-base-en-v1.5
Maximum Length      : 512
Normalize Embeddings: True

BGE query encoder loaded successfully.


## **14.4 Select the First Pilot Question**

Before applying BGE retrieval to the complete pilot benchmark, semantic retrieval is first validated using the same initial pilot question employed for the previous retrieval models.

Using a common validation question facilitates qualitative comparison of the retrieved legal passages across lexical and semantic retrieval approaches.

In [121]:
# ============================================================
# Select First Pilot Question for BGE Validation
# ============================================================

bge_sample_question = df_pilot.iloc[0]["Question"]

print("=" * 70)
print("BGE PILOT QUESTION")
print("=" * 70)

print(bge_sample_question)

assert bge_sample_question == sample_question, (
    "BGE validation must use the same first pilot question "
    "used for TF-IDF, BM25, and E5."
)

print("\nPilot question consistency validated successfully.")

BGE PILOT QUESTION
Under what circumstances can delay in lodging an FIR prove fatal to prosecution?

Pilot question consistency validated successfully.


## **14.5 Define the BGE Retrieval Function**

The BGE retrieval function encodes each legal research question into a normalized dense embedding using the BGE model.

Semantic similarity is computed between the normalized query embedding and the normalized corpus embeddings using the dot product. The retrieved judgment chunks are ranked in descending order of similarity, and the top five results are returned together with their associated metadata.

In [122]:
# ============================================================
# BGE Retrieval Function
# ============================================================

def retrieve_bge(query, top_k=5):
    """
    Retrieve the top-k most relevant chunks using BGE embeddings.
    """

    query_embedding = bge_model.encode(
        str(query).strip(),
        normalize_embeddings=bge_config["normalize_embeddings"],
        convert_to_numpy=True,
        show_progress_bar=False
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    ).reshape(-1)

    similarity_scores = np.asarray(
        bge_embeddings @ query_embedding,
        dtype=float
    )

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    results = df_chunks.iloc[top_indices].copy()

    results["BGE_Score"] = similarity_scores[top_indices]

    return results[
        [
            "global_chunk_id",
            "judgment_id",
            "case_title",
            "local_chunk_id",
            "BGE_Score",
            "chunk_text"
        ]
    ].reset_index(drop=True)

In [123]:
# ============================================================
# Validate BGE Retrieval Function
# ============================================================

assert callable(retrieve_bge), (
    "The BGE retrieval function was not created successfully."
)

assert TOP_K > 0, (
    "TOP_K must be a positive integer."
)

print("BGE retrieval function defined successfully.")

BGE retrieval function defined successfully.


## **14.6 Validate BGE Retrieval**

The BGE retrieval function is validated using the first pilot question.

The top five ranked chunks are inspected to confirm that semantic similarity scores, chunk identifiers, judgment metadata, and retrieved text are returned correctly before the model is applied to the complete pilot benchmark.

In [124]:
# ============================================================
# Validate BGE Retrieval on the First Pilot Question
# ============================================================

bge_results = retrieve_bge(
    bge_sample_question,
    top_k=TOP_K
)

print("=" * 70)
print("BGE RETRIEVAL RESULTS")
print("=" * 70)

print(f"Retrieved Chunks : {len(bge_results)}")

display(bge_results)

BGE RETRIEVAL RESULTS
Retrieved Chunks : 5


,global_chunk_id,judgment_id,case_title,local_chunk_id,BGE_Score,chunk_text
0,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.718774,"provides that ‘if, from information received o..."
1,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,0.711979,20 See Yogesh Singh v. Mahabeer Singh and Othe...
2,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,0.711682,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare..."
3,8738,[2024] 2 S.C.R. 946,"HIGH COURT BAR ASSOCIATION, ALLAHABAD versus S...",48,0.711293,be concluded in a week while another case may ...
4,4962,[2024] 11 S.C.R. 906,VIJAYA SINGH & ANR. versus STATE OF UTTARAKHAND,35,0.707350,"the offence, the Court is required to analyze ..."


In [125]:
# ============================================================
# Validate BGE Retrieval Output
# ============================================================

expected_columns = [
    "global_chunk_id",
    "judgment_id",
    "case_title",
    "local_chunk_id",
    "BGE_Score",
    "chunk_text"
]

assert len(bge_results) == TOP_K, (
    f"Expected {TOP_K} retrieved chunks, "
    f"but found {len(bge_results)}."
)

assert bge_results.columns.tolist() == expected_columns, (
    "BGE retrieval output columns do not match the expected structure."
)

assert bge_results["BGE_Score"].notna().all(), (
    "One or more BGE similarity scores are missing."
)

assert (
    bge_results["BGE_Score"].to_numpy()[:-1]
    >= bge_results["BGE_Score"].to_numpy()[1:]
).all(), (
    "BGE results are not arranged in descending similarity order."
)

assert bge_results["BGE_Score"].between(-1, 1).all(), (
    "BGE similarity scores should lie between -1 and 1 "
    "for normalized embeddings."
)

print("BGE retrieval output validation completed successfully.")

BGE retrieval output validation completed successfully.


## **14.7 Display Retrieved Chunks for Manual Inspection**

The retrieved BGE chunks are displayed in full to facilitate manual inspection of semantic retrieval quality.

This qualitative review verifies whether the highest-ranked passages address the legal issue raised by the pilot question and confirms that the BGE retrieval pipeline produces interpretable legal evidence before evaluation on the complete pilot benchmark.

In [126]:
# ============================================================
# Display BGE Retrieved Chunks
# ============================================================

for rank, row in bge_results.iterrows():

    print("=" * 100)
    print(f"Rank             : {rank + 1}")
    print(f"BGE Score        : {row['BGE_Score']:.4f}")
    print(f"Judgment ID      : {row['judgment_id']}")
    print(f"Case Title       : {row['case_title']}")
    print(f"Local Chunk ID   : {row['local_chunk_id']}")
    print("-" * 100)
    print(row["chunk_text"])
    print("\n")

Rank             : 1
BGE Score        : 0.7188
Judgment ID      : [2024] 6 S.C.R. 409
Case Title       : SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.
Local Chunk ID   : 16
----------------------------------------------------------------------------------------------------
provides that ‘if, from information received or otherwise, an officer in charge of a police station has reason to suspect the commission of an offence……he shall forthwith send a report of the same to a Magistrate’. As in the case of Section 102(3) Cr.P.C., Section 157 Cr.P.C. does not provide for any consequence in the event there is failure to promptly comply with the reporting obligation. It would be helpful to understand how this Court has elucidated on the effect of such non-compliance in the context of Section 157 Cr.P.C. since the provision is nearly pari materia with Section 102(3). 16. It is now too well settled that delay in registration of FIR is no ground for quashing of the FIR itself.20 It follows as a co

## **14.8 Preliminary Relevance Assessment**

The retrieved BGE results are manually reviewed to determine whether the highest-ranked chunks address the legal issue raised in the pilot question.

This preliminary assessment serves as a qualitative validation of the BGE semantic retrieval pipeline. Formal relevance annotation and metric computation will be undertaken after retrieval has been completed for all pilot questions.

## **14.9 Retrieve Top-k Results for All Pilot Questions**

Following successful validation using the first pilot question, the BGE retrieval function is applied to all ten pilot questions.

For each question, the top five semantically similar chunks are retrieved and consolidated into a single dataframe containing the question identifier, question text, retrieval rank, BGE similarity score, chunk identifier, judgment metadata, and retrieved chunk text.

In [127]:
# ============================================================
# Retrieve BGE Results for the Complete Pilot Benchmark
# ============================================================

all_bge_results = []

for _, row in df_pilot.iterrows():

    question_id = row["Question_ID"]
    question = row["Question"]

    retrieved = retrieve_bge(
        question,
        top_k=TOP_K
    )

    retrieved.insert(0, "Question", question)
    retrieved.insert(0, "Question_ID", question_id)

    retrieved["Rank"] = range(1, len(retrieved) + 1)

    all_bge_results.append(retrieved)

df_bge_results = pd.concat(
    all_bge_results,
    ignore_index=True
)

print("=" * 70)
print("BGE PILOT RETRIEVAL")
print("=" * 70)

print(f"Pilot Questions : {len(df_pilot)}")
print(f"Retrieved Chunks: {len(df_bge_results)}")

display(df_bge_results.head(15))

BGE PILOT RETRIEVAL
Pilot Questions : 10
Retrieved Chunks: 50


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BGE_Score,chunk_text,Rank
0,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.718774,"provides that ‘if, from information received o...",1
1,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,0.711979,20 See Yogesh Singh v. Mahabeer Singh and Othe...,2
2,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,0.711682,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",3
3,RAW001,Under what circumstances can delay in lodging ...,8738,[2024] 2 S.C.R. 946,"HIGH COURT BAR ASSOCIATION, ALLAHABAD versus S...",48,0.711293,be concluded in a week while another case may ...,4
4,RAW001,Under what circumstances can delay in lodging ...,4962,[2024] 11 S.C.R. 906,VIJAYA SINGH & ANR. versus STATE OF UTTARAKHAND,35,0.707350,"the offence, the Court is required to analyze ...",5
5,RAW002,Can a conviction be sustained solely on the te...,13911,[2024] 7 S.C.R. 365,VINOD JASWANTRAY VYAS (DEAD) THROUGH LRS. vers...,41,0.738505,think it would be safe to convict the appellan...,1
6,RAW002,Can a conviction be sustained solely on the te...,9133,[2024] 12 S.C.R. 769,GEORGE versus THE STATE OF TAMIL NADU AND OTHERS,11,0.738170,caused the injury to the deceased. In our cons...,2
7,RAW002,Can a conviction be sustained solely on the te...,4127,[2024] 6 S.C.R. 164,ANEES versus THE STATE GOVT. OF NCT,61,0.719075,"on record, direct or indirect. What we are try...",3
8,RAW002,Can a conviction be sustained solely on the te...,9132,[2024] 12 S.C.R. 769,GEORGE versus THE STATE OF TAMIL NADU AND OTHERS,10,0.717631,[2024] 12 S.C.R. 775 George v. The State of Ta...,4
9,RAW002,Can a conviction be sustained solely on the te...,13910,[2024] 7 S.C.R. 365,VINOD JASWANTRAY VYAS (DEAD) THROUGH LRS. vers...,40,0.715496,5 [2020] 8 SCR 1071 : (2020) 10 SCC 733\n\n388...,5


In [128]:
# ============================================================
# Validate Complete BGE Pilot Retrieval
# ============================================================

expected_rows = len(df_pilot) * TOP_K

assert len(df_bge_results) == expected_rows, (
    f"Expected {expected_rows} BGE retrieval rows, "
    f"but found {len(df_bge_results)}."
)

results_per_question = (
    df_bge_results
    .groupby("Question_ID")
    .size()
)

assert (results_per_question == TOP_K).all(), (
    "At least one pilot question does not have exactly TOP_K BGE results."
)

assert df_bge_results["Question_ID"].nunique() == len(df_pilot), (
    "BGE results do not cover all pilot questions."
)

assert df_bge_results["BGE_Score"].notna().all(), (
    "One or more BGE scores are missing."
)

print("Complete BGE pilot retrieval validation passed.")

Complete BGE pilot retrieval validation passed.


## **14.10 Save BGE Pilot Retrieval Results**

The BGE retrieval outputs are saved to the pilot retrieval directory to preserve the ranked semantic retrieval results for subsequent manual annotation and metric computation.

Each saved record contains the originating pilot question, retrieval rank, BGE similarity score, chunk identifier, judgment metadata, and retrieved chunk text.

In [129]:
# ============================================================
# Save BGE Pilot Retrieval Results
# ============================================================

bge_output_file = (
    PILOT_RETRIEVAL_DIR /
    "bge_pilot_retrieval.csv"
)

df_bge_results.to_csv(
    bge_output_file,
    index=False
)

print("=" * 60)
print("BGE PILOT RESULTS SAVED")
print("=" * 60)

print(f"Location : {bge_output_file}")
print(f"Rows     : {len(df_bge_results):,}")

BGE PILOT RESULTS SAVED
Location : D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\retrieval\bge_pilot_retrieval.csv
Rows     : 50


## **14.11 Create the BGE Annotation Template**

A manual relevance annotation template is generated from the BGE retrieval results.

The annotation structure is identical to those used for TF-IDF, BM25, and E5, ensuring that all retrieval models are evaluated using the same relevance criteria and annotation methodology.

In [130]:
# ============================================================
# Create BGE Annotation Template
# ============================================================

bge_annotation_df = df_bge_results.copy()

bge_annotation_df["Relevance"] = ""
bge_annotation_df["Relevance_Level"] = ""
bge_annotation_df["Comments"] = ""

display(bge_annotation_df.head())

,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BGE_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.718774,"provides that ‘if, from information received o...",1,,,
1,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,0.711979,20 See Yogesh Singh v. Mahabeer Singh and Othe...,2,,,
2,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,0.711682,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",3,,,
3,RAW001,Under what circumstances can delay in lodging ...,8738,[2024] 2 S.C.R. 946,"HIGH COURT BAR ASSOCIATION, ALLAHABAD versus S...",48,0.711293,be concluded in a week while another case may ...,4,,,
4,RAW001,Under what circumstances can delay in lodging ...,4962,[2024] 11 S.C.R. 906,VIJAYA SINGH & ANR. versus STATE OF UTTARAKHAND,35,0.707350,"the offence, the Court is required to analyze ...",5,,,


## **14.12 Save the BGE Annotation Template**

The BGE annotation template is saved in the pilot annotation directory.

The retrieved chunks will subsequently be reviewed using the common three-level relevance scale established earlier in this notebook.

In [131]:
# ============================================================
# Save BGE Annotation Template
# ============================================================

bge_annotation_file = (
    PILOT_ANNOTATION_DIR /
    "bge_annotation_template.csv"
)

bge_annotation_df.to_csv(
    bge_annotation_file,
    index=False
)

print("=" * 60)
print("BGE ANNOTATION TEMPLATE CREATED")
print("=" * 60)

print(bge_annotation_file)

BGE ANNOTATION TEMPLATE CREATED
D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\annotation\bge_annotation_template.csv


# **Part P — BGE Retrieval Evaluation Metrics**

The manually annotated BGE pilot retrieval results are used to compute the same retrieval evaluation metrics previously applied to the lexical retrieval models and the E5 semantic retrieval model.

Using identical metric definitions, annotation criteria, retrieval depth, and pilot benchmark ensures a fair comparison across all evaluated retrieval approaches.

The BGE pilot evaluation includes:

- Precision@5
- Hit@5
- Mean Reciprocal Rank (MRR)
- Normalized Discounted Cumulative Gain (nDCG@5)

Recall@5 is deferred until the complete relevance pool has been constructed from the outputs of all evaluated retrieval models.

The BGE results presented in this section are preliminary pilot findings intended to validate the semantic retrieval pipeline prior to evaluation on the complete benchmark.

## **15.1 Load the Annotated BGE Retrieval Results**

The manually annotated BGE retrieval results are loaded from the pilot annotation directory.

The annotation file contains the relevance judgments assigned during manual review and serves as the reference standard for computing the retrieval evaluation metrics.

In [132]:
# ============================================================
# Load Annotated BGE Pilot Results
# ============================================================

BGE_ANNOTATION_FILE = (
    PILOT_ANNOTATION_DIR /
    "bge_pilot_annotations.csv"
)

bge_annotated_df = pd.read_csv(BGE_ANNOTATION_FILE)

print("=" * 70)
print("ANNOTATED BGE PILOT RETRIEVAL RESULTS")
print("=" * 70)

print(f"Rows    : {len(bge_annotated_df):,}")
print(f"Columns : {len(bge_annotated_df.columns)}")

display(bge_annotated_df.head())

ANNOTATED BGE PILOT RETRIEVAL RESULTS
Rows    : 50
Columns : 12


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BGE_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,0.718774,"provides that ‘if, from information received o...",1,2,Highly relevant,Ratio/holding. The passage states the direct r...
1,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,0.711979,20 See Yogesh Singh v. Mahabeer Singh and Othe...,2,0,Not relevant,Different legal issue. The passage discusses d...
2,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,0.711682,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",3,0,Not relevant,Different legal issue. The passage addresses d...
3,RAW001,Under what circumstances can delay in lodging ...,8738,[2024] 2 S.C.R. 946,"HIGH COURT BAR ASSOCIATION, ALLAHABAD versus S...",48,0.711293,be concluded in a week while another case may ...,4,0,Not relevant,Different legal issue. The passage discusses t...
4,RAW001,Under what circumstances can delay in lodging ...,4962,[2024] 11 S.C.R. 906,VIJAYA SINGH & ANR. versus STATE OF UTTARAKHAND,35,0.707350,"the offence, the Court is required to analyze ...",5,1,Partially relevant,Partial rule. The passage addresses delay in F...


## **15.3 Create Binary Relevance Labels**

The manually assigned graded relevance scores are converted into binary relevance labels for the computation of Precision@5, Hit@5, and Mean Reciprocal Rank.

The conversion follows the same rule applied to the previous retrievers:

- Highly Relevant (2) and Partially Relevant (1) are treated as relevant.
- Not Relevant (0) is treated as non-relevant.

The original graded relevance scores are retained for the computation of nDCG@5.

In [133]:
# ============================================================
# Create Binary Relevance Labels for BGE
# ============================================================

bge_annotated_df["Binary_Relevance"] = (
    bge_annotated_df["Relevance"] > 0
).astype(int)

print("=" * 70)
print("BGE BINARY RELEVANCE LABELS")
print("=" * 70)

print(
    bge_annotated_df["Binary_Relevance"]
    .value_counts()
    .sort_index()
)

display(
    bge_annotated_df[
        [
            "Question_ID",
            "Rank",
            "Relevance",
            "Binary_Relevance"
        ]
    ].head(10)
)

BGE BINARY RELEVANCE LABELS
Binary_Relevance
0    18
1    32
Name: count, dtype: int64


,Question_ID,Rank,Relevance,Binary_Relevance
0,RAW001,1,2,1
1,RAW001,2,0,0
2,RAW001,3,0,0
3,RAW001,4,0,0
4,RAW001,5,1,1
5,RAW002,1,1,1
6,RAW002,2,2,1
7,RAW002,3,0,0
8,RAW002,4,2,1
9,RAW002,5,1,1


In [134]:
# ============================================================
# Validate BGE Binary Relevance Labels
# ============================================================

assert set(
    bge_annotated_df["Binary_Relevance"].unique()
).issubset({0, 1}), (
    "Binary relevance labels must contain only 0 and 1."
)

print("BGE binary relevance validation completed successfully.")

BGE binary relevance validation completed successfully.


## **15.4 Compute BGE Retrieval Evaluation Metrics**

The annotated BGE retrieval results are evaluated using the same metric definitions, relevance rules, retrieval depth, and aggregation procedure applied to TF-IDF, BM25, and E5.

Precision@5, Hit@5, Mean Reciprocal Rank, and nDCG@5 are computed for each pilot question and then averaged across the complete pilot benchmark.

Recall@5 remains deferred until a pooled relevance set has been constructed from all evaluated retrievers.

In [135]:
# ============================================================
# Compute BGE Pilot Retrieval Metrics
# ============================================================

from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

# -----------------------------
# Precision@5
# -----------------------------
bge_precision_per_question = (
    bge_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .mean()
    .reset_index(name="Precision@5")
)

bge_mean_precision_at_5 = (
    bge_precision_per_question["Precision@5"].mean()
)

# -----------------------------
# Hit@5
# -----------------------------
bge_hit_per_question = (
    bge_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .max()
    .reset_index(name="Hit@5")
)

bge_mean_hit_at_5 = (
    bge_hit_per_question["Hit@5"].mean()
)

# -----------------------------
# Mean Reciprocal Rank
# -----------------------------
def compute_reciprocal_rank(group):

    relevant_rows = group[
        group["Binary_Relevance"] == 1
    ]

    if relevant_rows.empty:
        return 0.0

    first_relevant_rank = (
        relevant_rows["Rank"].min()
    )

    return 1.0 / first_relevant_rank


bge_mrr_per_question = (
    bge_annotated_df
    .groupby("Question_ID")
    .apply(
        compute_reciprocal_rank,
        include_groups=False
    )
    .reset_index(name="Reciprocal_Rank")
)

bge_mean_mrr = (
    bge_mrr_per_question["Reciprocal_Rank"].mean()
)

# -----------------------------
# nDCG@5
# -----------------------------
bge_ndcg_records = []

for question_id, group in bge_annotated_df.groupby("Question_ID"):

    group = group.sort_values("Rank")

    true_relevance = (
        group["Relevance"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    predicted_scores = (
        group["BGE_Score"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    ndcg_value = ndcg_score(
        true_relevance,
        predicted_scores,
        k=TOP_K
    )

    bge_ndcg_records.append({

        "Question_ID": question_id,
        "nDCG@5": ndcg_value

    })

bge_ndcg_per_question = pd.DataFrame(
    bge_ndcg_records
)

bge_mean_ndcg_at_5 = (
    bge_ndcg_per_question["nDCG@5"].mean()
)

# -----------------------------
# Summary
# -----------------------------
bge_summary = pd.DataFrame({

    "Retriever": ["BGE"],

    "Precision@5": [
        bge_mean_precision_at_5
    ],

    "Hit@5": [
        bge_mean_hit_at_5
    ],

    "MRR": [
        bge_mean_mrr
    ],

    "nDCG@5": [
        bge_mean_ndcg_at_5
    ]

})

print("=" * 70)
print("BGE PILOT RETRIEVAL METRICS")
print("=" * 70)

display(
    bge_summary.round(4)
)

BGE PILOT RETRIEVAL METRICS


,Retriever,Precision@5,Hit@5,MRR,nDCG@5
0,BGE,0.64,0.8,0.7333,0.7072


In [136]:
# ============================================================
# Validate BGE Retrieval Metrics
# ============================================================

assert bge_precision_per_question["Precision@5"].between(0, 1).all()

assert bge_hit_per_question["Hit@5"].isin([0, 1]).all()

assert bge_mrr_per_question["Reciprocal_Rank"].between(0, 1).all()

assert bge_ndcg_per_question["nDCG@5"].between(0, 1).all()

assert len(bge_precision_per_question) == len(df_pilot)
assert len(bge_hit_per_question) == len(df_pilot)
assert len(bge_mrr_per_question) == len(df_pilot)
assert len(bge_ndcg_per_question) == len(df_pilot)

print("BGE retrieval metric validation completed successfully.")

BGE retrieval metric validation completed successfully.


## **15.5 Summary of BGE Pilot Evaluation**

The BGE dense semantic retrieval model was successfully evaluated using the same pilot benchmark, annotation protocol, and evaluation methodology employed for the previously evaluated retrieval models.

On the pilot benchmark, the BGE model demonstrated retrieval performance comparable to the lexical retrieval models, while remaining below the performance achieved by the E5 semantic retrieval model. These preliminary findings highlight that retrieval effectiveness varies across embedding models and reinforces the importance of empirical evaluation rather than assuming uniform performance among dense retrievers.

The pilot evaluation confirms that the BGE semantic retrieval pipeline, annotation workflow, and metric computation are functioning correctly. The same evaluation framework will now be applied to the InLegalBERT retrieval model.

In [137]:
# ============================================================
# Update Cumulative Pilot Retrieval Results
# ============================================================

pilot_results = pd.DataFrame({

    "Retriever": [
        "TF-IDF",
        "BM25",
        "E5",
        "BGE"
    ],

    "Type": [
        "Lexical",
        "Lexical",
        "Semantic",
        "Semantic"
    ],

    "Precision@5": [
        mean_precision_at_5,
        bm25_mean_precision_at_5,
        e5_mean_precision_at_5,
        bge_mean_precision_at_5
    ],

    "Hit@5": [
        mean_hit_at_5,
        bm25_mean_hit_at_5,
        e5_mean_hit_at_5,
        bge_mean_hit_at_5
    ],

    "MRR": [
        mean_mrr,
        bm25_mean_mrr,
        e5_mean_mrr,
        bge_mean_mrr
    ],

    "nDCG@5": [
        mean_ndcg,
        bm25_mean_ndcg_at_5,
        e5_mean_ndcg_at_5,
        bge_mean_ndcg_at_5
    ]

})

pilot_results = pilot_results.round(4)

print("=" * 80)
print("CUMULATIVE PILOT RETRIEVAL PERFORMANCE")
print("=" * 80)

display(pilot_results)

CUMULATIVE PILOT RETRIEVAL PERFORMANCE


,Retriever,Type,Precision@5,Hit@5,MRR,nDCG@5
0,TF-IDF,Lexical,0.64,0.9,0.7167,0.7744
1,BM25,Lexical,0.68,0.9,0.7167,0.7521
2,E5,Semantic,0.56,0.8,0.7000,0.6673
3,BGE,Semantic,0.64,0.8,0.7333,0.7072


# **Part Q — InLegalBERT Semantic Retrieval Evaluation**

The final dense semantic retrieval model evaluated in the pilot benchmark is InLegalBERT.

Unlike the general-purpose embedding models, InLegalBERT has been pre-trained specifically on Indian legal texts, enabling it to capture legal terminology, statutory language, and judicial expressions more effectively.

The InLegalBERT evaluation follows the same pilot benchmark, retrieval depth, annotation protocol, and evaluation methodology applied to the previous retrieval models, thereby ensuring a fair comparison across all retrieval approaches.

## **16.1 Inspect the Saved InLegalBERT Configuration**

Before loading the InLegalBERT query encoder, the saved model configuration is inspected to verify the model parameters used during corpus embedding generation.

In [138]:
# ============================================================
# Inspect Saved InLegalBERT Configuration
# ============================================================

with open(INLEGALBERT_CONFIG_FILE, "r", encoding="utf-8") as file:
    inlegalbert_config = json.load(file)

print("=" * 70)
print("INLEGALBERT CONFIGURATION")
print("=" * 70)

for key, value in inlegalbert_config.items():
    print(f"{key:<25}: {value}")

INLEGALBERT CONFIGURATION
model_name               : law-ai/InLegalBERT
embedding_dimension      : 768
normalize_embeddings     : True
max_sequence_length      : 512
total_embeddings         : 30521


## **16.2 Load and Validate the InLegalBERT Embedding Matrix**

The saved InLegalBERT embedding matrix is loaded and validated against the processed chunk dataset to ensure consistency with the corpus indexing stage.

Since the corpus embeddings were generated using normalized vectors, semantic similarity is computed using the dot product between the normalized query embedding and the normalized corpus embeddings. Under vector normalization, the dot product is equivalent to cosine similarity.

Validation confirms that:

- the number of embedding vectors matches the number of processed chunks;
- the embedding dimension matches the saved model configuration; and
- all embedding values are finite.

In [139]:
# ============================================================
# Load and Validate InLegalBERT Embeddings
# ============================================================

inlegalbert_embeddings = np.load(
    INLEGALBERT_EMBEDDINGS_FILE,
    mmap_mode="r"
)

print("=" * 70)
print("INLEGALBERT EMBEDDING MATRIX")
print("=" * 70)

print(f"Embedding Shape     : {inlegalbert_embeddings.shape}")
print(f"Embedding Data Type : {inlegalbert_embeddings.dtype}")
print(f"Processed Chunks    : {len(df_chunks):,}")
print(
    f"Expected Dimension  : "
    f"{inlegalbert_config['embedding_dimension']}"
)

assert inlegalbert_embeddings.shape[0] == len(df_chunks), (
    "InLegalBERT embedding rows are not aligned with the processed chunk dataset."
)

assert inlegalbert_embeddings.shape[1] == inlegalbert_config["embedding_dimension"], (
    "InLegalBERT embedding dimension does not match the saved configuration."
)

assert np.isfinite(inlegalbert_embeddings).all(), (
    "The InLegalBERT embedding matrix contains non-finite values."
)

print("\nInLegalBERT embedding alignment validated successfully.")

INLEGALBERT EMBEDDING MATRIX
Embedding Shape     : (30521, 768)
Embedding Data Type : float32
Processed Chunks    : 30,521
Expected Dimension  : 768

InLegalBERT embedding alignment validated successfully.


## **16.3 Load the InLegalBERT Query Encoder**

The InLegalBERT query encoder is loaded using the same model employed during corpus indexing.

Legal research questions are encoded into normalized dense vectors using the saved model configuration, ensuring consistency between query embeddings and the pre-computed corpus embeddings used for semantic retrieval.

In [140]:
# ============================================================
# Load InLegalBERT Query Encoder
# ============================================================

from sentence_transformers import SentenceTransformer

inlegalbert_model = SentenceTransformer(
    inlegalbert_config["model_name"]
)

inlegalbert_model.max_seq_length = (
    inlegalbert_config["max_sequence_length"]
)

print("=" * 70)
print("INLEGALBERT QUERY ENCODER LOADED")
print("=" * 70)

print(f"Model Name          : {inlegalbert_config['model_name']}")
print(f"Maximum Length      : {inlegalbert_model.max_seq_length}")
print(
    f"Normalize Embeddings: "
    f"{inlegalbert_config['normalize_embeddings']}"
)

print("\nInLegalBERT query encoder loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INLEGALBERT QUERY ENCODER LOADED
Model Name          : law-ai/InLegalBERT
Maximum Length      : 512
Normalize Embeddings: True

InLegalBERT query encoder loaded successfully.


## **16.4 Select the First Pilot Question**

Before applying InLegalBERT retrieval to the complete pilot benchmark, semantic retrieval is first validated using the same initial pilot question employed for TF-IDF, BM25, E5, and BGE.

Using a common validation question supports consistent qualitative comparison across all retrieval models.

In [141]:
# ============================================================
# Select First Pilot Question for InLegalBERT Validation
# ============================================================

inlegalbert_sample_question = df_pilot.iloc[0]["Question"]

print("=" * 70)
print("INLEGALBERT PILOT QUESTION")
print("=" * 70)

print(inlegalbert_sample_question)

assert inlegalbert_sample_question == sample_question, (
    "InLegalBERT validation must use the same first pilot "
    "question used for the previous retrievers."
)

print("\nPilot question consistency validated successfully.")

INLEGALBERT PILOT QUESTION
Under what circumstances can delay in lodging an FIR prove fatal to prosecution?

Pilot question consistency validated successfully.


## **16.5 Define the InLegalBERT Retrieval Function**

The InLegalBERT retrieval function encodes each legal research question into a normalized dense vector using the same model and configuration employed during corpus indexing.

Similarity is computed through the dot product between the normalized query embedding and the normalized corpus embeddings. The chunks are ranked in descending order of similarity, and the top five results are returned with their judgment metadata.

In [142]:
# ============================================================
# InLegalBERT Retrieval Function
# ============================================================

def retrieve_inlegalbert(query, top_k=5):
    """
    Retrieve the top-k most relevant chunks using
    InLegalBERT embeddings.
    """

    query_embedding = inlegalbert_model.encode(
        str(query).strip(),
        normalize_embeddings=(
            inlegalbert_config["normalize_embeddings"]
        ),
        convert_to_numpy=True,
        show_progress_bar=False
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    ).reshape(-1)

    similarity_scores = np.asarray(
        inlegalbert_embeddings @ query_embedding,
        dtype=float
    )

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    results = df_chunks.iloc[top_indices].copy()

    results["InLegalBERT_Score"] = (
        similarity_scores[top_indices]
    )

    return results[
        [
            "global_chunk_id",
            "judgment_id",
            "case_title",
            "local_chunk_id",
            "InLegalBERT_Score",
            "chunk_text"
        ]
    ].reset_index(drop=True)

In [143]:
# ============================================================
# Validate InLegalBERT Retrieval Function
# ============================================================

assert callable(retrieve_inlegalbert), (
    "The InLegalBERT retrieval function was not created successfully."
)

assert TOP_K > 0, (
    "TOP_K must be a positive integer."
)

print("InLegalBERT retrieval function defined successfully.")

InLegalBERT retrieval function defined successfully.


## **16.6 Validate InLegalBERT Retrieval**

The InLegalBERT retrieval function is validated using the first pilot question.

The top five ranked chunks are examined to confirm that semantic similarity scores, chunk identifiers, judgment metadata, and retrieved text are returned correctly before the model is applied to the complete pilot benchmark.

In [144]:
# ============================================================
# Validate InLegalBERT Retrieval on the First Pilot Question
# ============================================================

inlegalbert_results = retrieve_inlegalbert(
    inlegalbert_sample_question,
    top_k=TOP_K
)

print("=" * 70)
print("INLEGALBERT RETRIEVAL RESULTS")
print("=" * 70)

print(f"Retrieved Chunks : {len(inlegalbert_results)}")

display(inlegalbert_results)

INLEGALBERT RETRIEVAL RESULTS
Retrieved Chunks : 5


,global_chunk_id,judgment_id,case_title,local_chunk_id,InLegalBERT_Score,chunk_text
0,27117,[2024] 10 S.C.R. 893,SOMJEET MALLICK versus STATE OF JHARKHAND & OT...,8,0.781489,[2024] 10 S.C.R. 899 Somjeet Mallick v. State ...
1,27118,[2024] 10 S.C.R. 893,SOMJEET MALLICK versus STATE OF JHARKHAND & OT...,9,0.777992,"months together, while making false promises f..."
2,24052,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),7,0.776703,matches with the blood which was found on the ...
3,4117,[2024] 6 S.C.R. 164,ANEES versus THE STATE GOVT. OF NCT,51,0.776432,(Emphasis supplied) 54. Cases are frequently c...
4,26401,[2024] 2 S.C.R. 391,KALINGA @ KUSHAL versus STATE OF KARNATAKA BY ...,25,0.774125,7 (2022) 6 SCC 294 8 (2019) 5 SCC 436 9 (2020)...


In [145]:
# ============================================================
# Validate InLegalBERT Retrieval Output
# ============================================================

expected_columns = [
    "global_chunk_id",
    "judgment_id",
    "case_title",
    "local_chunk_id",
    "InLegalBERT_Score",
    "chunk_text"
]

assert len(inlegalbert_results) == TOP_K, (
    f"Expected {TOP_K} retrieved chunks, "
    f"but found {len(inlegalbert_results)}."
)

assert inlegalbert_results.columns.tolist() == expected_columns, (
    "InLegalBERT retrieval output columns do not match "
    "the expected structure."
)

assert inlegalbert_results["InLegalBERT_Score"].notna().all(), (
    "One or more InLegalBERT similarity scores are missing."
)

assert (
    inlegalbert_results["InLegalBERT_Score"].to_numpy()[:-1]
    >= inlegalbert_results["InLegalBERT_Score"].to_numpy()[1:]
).all(), (
    "InLegalBERT results are not arranged in descending "
    "similarity order."
)

assert inlegalbert_results["InLegalBERT_Score"].between(-1, 1).all(), (
    "InLegalBERT similarity scores should lie between -1 and 1 "
    "for normalized embeddings."
)

print("InLegalBERT retrieval output validation completed successfully.")

InLegalBERT retrieval output validation completed successfully.


## **16.7 Display Retrieved Chunks for Manual Inspection**

The retrieved InLegalBERT chunks are displayed in full to facilitate manual inspection of semantic retrieval quality.

This qualitative review verifies whether the highest-ranked passages address the legal issue raised by the pilot question and confirms that the InLegalBERT retrieval pipeline produces interpretable legal evidence before evaluation on the complete pilot benchmark.

In [146]:
# ============================================================
# Display InLegalBERT Retrieved Chunks
# ============================================================

for rank, row in inlegalbert_results.iterrows():

    print("=" * 100)
    print(f"Rank                 : {rank + 1}")
    print(
        f"InLegalBERT Score    : "
        f"{row['InLegalBERT_Score']:.4f}"
    )
    print(f"Judgment ID          : {row['judgment_id']}")
    print(f"Case Title           : {row['case_title']}")
    print(f"Local Chunk ID       : {row['local_chunk_id']}")
    print("-" * 100)
    print(row["chunk_text"])
    print("\n")

Rank                 : 1
InLegalBERT Score    : 0.7815
Judgment ID          : [2024] 10 S.C.R. 893
Case Title           : SOMJEET MALLICK versus STATE OF JHARKHAND & OTHERS
Local Chunk ID       : 8
----------------------------------------------------------------------------------------------------
[2024] 10 S.C.R. 899 Somjeet Mallick v. State of Jharkhand & Others

months together, while making false promises for its payment, a prima facie case, reflective of dishonest intention on the part of the accused, is made out which may require investigation. In such circumstances, if the FIR is quashed at the very inception, it would be nothing short of an act which thwarts a legitimate investigation. 18. It is trite law that FIR is not an encyclopedia of all imputations. Therefore, to test whether an FIR discloses commission of a cognizable offence what is to be looked at is not any omission in the accusations but the gravamen of the accusations contained therein to find out whether, prima fa

## **16.8 Preliminary Relevance Assessment**

The retrieved InLegalBERT results are manually reviewed to determine whether the highest-ranked chunks address the legal issue raised in the pilot question.

This preliminary assessment serves as a qualitative validation of the InLegalBERT semantic retrieval pipeline. Formal relevance annotation and metric computation will be undertaken after retrieval has been completed for all pilot questions.

## **16.9 Retrieve Top-k Results for All Pilot Questions**

Following successful validation using the first pilot question, the InLegalBERT retrieval function is applied to all ten pilot questions.

For each question, the top five semantically similar chunks are retrieved and consolidated into a single dataframe containing the question identifier, question text, retrieval rank, InLegalBERT similarity score, chunk identifier, judgment metadata, and retrieved chunk text.

In [147]:
# ============================================================
# Retrieve InLegalBERT Results for the Complete Pilot Benchmark
# ============================================================

all_inlegalbert_results = []

for _, row in df_pilot.iterrows():

    question_id = row["Question_ID"]
    question = row["Question"]

    retrieved = retrieve_inlegalbert(
        question,
        top_k=TOP_K
    )

    retrieved.insert(0, "Question", question)
    retrieved.insert(0, "Question_ID", question_id)

    retrieved["Rank"] = range(1, len(retrieved) + 1)

    all_inlegalbert_results.append(retrieved)

df_inlegalbert_results = pd.concat(
    all_inlegalbert_results,
    ignore_index=True
)

print("=" * 70)
print("INLEGALBERT PILOT RETRIEVAL")
print("=" * 70)

print(f"Pilot Questions : {len(df_pilot)}")
print(f"Retrieved Chunks: {len(df_inlegalbert_results)}")

display(df_inlegalbert_results.head(15))

INLEGALBERT PILOT RETRIEVAL
Pilot Questions : 10
Retrieved Chunks: 50


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,InLegalBERT_Score,chunk_text,Rank
0,RAW001,Under what circumstances can delay in lodging ...,27117,[2024] 10 S.C.R. 893,SOMJEET MALLICK versus STATE OF JHARKHAND & OT...,8,0.781489,[2024] 10 S.C.R. 899 Somjeet Mallick v. State ...,1
1,RAW001,Under what circumstances can delay in lodging ...,27118,[2024] 10 S.C.R. 893,SOMJEET MALLICK versus STATE OF JHARKHAND & OT...,9,0.777992,"months together, while making false promises f...",2
2,RAW001,Under what circumstances can delay in lodging ...,24052,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),7,0.776703,matches with the blood which was found on the ...,3
3,RAW001,Under what circumstances can delay in lodging ...,4117,[2024] 6 S.C.R. 164,ANEES versus THE STATE GOVT. OF NCT,51,0.776432,(Emphasis supplied) 54. Cases are frequently c...,4
4,RAW001,Under what circumstances can delay in lodging ...,26401,[2024] 2 S.C.R. 391,KALINGA @ KUSHAL versus STATE OF KARNATAKA BY ...,25,0.774125,7 (2022) 6 SCC 294 8 (2019) 5 SCC 436 9 (2020)...,5
5,RAW002,Can a conviction be sustained solely on the te...,26401,[2024] 2 S.C.R. 391,KALINGA @ KUSHAL versus STATE OF KARNATAKA BY ...,25,0.786012,7 (2022) 6 SCC 294 8 (2019) 5 SCC 436 9 (2020)...,1
6,RAW002,Can a conviction be sustained solely on the te...,10786,[2024] 2 S.C.R. 668,RAM SINGH versus THE STATE OF U.P.,23,0.785947,"22. From the above, it is evident that there a...",2
7,RAW002,Can a conviction be sustained solely on the te...,24052,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),7,0.785549,matches with the blood which was found on the ...,3
8,RAW002,Can a conviction be sustained solely on the te...,24051,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),6,0.782718,the manner of recovery of the knife from the a...,4
9,RAW002,Can a conviction be sustained solely on the te...,23490,[2024] 7 S.C.R. 218,DHARMENDRA KUMAR @ DHAMMA versus STATE OF MADH...,34,0.782076,magnification ought to be an exception and be ...,5


In [148]:
# ============================================================
# Validate Complete InLegalBERT Pilot Retrieval
# ============================================================

expected_rows = len(df_pilot) * TOP_K

assert len(df_inlegalbert_results) == expected_rows, (
    f"Expected {expected_rows} InLegalBERT retrieval rows, "
    f"but found {len(df_inlegalbert_results)}."
)

results_per_question = (
    df_inlegalbert_results
    .groupby("Question_ID")
    .size()
)

assert (results_per_question == TOP_K).all(), (
    "At least one pilot question does not have exactly TOP_K InLegalBERT results."
)

assert (
    df_inlegalbert_results["Question_ID"].nunique()
    == len(df_pilot)
), (
    "InLegalBERT results do not cover all pilot questions."
)

assert df_inlegalbert_results["InLegalBERT_Score"].notna().all(), (
    "One or more InLegalBERT similarity scores are missing."
)

print("Complete InLegalBERT pilot retrieval validation passed.")

Complete InLegalBERT pilot retrieval validation passed.


## **16.10 Save InLegalBERT Pilot Retrieval Results**

The InLegalBERT retrieval outputs are saved to the pilot retrieval directory to preserve the ranked semantic retrieval results for subsequent manual annotation and metric computation.

Each saved record contains the originating pilot question, retrieval rank, InLegalBERT similarity score, chunk identifier, judgment metadata, and retrieved chunk text.

In [149]:
# ============================================================
# Save InLegalBERT Pilot Retrieval Results
# ============================================================

inlegalbert_output_file = (
    PILOT_RETRIEVAL_DIR /
    "inlegalbert_pilot_retrieval.csv"
)

df_inlegalbert_results.to_csv(
    inlegalbert_output_file,
    index=False
)

print("=" * 60)
print("INLEGALBERT PILOT RESULTS SAVED")
print("=" * 60)

print(f"Location : {inlegalbert_output_file}")
print(f"Rows     : {len(df_inlegalbert_results):,}")

INLEGALBERT PILOT RESULTS SAVED
Location : D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\retrieval\inlegalbert_pilot_retrieval.csv
Rows     : 50


## **16.11 Create the InLegalBERT Annotation Template**

A manual relevance annotation template is generated from the InLegalBERT retrieval results.

The annotation structure is identical to those used for TF-IDF, BM25, E5, and BGE, thereby ensuring that all retrieval models are evaluated using the same relevance criteria and annotation methodology.

In [150]:
# ============================================================
# Create InLegalBERT Annotation Template
# ============================================================

inlegalbert_annotation_df = (
    df_inlegalbert_results.copy()
)

inlegalbert_annotation_df["Relevance"] = ""
inlegalbert_annotation_df["Relevance_Level"] = ""
inlegalbert_annotation_df["Comments"] = ""

display(
    inlegalbert_annotation_df.head()
)

,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,InLegalBERT_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,27117,[2024] 10 S.C.R. 893,SOMJEET MALLICK versus STATE OF JHARKHAND & OT...,8,0.781489,[2024] 10 S.C.R. 899 Somjeet Mallick v. State ...,1,,,
1,RAW001,Under what circumstances can delay in lodging ...,27118,[2024] 10 S.C.R. 893,SOMJEET MALLICK versus STATE OF JHARKHAND & OT...,9,0.777992,"months together, while making false promises f...",2,,,
2,RAW001,Under what circumstances can delay in lodging ...,24052,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),7,0.776703,matches with the blood which was found on the ...,3,,,
3,RAW001,Under what circumstances can delay in lodging ...,4117,[2024] 6 S.C.R. 164,ANEES versus THE STATE GOVT. OF NCT,51,0.776432,(Emphasis supplied) 54. Cases are frequently c...,4,,,
4,RAW001,Under what circumstances can delay in lodging ...,26401,[2024] 2 S.C.R. 391,KALINGA @ KUSHAL versus STATE OF KARNATAKA BY ...,25,0.774125,7 (2022) 6 SCC 294 8 (2019) 5 SCC 436 9 (2020)...,5,,,


In [151]:
# ============================================================
# Save InLegalBERT Annotation Template
# ============================================================

inlegalbert_annotation_file = (
    PILOT_ANNOTATION_DIR /
    "inlegalbert_annotation_template.csv"
)

inlegalbert_annotation_df.to_csv(
    inlegalbert_annotation_file,
    index=False
)

print("=" * 60)
print("INLEGALBERT ANNOTATION TEMPLATE CREATED")
print("=" * 60)

print(inlegalbert_annotation_file)

INLEGALBERT ANNOTATION TEMPLATE CREATED
D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\annotation\inlegalbert_annotation_template.csv


# **Part R — InLegalBERT Retrieval Evaluation Metrics**

The manually annotated InLegalBERT pilot retrieval results are used to compute the same retrieval evaluation metrics previously applied to the lexical retrieval models and the other dense semantic retrieval models.

Using identical metric definitions, annotation criteria, retrieval depth, and pilot benchmark ensures a fair comparison across all evaluated retrieval approaches.

The InLegalBERT pilot evaluation includes:

- Precision@5
- Hit@5
- Mean Reciprocal Rank (MRR)
- Normalized Discounted Cumulative Gain (nDCG@5)

Recall@5 is deferred until the complete relevance pool has been constructed from the outputs of all evaluated retrieval models.

The InLegalBERT results presented in this section are preliminary pilot findings intended to validate the semantic retrieval pipeline prior to evaluation on the complete benchmark.

## **17.1 Load the Annotated InLegalBERT Retrieval Results**

The manually annotated InLegalBERT retrieval results are loaded from the pilot annotation directory.

The annotation file contains the relevance judgments assigned during manual review and serves as the reference standard for computing the retrieval evaluation metrics.

In [152]:
# ============================================================
# Load Annotated InLegalBERT Pilot Results
# ============================================================

INLEGALBERT_ANNOTATION_FILE = (
    PILOT_ANNOTATION_DIR /
    "inlegalbert_pilot_annotations.csv"
)

inlegalbert_annotated_df = pd.read_csv(
    INLEGALBERT_ANNOTATION_FILE
)

print("=" * 70)
print("ANNOTATED INLEGALBERT PILOT RETRIEVAL RESULTS")
print("=" * 70)

print(f"Rows    : {len(inlegalbert_annotated_df):,}")
print(f"Columns : {len(inlegalbert_annotated_df.columns)}")

display(inlegalbert_annotated_df.head())

ANNOTATED INLEGALBERT PILOT RETRIEVAL RESULTS
Rows    : 50
Columns : 12


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,InLegalBERT_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,27117,[2024] 10 S.C.R. 893,SOMJEET MALLICK versus STATE OF JHARKHAND & OT...,8,0.781489,[2024] 10 S.C.R. 899 Somjeet Mallick v. State ...,1,1,Partially relevant,Passage type is holding/reasoning. It discusse...
1,RAW001,Under what circumstances can delay in lodging ...,27118,[2024] 10 S.C.R. 893,SOMJEET MALLICK versus STATE OF JHARKHAND & OT...,9,0.777992,"months together, while making false promises f...",2,1,Partially relevant,Passage type is holding/reasoning. It discusse...
2,RAW001,Under what circumstances can delay in lodging ...,24052,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),7,0.776703,matches with the blood which was found on the ...,3,0,Not relevant,Passage type is facts/reasoning. Does not addr...
3,RAW001,Under what circumstances can delay in lodging ...,4117,[2024] 6 S.C.R. 164,ANEES versus THE STATE GOVT. OF NCT,51,0.776432,(Emphasis supplied) 54. Cases are frequently c...,4,0,Not relevant,Passage type is facts/reasoning. Does not addr...
4,RAW001,Under what circumstances can delay in lodging ...,26401,[2024] 2 S.C.R. 391,KALINGA @ KUSHAL versus STATE OF KARNATAKA BY ...,25,0.774125,7 (2022) 6 SCC 294 8 (2019) 5 SCC 436 9 (2020)...,5,0,Not relevant,Passage type is facts/reasoning. Does not addr...


## **17.2 Validate the Annotation Dataset**

Before computing the retrieval evaluation metrics, the annotated InLegalBERT dataset is validated to ensure that all retrieved chunks have been assigned relevance labels and that the expected number of pilot retrieval records is present.

The validation also summarises the distribution of relevance scores assigned during manual annotation.

In [153]:
# ============================================================
# Validate InLegalBERT Annotation Dataset
# ============================================================

print("=" * 70)
print("INLEGALBERT ANNOTATION VALIDATION")
print("=" * 70)

print(f"Total Records : {len(inlegalbert_annotated_df)}")

print("\nMissing Values\n")
print(inlegalbert_annotated_df.isnull().sum())

print("\nRelevance Labels\n")
print(
    inlegalbert_annotated_df["Relevance"]
    .value_counts()
)

print("\nRelevance Levels\n")
print(
    inlegalbert_annotated_df["Relevance_Level"]
    .value_counts()
)

INLEGALBERT ANNOTATION VALIDATION
Total Records : 50

Missing Values

Question_ID          0
Question             0
global_chunk_id      0
judgment_id          0
case_title           0
local_chunk_id       0
InLegalBERT_Score    0
chunk_text           0
Rank                 0
Relevance            0
Relevance_Level      0
Comments             0
dtype: int64

Relevance Labels

Relevance
0    26
1    15
2     9
Name: count, dtype: int64

Relevance Levels

Relevance_Level
Not relevant          26
Partially relevant    15
Highly relevant        9
Name: count, dtype: int64


In [154]:
# ============================================================
# Standardize Relevance Level Labels
# ============================================================

inlegalbert_annotated_df["Relevance_Level"] = (
    inlegalbert_annotated_df["Relevance_Level"]
    .astype(str)
    .str.strip()
    .replace({
        "Relevant": "Partially Relevant"
    })
)

print("=" * 70)
print("STANDARDIZED RELEVANCE LEVELS")
print("=" * 70)

print(
    inlegalbert_annotated_df["Relevance_Level"]
    .value_counts()
)

STANDARDIZED RELEVANCE LEVELS
Relevance_Level
Not relevant          26
Partially relevant    15
Highly relevant        9
Name: count, dtype: int64


In [155]:
# ============================================================
# Validate Relevance Labels
# ============================================================

expected_levels = {
    2: "Highly Relevant",
    1: "Partially Relevant",
    0: "Not Relevant"
}

mismatches = inlegalbert_annotated_df[
    inlegalbert_annotated_df.apply(
        lambda row: expected_levels[row["Relevance"]]
        != row["Relevance_Level"],
        axis=1
    )
]

print(f"Label mismatches: {len(mismatches)}")

if len(mismatches) > 0:
    display(
        mismatches[
            [
                "Question_ID",
                "Rank",
                "Relevance",
                "Relevance_Level"
            ]
        ]
    )

Label mismatches: 50


,Question_ID,Rank,Relevance,Relevance_Level
0,RAW001,1,1,Partially relevant
1,RAW001,2,1,Partially relevant
2,RAW001,3,0,Not relevant
3,RAW001,4,0,Not relevant
4,RAW001,5,0,Not relevant
5,RAW002,1,0,Not relevant
6,RAW002,2,1,Partially relevant
7,RAW002,3,1,Partially relevant
8,RAW002,4,0,Not relevant
9,RAW002,5,1,Partially relevant


## **17.3 Create Binary Relevance Labels**

The manually assigned graded relevance scores are converted into binary relevance labels for the computation of Precision@5, Hit@5, and Mean Reciprocal Rank.

The conversion follows the same rule applied to the previous retrieval models:

- Highly Relevant (2) and Partially Relevant (1) are treated as relevant.
- Not Relevant (0) is treated as non-relevant.

The original graded relevance scores are retained for the computation of nDCG@5.

In [156]:
# ============================================================
# Create Binary Relevance Labels for InLegalBERT
# ============================================================

inlegalbert_annotated_df["Binary_Relevance"] = (
    inlegalbert_annotated_df["Relevance"] > 0
).astype(int)

print("=" * 70)
print("INLEGALBERT BINARY RELEVANCE LABELS")
print("=" * 70)

print(
    inlegalbert_annotated_df["Binary_Relevance"]
    .value_counts()
    .sort_index()
)

display(
    inlegalbert_annotated_df[
        [
            "Question_ID",
            "Rank",
            "Relevance",
            "Binary_Relevance"
        ]
    ].head(10)
)

INLEGALBERT BINARY RELEVANCE LABELS
Binary_Relevance
0    26
1    24
Name: count, dtype: int64


,Question_ID,Rank,Relevance,Binary_Relevance
0,RAW001,1,1,1
1,RAW001,2,1,1
2,RAW001,3,0,0
3,RAW001,4,0,0
4,RAW001,5,0,0
5,RAW002,1,0,0
6,RAW002,2,1,1
7,RAW002,3,1,1
8,RAW002,4,0,0
9,RAW002,5,1,1


In [157]:
# ============================================================
# Validate InLegalBERT Binary Relevance Labels
# ============================================================

assert set(
    inlegalbert_annotated_df["Binary_Relevance"].unique()
).issubset({0, 1}), (
    "Binary relevance labels must contain only 0 and 1."
)

print("InLegalBERT binary relevance validation completed successfully.")

InLegalBERT binary relevance validation completed successfully.


## **17.4 Compute InLegalBERT Retrieval Evaluation Metrics**

The annotated InLegalBERT retrieval results are evaluated using the same metric definitions, relevance rules, retrieval depth, and aggregation procedure applied to the previously evaluated retrieval models.

Precision@5, Hit@5, Mean Reciprocal Rank (MRR), and nDCG@5 are computed for each pilot question and subsequently averaged across the complete pilot benchmark.

Recall@5 remains deferred until a pooled relevance set has been constructed from all evaluated retrievers.

In [158]:
# ============================================================
# Compute InLegalBERT Pilot Retrieval Metrics
# ============================================================

from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

# -----------------------------
# Precision@5
# -----------------------------
inlegalbert_precision_per_question = (
    inlegalbert_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .mean()
    .reset_index(name="Precision@5")
)

inlegalbert_mean_precision_at_5 = (
    inlegalbert_precision_per_question["Precision@5"].mean()
)

# -----------------------------
# Hit@5
# -----------------------------
inlegalbert_hit_per_question = (
    inlegalbert_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .max()
    .reset_index(name="Hit@5")
)

inlegalbert_mean_hit_at_5 = (
    inlegalbert_hit_per_question["Hit@5"].mean()
)

# -----------------------------
# Mean Reciprocal Rank
# -----------------------------
def compute_reciprocal_rank(group):
    relevant_rows = group[
        group["Binary_Relevance"] == 1
    ]

    if relevant_rows.empty:
        return 0.0

    first_relevant_rank = relevant_rows["Rank"].min()

    return 1.0 / first_relevant_rank


inlegalbert_mrr_per_question = (
    inlegalbert_annotated_df
    .groupby("Question_ID")
    .apply(
        compute_reciprocal_rank,
        include_groups=False
    )
    .reset_index(name="Reciprocal_Rank")
)

inlegalbert_mean_mrr = (
    inlegalbert_mrr_per_question[
        "Reciprocal_Rank"
    ].mean()
)

# -----------------------------
# nDCG@5
# -----------------------------
inlegalbert_ndcg_records = []

for question_id, group in (
    inlegalbert_annotated_df.groupby("Question_ID")
):

    group = group.sort_values("Rank")

    true_relevance = (
        group["Relevance"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    predicted_scores = (
        group["InLegalBERT_Score"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    ndcg_value = ndcg_score(
        true_relevance,
        predicted_scores,
        k=TOP_K
    )

    inlegalbert_ndcg_records.append({
        "Question_ID": question_id,
        "nDCG@5": ndcg_value
    })

inlegalbert_ndcg_per_question = pd.DataFrame(
    inlegalbert_ndcg_records
)

inlegalbert_mean_ndcg_at_5 = (
    inlegalbert_ndcg_per_question["nDCG@5"].mean()
)

# -----------------------------
# Summary
# -----------------------------
inlegalbert_summary = pd.DataFrame({
    "Retriever": ["InLegalBERT"],
    "Precision@5": [
        inlegalbert_mean_precision_at_5
    ],
    "Hit@5": [
        inlegalbert_mean_hit_at_5
    ],
    "MRR": [
        inlegalbert_mean_mrr
    ],
    "nDCG@5": [
        inlegalbert_mean_ndcg_at_5
    ]
})

print("=" * 70)
print("INLEGALBERT PILOT RETRIEVAL METRICS")
print("=" * 70)

display(inlegalbert_summary.round(4))

INLEGALBERT PILOT RETRIEVAL METRICS


,Retriever,Precision@5,Hit@5,MRR,nDCG@5
0,InLegalBERT,0.48,0.8,0.575,0.6449


In [159]:
# ============================================================
# Validate InLegalBERT Retrieval Metrics
# ============================================================

assert (
    inlegalbert_precision_per_question["Precision@5"]
    .between(0, 1)
    .all()
)

assert (
    inlegalbert_hit_per_question["Hit@5"]
    .isin([0, 1])
    .all()
)

assert (
    inlegalbert_mrr_per_question["Reciprocal_Rank"]
    .between(0, 1)
    .all()
)

assert (
    inlegalbert_ndcg_per_question["nDCG@5"]
    .between(0, 1)
    .all()
)

assert len(inlegalbert_precision_per_question) == len(df_pilot)
assert len(inlegalbert_hit_per_question) == len(df_pilot)
assert len(inlegalbert_mrr_per_question) == len(df_pilot)
assert len(inlegalbert_ndcg_per_question) == len(df_pilot)

print(
    "InLegalBERT retrieval metric validation "
    "completed successfully."
)

InLegalBERT retrieval metric validation completed successfully.


## **17.5 Summary of InLegalBERT Pilot Evaluation**

The InLegalBERT dense semantic retrieval model was successfully evaluated using the same pilot benchmark, annotation protocol, and evaluation methodology employed for the previous retrieval models.

On the pilot benchmark, InLegalBERT achieved lower retrieval performance than both the general-purpose semantic embedding models and the lexical retrieval models. These preliminary findings suggest that domain-specific pre-training alone does not necessarily translate into superior dense retrieval performance and further reinforce the need for empirical evaluation of retrieval models within the NYAYaRAG framework.

The pilot evaluation confirms that the InLegalBERT retrieval pipeline, annotation workflow, and metric computation are functioning correctly. The next stage evaluates a hybrid retrieval strategy combining lexical and dense semantic retrieval.

In [160]:
# ============================================================
# Update Cumulative Pilot Retrieval Results
# ============================================================

pilot_results = pd.DataFrame({

    "Retriever": [
        "TF-IDF",
        "BM25",
        "E5",
        "BGE",
        "InLegalBERT"
    ],

    "Type": [
        "Lexical",
        "Lexical",
        "Semantic",
        "Semantic",
        "Semantic"
    ],

    "Precision@5": [
        mean_precision_at_5,
        bm25_mean_precision_at_5,
        e5_mean_precision_at_5,
        bge_mean_precision_at_5,
        inlegalbert_mean_precision_at_5
    ],

    "Hit@5": [
        mean_hit_at_5,
        bm25_mean_hit_at_5,
        e5_mean_hit_at_5,
        bge_mean_hit_at_5,
        inlegalbert_mean_hit_at_5
    ],

    "MRR": [
        mean_mrr,
        bm25_mean_mrr,
        e5_mean_mrr,
        bge_mean_mrr,
        inlegalbert_mean_mrr
    ],

    "nDCG@5": [
        mean_ndcg,
        bm25_mean_ndcg_at_5,
        e5_mean_ndcg_at_5,
        bge_mean_ndcg_at_5,
        inlegalbert_mean_ndcg_at_5
    ]

})

pilot_results = pilot_results.round(4)

print("=" * 80)
print("CUMULATIVE PILOT RETRIEVAL PERFORMANCE")
print("=" * 80)

display(pilot_results)

CUMULATIVE PILOT RETRIEVAL PERFORMANCE


,Retriever,Type,Precision@5,Hit@5,MRR,nDCG@5
0,TF-IDF,Lexical,0.64,0.9,0.7167,0.7744
1,BM25,Lexical,0.68,0.9,0.7167,0.7521
2,E5,Semantic,0.56,0.8,0.7000,0.6673
3,BGE,Semantic,0.64,0.8,0.7333,0.7072
4,InLegalBERT,Semantic,0.48,0.8,0.5750,0.6449


# **Part S — Hybrid Retrieval Evaluation using Reciprocal Rank Fusion**

The final retrieval strategy evaluated in the pilot benchmark is a hybrid retrieval model combining the strongest lexical retriever (BM25) and the strongest dense semantic retriever (E5).

Hybrid retrieval seeks to exploit the complementary strengths of lexical and semantic retrieval. BM25 effectively captures exact legal terminology and statutory references, whereas E5 retrieves semantically related legal concepts even when the wording differs from the query.

The rankings produced by BM25 and E5 are combined using Reciprocal Rank Fusion (RRF). RRF combines ranked retrieval lists without requiring score normalisation and has been widely adopted as an effective and robust rank fusion technique in information retrieval.

The hybrid retrieval model is evaluated using the same pilot benchmark, annotation protocol, retrieval depth, and evaluation metrics employed for all previous retrieval models.

## **18.1 Reciprocal Rank Fusion**

The hybrid retrieval model combines the ranked outputs of the BM25 lexical retriever and the E5 semantic retriever using **Reciprocal Rank Fusion (RRF)**.

Unlike score-based fusion methods, which require retrieval scores from different models to be normalised before combination, RRF operates solely on the ranked positions of the retrieved chunks. Consequently, it provides a simple, robust, and model-independent approach for combining heterogeneous retrieval methods.

For each retrieved chunk, the hybrid fusion score is computed as:

$$
\mathrm{RRF}(d)=
\frac{1}{60+r_{\mathrm{BM25}}(d)}
+
\frac{1}{60+r_{\mathrm{E5}}(d)}
$$

where:

- **BM25 Rank** denotes the position assigned to a retrieved chunk by the BM25 lexical retriever.
- **E5 Rank** denotes the position assigned to the same retrieved chunk by the E5 semantic retriever.
- **60** is the standard Reciprocal Rank Fusion constant used to reduce the influence of small rank differences between ranked retrieval lists.

Chunks that receive high rankings from both BM25 and E5 obtain larger RRF scores and are therefore ranked higher in the final hybrid retrieval results.


## 18.3 Define the Reciprocal Rank Fusion Algorithm

The Reciprocal Rank Fusion algorithm combines the ranked outputs produced by the BM25 lexical retriever and the E5 semantic retriever into a single hybrid ranking.

Each retrieved chunk receives an RRF score based on its rank in both retrieval lists. Chunks retrieved by both models receive higher fusion scores than chunks retrieved by only one model.

After computing the RRF scores, the retrieved chunks are ranked in descending order of their fusion scores, and the highest-ranked chunks are selected for subsequent evaluation.

In [161]:
# ============================================================
# Reciprocal Rank Fusion (RRF)
# ============================================================

def reciprocal_rank_fusion(
    bm25_results,
    e5_results,
    rrf_k=60
):
    """
    Combine BM25 and E5 retrieval results using
    Reciprocal Rank Fusion (RRF).
    """

    fused_scores = {}

    # -------------------------
    # BM25 contribution
    # -------------------------
    for rank, row in enumerate(
        bm25_results.itertuples(index=False),
        start=1
    ):

        chunk_id = row.global_chunk_id

        fused_scores.setdefault(chunk_id, 0)

        fused_scores[chunk_id] += (
            1 / (rrf_k + rank)
        )

    # -------------------------
    # E5 contribution
    # -------------------------
    for rank, row in enumerate(
        e5_results.itertuples(index=False),
        start=1
    ):

        chunk_id = row.global_chunk_id

        fused_scores.setdefault(chunk_id, 0)

        fused_scores[chunk_id] += (
            1 / (rrf_k + rank)
        )

    return fused_scores

In [162]:
# ============================================================
# Validate RRF Function
# ============================================================

assert callable(
    reciprocal_rank_fusion
), "RRF function was not created."

print(
    "Reciprocal Rank Fusion function created successfully."
)

Reciprocal Rank Fusion function created successfully.


## 18.4 Define the Hybrid Retrieval Function

The hybrid retrieval function independently retrieves candidate chunks using BM25 and E5, combines their ranked outputs through Reciprocal Rank Fusion, and returns the highest-ranked fused results.

For each query:

1. BM25 retrieves the top 50 lexical candidates.
2. E5 retrieves the top 50 semantic candidates.
3. Reciprocal Rank Fusion assigns a combined score to each unique chunk.
4. The fused candidates are ranked in descending order of RRF score.
5. The final top five chunks are returned for annotation and evaluation.

In [163]:
# ============================================================
# Hybrid BM25 + E5 Retrieval Function using RRF
# ============================================================

def retrieve_hybrid(
    query,
    candidate_k=50,
    top_k=5,
    rrf_k=60
):
    """
    Retrieve top-k chunks using BM25 + E5 Reciprocal Rank Fusion.
    """

    # Retrieve candidate sets
    bm25_candidates = retrieve_bm25(
        query,
        top_k=candidate_k
    ).copy()

    e5_candidates = retrieve_e5(
        query,
        top_k=candidate_k
    ).copy()

    # Preserve original ranks
    bm25_candidates["BM25_Rank"] = range(
        1,
        len(bm25_candidates) + 1
    )

    e5_candidates["E5_Rank"] = range(
        1,
        len(e5_candidates) + 1
    )

    # Compute fused scores
    fused_scores = reciprocal_rank_fusion(
        bm25_candidates,
        e5_candidates,
        rrf_k=rrf_k
    )

    # Build candidate metadata from the union of both result sets
    candidate_ids = list(fused_scores.keys())

    hybrid_results = (
        df_chunks[
            df_chunks["global_chunk_id"].isin(candidate_ids)
        ]
        .copy()
    )

    hybrid_results["RRF_Score"] = (
        hybrid_results["global_chunk_id"]
        .map(fused_scores)
    )

    # Add original model ranks where available
    bm25_rank_map = dict(
        zip(
            bm25_candidates["global_chunk_id"],
            bm25_candidates["BM25_Rank"]
        )
    )

    e5_rank_map = dict(
        zip(
            e5_candidates["global_chunk_id"],
            e5_candidates["E5_Rank"]
        )
    )

    hybrid_results["BM25_Rank"] = (
        hybrid_results["global_chunk_id"]
        .map(bm25_rank_map)
    )

    hybrid_results["E5_Rank"] = (
        hybrid_results["global_chunk_id"]
        .map(e5_rank_map)
    )

    # Sort by fused score
    hybrid_results = (
        hybrid_results
        .sort_values(
            by=[
                "RRF_Score",
                "BM25_Rank",
                "E5_Rank"
            ],
            ascending=[
                False,
                True,
                True
            ],
            na_position="last"
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    return hybrid_results[
        [
            "global_chunk_id",
            "judgment_id",
            "case_title",
            "local_chunk_id",
            "BM25_Rank",
            "E5_Rank",
            "RRF_Score",
            "chunk_text"
        ]
    ]

In [164]:
# ============================================================
# Hybrid Retrieval Configuration
# ============================================================

RRF_K = 60

HYBRID_CANDIDATES = 50

HYBRID_TOP_K = 5

print("=" * 70)
print("HYBRID RETRIEVAL CONFIGURATION")
print("=" * 70)

print(f"BM25 Candidates      : {HYBRID_CANDIDATES}")
print(f"E5 Candidates        : {HYBRID_CANDIDATES}")
print(f"Fusion Constant (k)  : {RRF_K}")
print(f"Final Top-k          : {HYBRID_TOP_K}")

HYBRID RETRIEVAL CONFIGURATION
BM25 Candidates      : 50
E5 Candidates        : 50
Fusion Constant (k)  : 60
Final Top-k          : 5


In [165]:
# ============================================================
# Validate Hybrid Retrieval Function
# ============================================================

assert callable(retrieve_hybrid), (
    "The hybrid retrieval function was not created successfully."
)

assert HYBRID_CANDIDATES >= HYBRID_TOP_K, (
    "Candidate depth must be greater than or equal to final Top-k."
)

assert RRF_K > 0, (
    "The RRF constant must be positive."
)

print("Hybrid retrieval function defined successfully.")

Hybrid retrieval function defined successfully.


## 18.5 Validate Hybrid Retrieval

The hybrid retrieval function is validated using the same first pilot question previously used for TF-IDF, BM25, E5, BGE, and InLegalBERT.

The top five fused results are examined to confirm that:

- the final ranking contains exactly five chunks;
- RRF scores are present and arranged in descending order;
- BM25 and E5 source ranks are preserved where applicable; and
- chunk identifiers, judgment metadata, and retrieved text remain correctly aligned.

In [166]:
# ============================================================
# Validate Hybrid Retrieval on the First Pilot Question
# ============================================================

hybrid_sample_results = retrieve_hybrid(
    sample_question,
    candidate_k=HYBRID_CANDIDATES,
    top_k=HYBRID_TOP_K,
    rrf_k=RRF_K
)

print("=" * 70)
print("HYBRID BM25 + E5 RETRIEVAL RESULTS")
print("=" * 70)

print(f"Retrieved Chunks : {len(hybrid_sample_results)}")

display(hybrid_sample_results)

HYBRID BM25 + E5 RETRIEVAL RESULTS
Retrieved Chunks : 5


,global_chunk_id,judgment_id,case_title,local_chunk_id,BM25_Rank,E5_Rank,RRF_Score,chunk_text
0,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,2.0,1.0,0.032522,20 See Yogesh Singh v. Mahabeer Singh and Othe...
1,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,1.0,3.0,0.032266,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare..."
2,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,3.0,2.0,0.032002,"provides that ‘if, from information received o..."
3,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,4.0,4.0,0.031250,15 Nevada Properties (P) Ltd. Vs. State of Mah...
4,14762,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,44,9.0,40.0,0.024493,"in the commission of offence, then this delay ..."


In [167]:
# ============================================================
# Validate Hybrid Retrieval Output
# ============================================================

expected_columns = [
    "global_chunk_id",
    "judgment_id",
    "case_title",
    "local_chunk_id",
    "BM25_Rank",
    "E5_Rank",
    "RRF_Score",
    "chunk_text"
]

assert len(hybrid_sample_results) == HYBRID_TOP_K, (
    f"Expected {HYBRID_TOP_K} hybrid results, "
    f"but found {len(hybrid_sample_results)}."
)

assert hybrid_sample_results.columns.tolist() == expected_columns, (
    "Hybrid retrieval output columns do not match the expected structure."
)

assert hybrid_sample_results["RRF_Score"].notna().all(), (
    "One or more RRF scores are missing."
)

assert (
    hybrid_sample_results["RRF_Score"].to_numpy()[:-1]
    >= hybrid_sample_results["RRF_Score"].to_numpy()[1:]
).all(), (
    "Hybrid results are not arranged in descending RRF score order."
)

assert hybrid_sample_results["global_chunk_id"].is_unique, (
    "Duplicate chunks were found in the final hybrid Top-k results."
)

print("Hybrid retrieval output validation completed successfully.")

Hybrid retrieval output validation completed successfully.


## **18.6 Display Hybrid Retrieved Chunks for Manual Inspection**

The fused hybrid retrieval results are displayed in full to facilitate qualitative inspection.

For each retrieved chunk, the output includes its original BM25 rank, E5 rank, final Reciprocal Rank Fusion score, judgment metadata, and complete chunk text. This enables verification of how lexical and semantic rankings contribute to the final hybrid result.

In [168]:
# ============================================================
# Display Hybrid Retrieved Chunks
# ============================================================

for rank, row in hybrid_sample_results.iterrows():

    bm25_rank = (
        int(row["BM25_Rank"])
        if pd.notna(row["BM25_Rank"])
        else "Not in Top-50"
    )

    e5_rank = (
        int(row["E5_Rank"])
        if pd.notna(row["E5_Rank"])
        else "Not in Top-50"
    )

    print("=" * 100)
    print(f"Hybrid Rank       : {rank + 1}")
    print(f"RRF Score         : {row['RRF_Score']:.6f}")
    print(f"BM25 Rank         : {bm25_rank}")
    print(f"E5 Rank           : {e5_rank}")
    print(f"Judgment ID       : {row['judgment_id']}")
    print(f"Case Title        : {row['case_title']}")
    print(f"Local Chunk ID    : {row['local_chunk_id']}")
    print("-" * 100)
    print(row["chunk_text"])
    print()

Hybrid Rank       : 1
RRF Score         : 0.032522
BM25 Rank         : 2
E5 Rank           : 1
Judgment ID       : [2024] 10 S.C.R. 1313
Case Title        : RAMA DEVI versus THE STATE OF BIHAR AND OTHERS
Local Chunk ID    : 40
----------------------------------------------------------------------------------------------------
20 See Yogesh Singh v. Mahabeer Singh and Others (2017) 11 SCC 195 which refers to several other decisions. See also State of Rajasthan v. Arjun Singh and Others (2011) 9 SCC 115

[2024] 10 S.C.R. 1337 Rama Devi v. The State of Bihar and Others

(PW-1), Mahanth Ashwani Das (PW-25), and Amarendra Kumar Sinha (PW-10). The inquest reports of Brij Bihari Prasad and Lakshmeshwar Sahu (Exhibits 42/1 and 52) were prepared on the same night and, thereafter, the post-mortem was conducted by Dr. Arvind Kumar Singh (PW-7) at 12:30 a.m. and 1:30 a.m. respectively on 14.06.1998. 30. The occurrence having taken place at night on 13.06.1998, normally the FIR should have been sen

## 18.7 Preliminary Relevance Assessment

The hybrid retrieval results are manually reviewed to determine whether the fused ranking provides relevant legal evidence for the pilot question.

Particular attention is given to whether chunks ranked highly by both BM25 and E5 are promoted in the final ranking and whether the hybrid output improves upon the standalone lexical and semantic retrieval results.

This assessment serves as a qualitative validation only. Formal relevance annotation and metric computation will be conducted after hybrid retrieval has been completed for all pilot questions.

## **18.8 Retrieve Top-k Hybrid Results for All Pilot Questions**

Following successful validation using the first pilot question, the BM25–E5 hybrid retrieval function is applied to all ten pilot questions.

For each question, BM25 and E5 independently retrieve the top 50 candidate chunks. Reciprocal Rank Fusion is then applied to the combined candidate set, and the final top five fused chunks are retained.

The resulting dataframe contains the question identifier, question text, final hybrid rank, source-model ranks, RRF score, chunk identifier, judgment metadata, and retrieved chunk text.

In [169]:
# ============================================================
# Retrieve Hybrid Results for the Complete Pilot Benchmark
# ============================================================

all_hybrid_results = []

for _, row in df_pilot.iterrows():

    question_id = row["Question_ID"]
    question = row["Question"]

    retrieved = retrieve_hybrid(
        question,
        candidate_k=HYBRID_CANDIDATES,
        top_k=HYBRID_TOP_K,
        rrf_k=RRF_K
    ).copy()

    retrieved.insert(0, "Question", question)
    retrieved.insert(0, "Question_ID", question_id)

    retrieved["Rank"] = range(1, len(retrieved) + 1)

    all_hybrid_results.append(retrieved)

df_hybrid_results = pd.concat(
    all_hybrid_results,
    ignore_index=True
)

print("=" * 70)
print("HYBRID PILOT RETRIEVAL")
print("=" * 70)

print(f"Pilot Questions : {len(df_pilot)}")
print(f"Retrieved Chunks: {len(df_hybrid_results)}")

display(df_hybrid_results.head(15))

HYBRID PILOT RETRIEVAL
Pilot Questions : 10
Retrieved Chunks: 50


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BM25_Rank,E5_Rank,RRF_Score,chunk_text,Rank
0,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,2.0,1.0,0.032522,20 See Yogesh Singh v. Mahabeer Singh and Othe...,1
1,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,1.0,3.0,0.032266,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",2
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,3.0,2.0,0.032002,"provides that ‘if, from information received o...",3
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,4.0,4.0,0.031250,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4
4,RAW001,Under what circumstances can delay in lodging ...,14762,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,44,9.0,40.0,0.024493,"in the commission of offence, then this delay ...",5
5,RAW002,Can a conviction be sustained solely on the te...,13909,[2024] 7 S.C.R. 365,VINOD JASWANTRAY VYAS (DEAD) THROUGH LRS. vers...,39,1.0,8.0,0.031099,"serious doubt. In other words, the conduct of ...",1
6,RAW002,Can a conviction be sustained solely on the te...,9132,[2024] 12 S.C.R. 769,GEORGE versus THE STATE OF TAMIL NADU AND OTHERS,10,11.0,6.0,0.029236,[2024] 12 S.C.R. 775 George v. The State of Ta...,2
7,RAW002,Can a conviction be sustained solely on the te...,9133,[2024] 12 S.C.R. 769,GEORGE versus THE STATE OF TAMIL NADU AND OTHERS,11,12.0,10.0,0.028175,caused the injury to the deceased. In our cons...,3
8,RAW002,Can a conviction be sustained solely on the te...,24052,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),7,20.0,4.0,0.028125,matches with the blood which was found on the ...,4
9,RAW002,Can a conviction be sustained solely on the te...,24051,[2024] 4 S.C.R. 94,CHANDAN versus THE STATE (DELHI ADMN.),6,22.0,7.0,0.027120,the manner of recovery of the knife from the a...,5


In [170]:
# ============================================================
# Validate Complete Hybrid Pilot Retrieval
# ============================================================

expected_rows = len(df_pilot) * HYBRID_TOP_K

assert len(df_hybrid_results) == expected_rows, (
    f"Expected {expected_rows} hybrid retrieval rows, "
    f"but found {len(df_hybrid_results)}."
)

results_per_question = (
    df_hybrid_results
    .groupby("Question_ID")
    .size()
)

assert (results_per_question == HYBRID_TOP_K).all(), (
    "At least one pilot question does not have exactly "
    "HYBRID_TOP_K hybrid results."
)

assert (
    df_hybrid_results["Question_ID"].nunique()
    == len(df_pilot)
), (
    "Hybrid results do not cover all pilot questions."
)

assert df_hybrid_results["RRF_Score"].notna().all(), (
    "One or more RRF scores are missing."
)

duplicate_check = (
    df_hybrid_results
    .duplicated(
        subset=["Question_ID", "global_chunk_id"]
    )
    .any()
)

assert not duplicate_check, (
    "Duplicate chunks were found within at least one "
    "question's hybrid Top-k results."
)

print("Complete hybrid pilot retrieval validation passed.")

Complete hybrid pilot retrieval validation passed.


## **18.9 Save Hybrid Pilot Retrieval Results**

The hybrid retrieval outputs are saved to the pilot retrieval directory to preserve the fused rankings for subsequent manual relevance annotation and metric computation.

Each saved record contains the pilot question, final hybrid rank, BM25 rank, E5 rank, Reciprocal Rank Fusion score, chunk identifier, judgment metadata, and retrieved chunk text.

In [171]:
# ============================================================
# Save Hybrid Pilot Retrieval Results
# ============================================================

hybrid_output_file = (
    PILOT_RETRIEVAL_DIR /
    "hybrid_pilot_retrieval.csv"
)

df_hybrid_results.to_csv(
    hybrid_output_file,
    index=False
)

print("=" * 60)
print("HYBRID PILOT RESULTS SAVED")
print("=" * 60)

print(f"Location : {hybrid_output_file}")
print(f"Rows     : {len(df_hybrid_results):,}")

HYBRID PILOT RESULTS SAVED
Location : D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\retrieval\hybrid_pilot_retrieval.csv
Rows     : 50


## **18.10 Create the Hybrid Annotation Template**

A manual relevance annotation template is generated from the hybrid retrieval results.

The annotation structure remains identical to that used for TF-IDF, BM25, E5, BGE, and InLegalBERT, ensuring that the hybrid retriever is evaluated under the same relevance criteria and annotation procedure.

In [172]:
# ============================================================
# Create Hybrid Annotation Template
# ============================================================

hybrid_annotation_df = df_hybrid_results.copy()

hybrid_annotation_df["Relevance"] = ""
hybrid_annotation_df["Relevance_Level"] = ""
hybrid_annotation_df["Comments"] = ""

display(hybrid_annotation_df.head())

,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BM25_Rank,E5_Rank,RRF_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,2.0,1.0,0.032522,20 See Yogesh Singh v. Mahabeer Singh and Othe...,1,,,
1,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,1.0,3.0,0.032266,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",2,,,
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,3.0,2.0,0.032002,"provides that ‘if, from information received o...",3,,,
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,4.0,4.0,0.031250,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4,,,
4,RAW001,Under what circumstances can delay in lodging ...,14762,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,44,9.0,40.0,0.024493,"in the commission of offence, then this delay ...",5,,,


## **18.11 Save the Hybrid Annotation Template**

The hybrid annotation template is saved in the pilot annotation directory.

The retrieved chunks will be manually reviewed using the common three-level relevance scale established earlier in the notebook.

In [173]:
# ============================================================
# Save Hybrid Annotation Template
# ============================================================

hybrid_annotation_file = (
    PILOT_ANNOTATION_DIR /
    "hybrid_annotation_template.csv"
)

hybrid_annotation_df.to_csv(
    hybrid_annotation_file,
    index=False
)

print("=" * 60)
print("HYBRID ANNOTATION TEMPLATE CREATED")
print("=" * 60)

print(hybrid_annotation_file)

HYBRID ANNOTATION TEMPLATE CREATED
D:\_ResearchNotebooks\NyayaRAG\evaluation\pilot\annotation\hybrid_annotation_template.csv


# **Part T — Hybrid Retrieval Evaluation Metrics**

The manually annotated hybrid pilot retrieval results are used to compute the same retrieval evaluation metrics applied to the five standalone retrievers.

Using identical relevance criteria, retrieval depth, metric definitions, and per-question aggregation ensures that the BM25–E5 hybrid retriever is evaluated on the same basis as TF-IDF, BM25, E5, BGE, and InLegalBERT.

The hybrid pilot evaluation includes Precision@5, Hit@5, Mean Reciprocal Rank, and nDCG@5. Recall@5 remains deferred until the pooled relevance set has been constructed from the outputs of all retrievers.

In [174]:
## 19.1 Load and Validate the Annotated Hybrid Results

# ============================================================
# Load Annotated Hybrid Pilot Results
# ============================================================

HYBRID_ANNOTATION_FILE = (
    PILOT_ANNOTATION_DIR /
    "hybrid_pilot_annotations.csv"
)

hybrid_annotated_df = pd.read_csv(
    HYBRID_ANNOTATION_FILE
)

print("=" * 70)
print("ANNOTATED HYBRID PILOT RETRIEVAL RESULTS")
print("=" * 70)

print(f"Rows    : {len(hybrid_annotated_df):,}")
print(f"Columns : {len(hybrid_annotated_df.columns)}")

display(hybrid_annotated_df.head())

ANNOTATED HYBRID PILOT RETRIEVAL RESULTS
Rows    : 50
Columns : 14


,Question_ID,Question,global_chunk_id,judgment_id,case_title,local_chunk_id,BM25_Rank,E5_Rank,RRF_Score,chunk_text,Rank,Relevance,Relevance_Level,Comments
0,RAW001,Under what circumstances can delay in lodging ...,11162,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,40,2.0,1.0,0.032522,20 See Yogesh Singh v. Mahabeer Singh and Othe...,1,0,Not relevant,The passage discusses the legal rule for delay...
1,RAW001,Under what circumstances can delay in lodging ...,11163,[2024] 10 S.C.R. 1313,RAMA DEVI versus THE STATE OF BIHAR AND OTHERS,41,1.0,3.0,0.032266,"(PW-1), Mahanth Ashwani Das (PW-25), and Amare...",2,0,Not relevant,The passage provides reasoning on the delay in...
2,RAW001,Under what circumstances can delay in lodging ...,4254,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,16,3.0,2.0,0.032002,"provides that ‘if, from information received o...",3,1,Partially relevant,The passage contains a partial rule that 'dela...
3,RAW001,Under what circumstances can delay in lodging ...,4253,[2024] 6 S.C.R. 409,SHENTO VARGHESE versus JULFIKAR HUSEN & ORS.,15,4.0,4.0,0.031250,15 Nevada Properties (P) Ltd. Vs. State of Mah...,4,1,Partially relevant,The passage states that 'delay in registration...
4,RAW001,Under what circumstances can delay in lodging ...,14762,[2024] 7 S.C.R. 333,GAURAV MAINI versus THE STATE OF HARYANA,44,9.0,40.0,0.024493,"in the commission of offence, then this delay ...",5,0,Not relevant,The passage merely applies the law to the fact...


In [175]:
# ============================================================
# Validate Hybrid Annotation Dataset
# ============================================================

print("=" * 70)
print("HYBRID ANNOTATION VALIDATION")
print("=" * 70)

print(f"Total Records : {len(hybrid_annotated_df)}")

print("\nMissing Values\n")
print(hybrid_annotated_df.isnull().sum())

print("\nRelevance Labels\n")
print(
    hybrid_annotated_df["Relevance"]
    .value_counts()
    .sort_index()
)

print("\nRelevance Levels\n")
print(
    hybrid_annotated_df["Relevance_Level"]
    .value_counts()
)

HYBRID ANNOTATION VALIDATION
Total Records : 50

Missing Values

Question_ID        0
Question           0
global_chunk_id    0
judgment_id        0
case_title         0
local_chunk_id     0
BM25_Rank          2
E5_Rank            2
RRF_Score          0
chunk_text         0
Rank               0
Relevance          0
Relevance_Level    0
Comments           0
dtype: int64

Relevance Labels

Relevance
0    17
1    22
2    11
Name: count, dtype: int64

Relevance Levels

Relevance_Level
Partially relevant    22
Not relevant          17
Highly relevant       11
Name: count, dtype: int64


## **19.2 Create Binary Relevance Labels**

The manually assigned graded relevance scores are converted into binary relevance labels for the computation of Precision@5, Hit@5, and Mean Reciprocal Rank.

The conversion follows the same rule applied throughout the retrieval evaluation:

- Highly Relevant (2) and Partially Relevant (1) are treated as relevant.
- Not Relevant (0) is treated as non-relevant.

The original graded relevance scores are retained for the computation of nDCG@5.

In [176]:
# ============================================================
# Create Binary Relevance Labels for Hybrid
# ============================================================

hybrid_annotated_df["Binary_Relevance"] = (
    hybrid_annotated_df["Relevance"] > 0
).astype(int)

print("=" * 70)
print("HYBRID BINARY RELEVANCE LABELS")
print("=" * 70)

print(
    hybrid_annotated_df["Binary_Relevance"]
    .value_counts()
    .sort_index()
)

display(
    hybrid_annotated_df[
        [
            "Question_ID",
            "Rank",
            "Relevance",
            "Binary_Relevance"
        ]
    ].head(10)
)

HYBRID BINARY RELEVANCE LABELS
Binary_Relevance
0    17
1    33
Name: count, dtype: int64


,Question_ID,Rank,Relevance,Binary_Relevance
0,RAW001,1,0,0
1,RAW001,2,0,0
2,RAW001,3,1,1
3,RAW001,4,1,1
4,RAW001,5,0,0
5,RAW002,1,1,1
6,RAW002,2,2,1
7,RAW002,3,2,1
8,RAW002,4,0,0
9,RAW002,5,0,0


In [177]:
# ============================================================
# Validate Hybrid Binary Relevance Labels
# ============================================================

assert set(
    hybrid_annotated_df["Binary_Relevance"].unique()
).issubset({0, 1}), (
    "Binary relevance labels must contain only 0 and 1."
)

print("Hybrid binary relevance validation completed successfully.")

Hybrid binary relevance validation completed successfully.


## **19.3 Compute Hybrid Retrieval Evaluation Metrics**

The annotated hybrid retrieval results are evaluated using the same metric definitions, relevance rules, retrieval depth, and aggregation procedure applied to all previously evaluated retrieval models.

Precision@5, Hit@5, Mean Reciprocal Rank (MRR), and nDCG@5 are computed for each pilot question and subsequently averaged across the complete pilot benchmark.

Recall@5 remains deferred until a pooled relevance set has been constructed from all evaluated retrievers.

In [178]:
# ============================================================
# Compute Hybrid Pilot Retrieval Metrics
# ============================================================

from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

# -----------------------------
# Precision@5
# -----------------------------
hybrid_precision_per_question = (
    hybrid_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .mean()
    .reset_index(name="Precision@5")
)

hybrid_mean_precision_at_5 = (
    hybrid_precision_per_question["Precision@5"].mean()
)

# -----------------------------
# Hit@5
# -----------------------------
hybrid_hit_per_question = (
    hybrid_annotated_df
    .groupby("Question_ID")["Binary_Relevance"]
    .max()
    .reset_index(name="Hit@5")
)

hybrid_mean_hit_at_5 = (
    hybrid_hit_per_question["Hit@5"].mean()
)

# -----------------------------
# Mean Reciprocal Rank
# -----------------------------
def compute_reciprocal_rank(group):

    relevant_rows = group[
        group["Binary_Relevance"] == 1
    ]

    if relevant_rows.empty:
        return 0.0

    first_relevant_rank = (
        relevant_rows["Rank"].min()
    )

    return 1.0 / first_relevant_rank


hybrid_mrr_per_question = (
    hybrid_annotated_df
    .groupby("Question_ID")
    .apply(
        compute_reciprocal_rank,
        include_groups=False
    )
    .reset_index(name="Reciprocal_Rank")
)

hybrid_mean_mrr = (
    hybrid_mrr_per_question["Reciprocal_Rank"].mean()
)

# -----------------------------
# nDCG@5
# -----------------------------
hybrid_ndcg_records = []

for question_id, group in (
    hybrid_annotated_df.groupby("Question_ID")
):

    group = group.sort_values("Rank")

    true_relevance = (
        group["Relevance"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    predicted_scores = (
        group["RRF_Score"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    ndcg_value = ndcg_score(
        true_relevance,
        predicted_scores,
        k=HYBRID_TOP_K
    )

    hybrid_ndcg_records.append({
        "Question_ID": question_id,
        "nDCG@5": ndcg_value
    })

hybrid_ndcg_per_question = pd.DataFrame(
    hybrid_ndcg_records
)

hybrid_mean_ndcg_at_5 = (
    hybrid_ndcg_per_question["nDCG@5"].mean()
)

# -----------------------------
# Summary
# -----------------------------
hybrid_summary = pd.DataFrame({

    "Retriever": ["Hybrid (BM25 + E5)"],

    "Precision@5": [
        hybrid_mean_precision_at_5
    ],

    "Hit@5": [
        hybrid_mean_hit_at_5
    ],

    "MRR": [
        hybrid_mean_mrr
    ],

    "nDCG@5": [
        hybrid_mean_ndcg_at_5
    ]

})

print("=" * 70)
print("HYBRID PILOT RETRIEVAL METRICS")
print("=" * 70)

display(
    hybrid_summary.round(4)
)

HYBRID PILOT RETRIEVAL METRICS


,Retriever,Precision@5,Hit@5,MRR,nDCG@5
0,Hybrid (BM25 + E5),0.66,1.0,0.9333,0.9362


In [179]:
# ============================================================
# Validate Hybrid Retrieval Metrics
# ============================================================

assert (
    hybrid_precision_per_question["Precision@5"]
    .between(0, 1)
    .all()
)

assert (
    hybrid_hit_per_question["Hit@5"]
    .isin([0, 1])
    .all()
)

assert (
    hybrid_mrr_per_question["Reciprocal_Rank"]
    .between(0, 1)
    .all()
)

assert (
    hybrid_ndcg_per_question["nDCG@5"]
    .between(0, 1)
    .all()
)

assert len(hybrid_precision_per_question) == len(df_pilot)
assert len(hybrid_hit_per_question) == len(df_pilot)
assert len(hybrid_mrr_per_question) == len(df_pilot)
assert len(hybrid_ndcg_per_question) == len(df_pilot)

print(
    "Hybrid retrieval metric validation completed successfully."
)

Hybrid retrieval metric validation completed successfully.


## **19.4 Summary of Hybrid Pilot Evaluation**

The BM25–E5 hybrid retrieval model was successfully evaluated using the same pilot benchmark, annotation protocol, and evaluation methodology applied to the standalone retrieval models.

Among the evaluated retrievers, the hybrid model achieved the highest Precision@5 and nDCG@5, indicating that combining lexical and semantic retrieval through Reciprocal Rank Fusion improves the overall relevance and ranking quality of the retrieved legal evidence.

Although the E5 semantic retriever achieved a slightly higher Mean Reciprocal Rank, the hybrid model demonstrated the strongest overall retrieval performance on the pilot benchmark. These preliminary findings support the use of hybrid retrieval within the proposed NYAYaRAG framework.

In [180]:
# ============================================================
# Final Pilot Retrieval Performance
# ============================================================

pilot_results = pd.DataFrame({

    "Retriever": [
        "TF-IDF",
        "BM25",
        "E5",
        "BGE",
        "InLegalBERT",
        "Hybrid (BM25 + E5)"
    ],

    "Type": [
        "Lexical",
        "Lexical",
        "Semantic",
        "Semantic",
        "Semantic",
        "Hybrid"
    ],

    "Precision@5": [
        mean_precision_at_5,
        bm25_mean_precision_at_5,
        e5_mean_precision_at_5,
        bge_mean_precision_at_5,
        inlegalbert_mean_precision_at_5,
        hybrid_mean_precision_at_5
    ],

    "Hit@5": [
        mean_hit_at_5,
        bm25_mean_hit_at_5,
        e5_mean_hit_at_5,
        bge_mean_hit_at_5,
        inlegalbert_mean_hit_at_5,
        hybrid_mean_hit_at_5
    ],

    "MRR": [
        mean_mrr,
        bm25_mean_mrr,
        e5_mean_mrr,
        bge_mean_mrr,
        inlegalbert_mean_mrr,
        hybrid_mean_mrr
    ],

    "nDCG@5": [
        mean_ndcg,
        bm25_mean_ndcg_at_5,
        e5_mean_ndcg_at_5,
        bge_mean_ndcg_at_5,
        inlegalbert_mean_ndcg_at_5,
        hybrid_mean_ndcg_at_5
    ]

})

pilot_results = pilot_results.round(4)

print("=" * 90)
print("FINAL PILOT RETRIEVAL PERFORMANCE COMPARISON")
print("=" * 90)

display(pilot_results)

FINAL PILOT RETRIEVAL PERFORMANCE COMPARISON


,Retriever,Type,Precision@5,Hit@5,MRR,nDCG@5
0,TF-IDF,Lexical,0.64,0.9,0.7167,0.7744
1,BM25,Lexical,0.68,0.9,0.7167,0.7521
2,E5,Semantic,0.56,0.8,0.7000,0.6673
3,BGE,Semantic,0.64,0.8,0.7333,0.7072
4,InLegalBERT,Semantic,0.48,0.8,0.5750,0.6449
5,Hybrid (BM25 + E5),Hybrid,0.66,1.0,0.9333,0.9362


## **20.1 Overall Retrieval Performance**


In [181]:
print("=" * 90)
print("OVERALL PILOT RETRIEVAL PERFORMANCE")
print("=" * 90)

display(pilot_results)

OVERALL PILOT RETRIEVAL PERFORMANCE


,Retriever,Type,Precision@5,Hit@5,MRR,nDCG@5
0,TF-IDF,Lexical,0.64,0.9,0.7167,0.7744
1,BM25,Lexical,0.68,0.9,0.7167,0.7521
2,E5,Semantic,0.56,0.8,0.7000,0.6673
3,BGE,Semantic,0.64,0.8,0.7333,0.7072
4,InLegalBERT,Semantic,0.48,0.8,0.5750,0.6449
5,Hybrid (BM25 + E5),Hybrid,0.66,1.0,0.9333,0.9362


### **Observations**

The pilot evaluation demonstrates noticeable differences in retrieval effectiveness across the evaluated retrieval models.

Among the standalone retrieval models, the E5 semantic retriever achieved the highest Mean Reciprocal Rank, indicating that it most frequently placed the first relevant legal passage at the highest rank.

The hybrid BM25–E5 retriever achieved the highest Precision@5 and nDCG@5, suggesting that combining lexical and semantic retrieval through Reciprocal Rank Fusion improved the overall quality of the retrieved legal evidence.

In contrast, InLegalBERT demonstrated the lowest retrieval performance across all evaluated metrics on the pilot benchmark.

These descriptive findings provide preliminary evidence that retrieval effectiveness differs across retrieval methodologies and motivate the inferential statistical analyses presented in the subsequent sections.

# **Part V — Research Question 1**

## **RQ1**

**Do semantic retrieval methodologies significantly outperform lexical retrieval methodologies for legal information retrieval?**

The first research question evaluates whether dense semantic retrieval methods achieve significantly better retrieval performance than traditional lexical retrieval methods on the pilot benchmark.

The comparison is performed at the retrieval-paradigm level rather than the individual model level.

The retrieval paradigms are defined as follows:

- **Lexical Retrieval:** TF-IDF and BM25
- **Semantic Retrieval:** E5, BGE, and InLegalBERT

For each pilot question, the retrieval performance of the lexical paradigm is represented by the average performance of TF-IDF and BM25, whereas the semantic paradigm is represented by the average performance of E5, BGE, and InLegalBERT.

The paired observations are analysed using a **one-sided Wilcoxon signed-rank test**, which evaluates whether semantic retrieval significantly outperforms lexical retrieval.

The analysis is performed separately for each retrieval evaluation metric.

## **21.2 Prepare Per-Question Retrieval Metrics**

RQ1 requires paired performance observations for the lexical and semantic retrieval paradigms.

For each pilot question:

- the **lexical score** is calculated as the mean performance of TF-IDF and BM25;
- the **semantic score** is calculated as the mean performance of E5, BGE, and InLegalBERT.

The hybrid retriever is excluded because it combines both lexical and semantic retrieval and therefore does not belong exclusively to either comparison group.

The paradigm-level scores are calculated separately for Precision@5, Hit@5, MRR, and nDCG@5.

In [182]:
# ============================================================
# Prepare Per-Question Metrics for RQ1
# ============================================================

# Rename per-question metric columns before merging
tfidf_rq1 = (
    precision_at_5
    .rename(columns={"Precision@5": "TFIDF_Precision@5"})
    .merge(
        hit_at_5.rename(columns={"Hit@5": "TFIDF_Hit@5"}),
        on="Question_ID"
    )
    .merge(
        mrr_per_question.rename(
            columns={"Reciprocal_Rank": "TFIDF_MRR"}
        ),
        on="Question_ID"
    )
    .merge(
        ndcg_results.rename(
            columns={"nDCG@5": "TFIDF_nDCG@5"}
        ),
        on="Question_ID"
    )
)

bm25_rq1 = (
    bm25_precision_per_question
    .rename(columns={"Precision@5": "BM25_Precision@5"})
    .merge(
        bm25_hit_per_question.rename(
            columns={"Hit@5": "BM25_Hit@5"}
        ),
        on="Question_ID"
    )
    .merge(
        bm25_mrr_per_question.rename(
            columns={"Reciprocal_Rank": "BM25_MRR"}
        ),
        on="Question_ID"
    )
    .merge(
        bm25_ndcg_per_question.rename(
            columns={"nDCG@5": "BM25_nDCG@5"}
        ),
        on="Question_ID"
    )
)

e5_rq1 = (
    e5_precision_per_question
    .rename(columns={"Precision@5": "E5_Precision@5"})
    .merge(
        e5_hit_per_question.rename(
            columns={"Hit@5": "E5_Hit@5"}
        ),
        on="Question_ID"
    )
    .merge(
        e5_mrr_per_question.rename(
            columns={"Reciprocal_Rank": "E5_MRR"}
        ),
        on="Question_ID"
    )
    .merge(
        e5_ndcg_per_question.rename(
            columns={"nDCG@5": "E5_nDCG@5"}
        ),
        on="Question_ID"
    )
)

bge_rq1 = (
    bge_precision_per_question
    .rename(columns={"Precision@5": "BGE_Precision@5"})
    .merge(
        bge_hit_per_question.rename(
            columns={"Hit@5": "BGE_Hit@5"}
        ),
        on="Question_ID"
    )
    .merge(
        bge_mrr_per_question.rename(
            columns={"Reciprocal_Rank": "BGE_MRR"}
        ),
        on="Question_ID"
    )
    .merge(
        bge_ndcg_per_question.rename(
            columns={"nDCG@5": "BGE_nDCG@5"}
        ),
        on="Question_ID"
    )
)

inlegalbert_rq1 = (
    inlegalbert_precision_per_question
    .rename(
        columns={"Precision@5": "InLegalBERT_Precision@5"}
    )
    .merge(
        inlegalbert_hit_per_question.rename(
            columns={"Hit@5": "InLegalBERT_Hit@5"}
        ),
        on="Question_ID"
    )
    .merge(
        inlegalbert_mrr_per_question.rename(
            columns={"Reciprocal_Rank": "InLegalBERT_MRR"}
        ),
        on="Question_ID"
    )
    .merge(
        inlegalbert_ndcg_per_question.rename(
            columns={"nDCG@5": "InLegalBERT_nDCG@5"}
        ),
        on="Question_ID"
    )
)

# Combine all standalone retriever metrics
rq1_metrics = (
    tfidf_rq1
    .merge(bm25_rq1, on="Question_ID")
    .merge(e5_rq1, on="Question_ID")
    .merge(bge_rq1, on="Question_ID")
    .merge(inlegalbert_rq1, on="Question_ID")
)

# Compute paradigm-level averages
rq1_metrics["Lexical_Precision@5"] = (
    rq1_metrics[
        ["TFIDF_Precision@5", "BM25_Precision@5"]
    ].mean(axis=1)
)

rq1_metrics["Semantic_Precision@5"] = (
    rq1_metrics[
        [
            "E5_Precision@5",
            "BGE_Precision@5",
            "InLegalBERT_Precision@5"
        ]
    ].mean(axis=1)
)

rq1_metrics["Lexical_Hit@5"] = (
    rq1_metrics[
        ["TFIDF_Hit@5", "BM25_Hit@5"]
    ].mean(axis=1)
)

rq1_metrics["Semantic_Hit@5"] = (
    rq1_metrics[
        ["E5_Hit@5", "BGE_Hit@5", "InLegalBERT_Hit@5"]
    ].mean(axis=1)
)

rq1_metrics["Lexical_MRR"] = (
    rq1_metrics[
        ["TFIDF_MRR", "BM25_MRR"]
    ].mean(axis=1)
)

rq1_metrics["Semantic_MRR"] = (
    rq1_metrics[
        ["E5_MRR", "BGE_MRR", "InLegalBERT_MRR"]
    ].mean(axis=1)
)

rq1_metrics["Lexical_nDCG@5"] = (
    rq1_metrics[
        ["TFIDF_nDCG@5", "BM25_nDCG@5"]
    ].mean(axis=1)
)

rq1_metrics["Semantic_nDCG@5"] = (
    rq1_metrics[
        [
            "E5_nDCG@5",
            "BGE_nDCG@5",
            "InLegalBERT_nDCG@5"
        ]
    ].mean(axis=1)
)

rq1_comparison = rq1_metrics[
    [
        "Question_ID",
        "Lexical_Precision@5",
        "Semantic_Precision@5",
        "Lexical_Hit@5",
        "Semantic_Hit@5",
        "Lexical_MRR",
        "Semantic_MRR",
        "Lexical_nDCG@5",
        "Semantic_nDCG@5"
    ]
].copy()

display(rq1_comparison.round(4))

,Question_ID,Lexical_Precision@5,Semantic_Precision@5,Lexical_Hit@5,Semantic_Hit@5,Lexical_MRR,Semantic_MRR,Lexical_nDCG@5,Semantic_nDCG@5
0,RAW001,0.6,0.4667,1.0,1.0000,0.6667,0.8333,0.7853,0.8311
1,RAW002,0.5,0.6000,1.0,1.0000,0.7500,0.6667,0.8336,0.7335
2,RAW004,0.9,0.8667,1.0,1.0000,0.7500,1.0000,0.8131,0.9285
3,RAW005,1.0,0.9333,1.0,1.0000,1.0000,1.0000,0.8995,0.9173
4,RAW006,0.6,0.2000,1.0,0.3333,0.6667,0.3333,0.7853,0.2767
5,RAW007,0.5,0.6000,1.0,1.0000,1.0000,0.8333,0.9735,0.8479
6,RAW010,1.0,0.6667,1.0,1.0000,1.0000,0.7500,0.9235,0.8281
7,RAW012,0.0,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000
8,RAW015,0.7,0.3333,1.0,0.6667,0.6667,0.4444,0.8091,0.5146
9,RAW017,0.8,0.9333,1.0,1.0000,0.6667,0.8333,0.8091,0.8538


In [183]:
# ============================================================
# Validate RQ1 Paired Dataset
# ============================================================

assert len(rq1_comparison) == len(df_pilot), (
    "RQ1 paired dataset does not contain all pilot questions."
)

assert rq1_comparison["Question_ID"].is_unique, (
    "Duplicate Question_ID values found in the RQ1 dataset."
)

assert rq1_comparison.drop(
    columns="Question_ID"
).notna().all().all(), (
    "Missing values detected in the RQ1 paired observations."
)

assert rq1_comparison.drop(
    columns="Question_ID"
).apply(lambda column: column.between(0, 1).all()).all(), (
    "RQ1 metric values must lie between 0 and 1."
)

print("RQ1 paired metric dataset validated successfully.")

RQ1 paired metric dataset validated successfully.


## **21.3 Descriptive Statistics**

Before conducting the inferential statistical analysis, descriptive statistics are computed for the lexical and semantic retrieval paradigms.

The mean and standard deviation of each retrieval metric are reported to provide an overview of the relative retrieval performance prior to hypothesis testing.

In [184]:
# ============================================================
# Descriptive Statistics for RQ1
# ============================================================

rq1_summary = pd.DataFrame({

    "Metric": [
        "Precision@5",
        "Hit@5",
        "MRR",
        "nDCG@5"
    ],

    "Lexical Mean": [

        rq1_comparison["Lexical_Precision@5"].mean(),

        rq1_comparison["Lexical_Hit@5"].mean(),

        rq1_comparison["Lexical_MRR"].mean(),

        rq1_comparison["Lexical_nDCG@5"].mean()

    ],

    "Semantic Mean": [

        rq1_comparison["Semantic_Precision@5"].mean(),

        rq1_comparison["Semantic_Hit@5"].mean(),

        rq1_comparison["Semantic_MRR"].mean(),

        rq1_comparison["Semantic_nDCG@5"].mean()

    ],

    "Lexical SD": [

        rq1_comparison["Lexical_Precision@5"].std(),

        rq1_comparison["Lexical_Hit@5"].std(),

        rq1_comparison["Lexical_MRR"].std(),

        rq1_comparison["Lexical_nDCG@5"].std()

    ],

    "Semantic SD": [

        rq1_comparison["Semantic_Precision@5"].std(),

        rq1_comparison["Semantic_Hit@5"].std(),

        rq1_comparison["Semantic_MRR"].std(),

        rq1_comparison["Semantic_nDCG@5"].std()

    ]

})

display(
    rq1_summary.round(4)
)

,Metric,Lexical Mean,Semantic Mean,Lexical SD,Semantic SD
0,Precision@5,0.6600,0.5600,0.2989,0.3146
1,Hit@5,0.9000,0.8000,0.3162,0.3583
2,MRR,0.7167,0.6694,0.2919,0.3193
3,nDCG@5,0.7632,0.6732,0.2756,0.3120


## **21.4 Wilcoxon Signed-Rank Test**

The lexical and semantic retrieval paradigms are compared using the one-sided Wilcoxon signed-rank test.

The Wilcoxon signed-rank test is appropriate because:

- retrieval performance is measured on the same set of pilot questions;
- the observations are paired;
- the sample size is small; and
- no assumption of normality is required.

The test is performed separately for Precision@5, Hit@5, MRR, and nDCG@5 at a significance level of α = 0.05.

In [185]:
# ============================================================
# Wilcoxon Signed-Rank Test Function
# ============================================================

from scipy.stats import wilcoxon
import numpy as np
import pandas as pd

ALPHA = 0.05


def run_wilcoxon_test(
    data,
    lexical_column,
    semantic_column,
    metric_name
):
    """
    Performs a one-sided Wilcoxon signed-rank test
    comparing Semantic > Lexical.
    """

    lexical = data[lexical_column]

    semantic = data[semantic_column]

    # One-sided test
    statistic, p_value = wilcoxon(
        semantic,
        lexical,
        alternative="greater",
        zero_method="wilcox"
    )

    # Effect size
    n = np.sum((semantic - lexical) != 0)

    z = (
        statistic
        - (n * (n + 1) / 4)
    ) / np.sqrt(
        n * (n + 1) * (2 * n + 1) / 24
    )

    effect_size = abs(z) / np.sqrt(n)

    decision = (
        "Reject H₀"
        if p_value < ALPHA
        else "Fail to Reject H₀"
    )

    return {

        "Metric": metric_name,

        "Wilcoxon Statistic": statistic,

        "p-value": p_value,

        "Effect Size (r)": effect_size,

        "Decision": decision

    }

In [186]:
# ============================================================
# Validate Wilcoxon Function
# ============================================================

assert callable(run_wilcoxon_test)

print(
    "Wilcoxon signed-rank function created successfully."
)

Wilcoxon signed-rank function created successfully.


## **21.6 Perform the Wilcoxon Signed-Rank Tests**

The Wilcoxon signed-rank test is applied separately to each retrieval evaluation metric.

For each metric, the lexical and semantic retrieval paradigms are compared using paired observations derived from the pilot benchmark.

The resulting p-values are interpreted at a significance level of α = 0.05.

In [187]:
# ============================================================
# Perform Wilcoxon Signed-Rank Tests
# ============================================================

rq1_results = pd.DataFrame([

    run_wilcoxon_test(
        rq1_comparison,
        "Lexical_Precision@5",
        "Semantic_Precision@5",
        "Precision@5"
    ),

    run_wilcoxon_test(
        rq1_comparison,
        "Lexical_Hit@5",
        "Semantic_Hit@5",
        "Hit@5"
    ),

    run_wilcoxon_test(
        rq1_comparison,
        "Lexical_MRR",
        "Semantic_MRR",
        "MRR"
    ),

    run_wilcoxon_test(
        rq1_comparison,
        "Lexical_nDCG@5",
        "Semantic_nDCG@5",
        "nDCG@5"
    )

])

rq1_results = rq1_results.round(4)

print("=" * 90)
print("RQ1 WILCOXON SIGNED-RANK TEST RESULTS")
print("=" * 90)

display(rq1_results)

RQ1 WILCOXON SIGNED-RANK TEST RESULTS


,Metric,Wilcoxon Statistic,p-value,Effect Size (r),Decision
0,Precision@5,12.0,0.8984,0.4146,Fail to Reject H₀
1,Hit@5,0.0,1.0000,0.9487,Fail to Reject H₀
2,MRR,13.5,0.7344,0.2228,Fail to Reject H₀
3,nDCG@5,12.0,0.8984,0.4146,Fail to Reject H₀


In [188]:
# ============================================================
# Validate RQ1 Statistical Results
# ============================================================

assert len(rq1_results) == 4

assert rq1_results["p-value"].between(0, 1).all()

assert rq1_results["Effect Size (r)"].between(0, 1).all()

print("RQ1 statistical analysis completed successfully.")

RQ1 statistical analysis completed successfully.


## **21.7 Interpretation of the Wilcoxon Signed-Rank Test**

The one-sided Wilcoxon signed-rank test was performed separately for each retrieval evaluation metric to determine whether the semantic retrieval paradigm significantly outperformed the lexical retrieval paradigm.

For all four evaluation metrics, the obtained p-values exceeded the significance threshold of α = 0.05. Consequently, the null hypothesis could not be rejected for Precision@5, Hit@5, Mean Reciprocal Rank (MRR), or nDCG@5.

Although the descriptive statistics indicated small differences between the lexical and semantic retrieval paradigms across individual metrics, these differences were not statistically significant on the pilot benchmark.

These findings suggest that, based on the pilot dataset comprising ten legal research questions, there is insufficient evidence to conclude that semantic retrieval methods outperform lexical retrieval methods.

## **21.8 Conclusion for RQ1**

The pilot evaluation does not provide statistically significant evidence that semantic retrieval methodologies outperform lexical retrieval methodologies for legal information retrieval.

Accordingly, the null hypothesis is retained for all evaluated retrieval metrics.

These results are consistent with the limited size of the pilot benchmark and should be regarded as preliminary. The formal evaluation conducted on the complete benchmark of fifty legal research questions will provide greater statistical power and serve as the basis for the final conclusions reported in the dissertation.

# **Part W — Research Question 2**

## **RQ2**

**Which retrieval methodology performs best overall for legal information retrieval?**

The second research question compares the retrieval performance of all evaluated retrieval methodologies.

Unlike RQ1, which compares retrieval paradigms, RQ2 evaluates each retrieval model individually.

The retrieval methodologies included in the analysis are:

- TF-IDF
- BM25
- E5
- BGE
- InLegalBERT
- Hybrid (BM25 + E5)

The comparison is performed using the Friedman test, which is appropriate for comparing more than two related retrieval methods evaluated on the same benchmark questions.

Where the Friedman test indicates statistically significant differences, pairwise Wilcoxon signed-rank post-hoc comparisons with Holm correction are performed to identify which retrieval models differ significantly.

## **22.1 Statistical Hypotheses**

The following hypotheses are tested separately for each retrieval evaluation metric.

**Null Hypothesis (H₀)**

There is no statistically significant difference in retrieval performance among the evaluated retrieval methodologies.

$$
H_0:
\text{All retrieval methodologies perform equally.}
$$

**Alternative Hypothesis (H₁)**

At least one retrieval methodology performs significantly differently from the others.

$$
H_1:
\text{At least one retrieval methodology differs significantly.}
$$

The hypotheses are evaluated using the Friedman test at a significance level of **α = 0.05**.

## **22.2 Prepare the Friedman Test Dataset**

The Friedman test requires paired observations for each retrieval methodology evaluated on the same benchmark questions.

Accordingly, separate paired datasets are prepared for each retrieval evaluation metric. Each row represents one pilot question, while each column represents the performance of a particular retrieval methodology.

The prepared datasets form the input to the Friedman test and subsequent pairwise post-hoc comparisons.

In [189]:
# ============================================================
# Prepare Friedman Dataset
# ============================================================

def prepare_friedman_dataset(
    metric_name,
    tfidf_df,
    bm25_df,
    e5_df,
    bge_df,
    inlegalbert_df,
    hybrid_df,
    metric_column
):
    """
    Creates the paired dataset required for the Friedman test.
    """

    dataset = (
        tfidf_df[
            ["Question_ID", metric_column]
        ].rename(
            columns={metric_column: "TF-IDF"}
        )
        .merge(
            bm25_df[
                ["Question_ID", metric_column]
            ].rename(
                columns={metric_column: "BM25"}
            ),
            on="Question_ID"
        )
        .merge(
            e5_df[
                ["Question_ID", metric_column]
            ].rename(
                columns={metric_column: "E5"}
            ),
            on="Question_ID"
        )
        .merge(
            bge_df[
                ["Question_ID", metric_column]
            ].rename(
                columns={metric_column: "BGE"}
            ),
            on="Question_ID"
        )
        .merge(
            inlegalbert_df[
                ["Question_ID", metric_column]
            ].rename(
                columns={metric_column: "InLegalBERT"}
            ),
            on="Question_ID"
        )
        .merge(
            hybrid_df[
                ["Question_ID", metric_column]
            ].rename(
                columns={metric_column: "Hybrid"}
            ),
            on="Question_ID"
        )
    )

    print("=" * 70)
    print(metric_name.upper())
    print("=" * 70)

    display(dataset.round(4))

    return dataset

In [190]:
assert callable(
    prepare_friedman_dataset
)

print(
    "Friedman dataset preparation function created successfully."
)

Friedman dataset preparation function created successfully.


## **22.3 Create the Friedman Test Datasets**

The paired datasets for Precision@5, Hit@5, Mean Reciprocal Rank (MRR), and nDCG@5 are prepared using the common dataset preparation function.

Each dataset contains one observation per pilot question for every evaluated retrieval methodology and forms the input to the Friedman test.

In [191]:
# ============================================================
# Create Friedman Datasets
# ============================================================

friedman_precision = prepare_friedman_dataset(
    metric_name="Precision@5",
    tfidf_df=precision_at_5,
    bm25_df=bm25_precision_per_question,
    e5_df=e5_precision_per_question,
    bge_df=bge_precision_per_question,
    inlegalbert_df=inlegalbert_precision_per_question,
    hybrid_df=hybrid_precision_per_question,
    metric_column="Precision@5"
)

friedman_hit = prepare_friedman_dataset(
    metric_name="Hit@5",
    tfidf_df=hit_at_5,
    bm25_df=bm25_hit_per_question,
    e5_df=e5_hit_per_question,
    bge_df=bge_hit_per_question,
    inlegalbert_df=inlegalbert_hit_per_question,
    hybrid_df=hybrid_hit_per_question,
    metric_column="Hit@5"
)

friedman_mrr = prepare_friedman_dataset(
    metric_name="MRR",
    tfidf_df=mrr_per_question.rename(
        columns={"Reciprocal_Rank": "MRR"}
    ),
    bm25_df=bm25_mrr_per_question.rename(
        columns={"Reciprocal_Rank": "MRR"}
    ),
    e5_df=e5_mrr_per_question.rename(
        columns={"Reciprocal_Rank": "MRR"}
    ),
    bge_df=bge_mrr_per_question.rename(
        columns={"Reciprocal_Rank": "MRR"}
    ),
    inlegalbert_df=inlegalbert_mrr_per_question.rename(
        columns={"Reciprocal_Rank": "MRR"}
    ),
    hybrid_df=hybrid_mrr_per_question.rename(
        columns={"Reciprocal_Rank": "MRR"}
    ),
    metric_column="MRR"
)

friedman_ndcg = prepare_friedman_dataset(
    metric_name="nDCG@5",
    tfidf_df=ndcg_results,
    bm25_df=bm25_ndcg_per_question,
    e5_df=e5_ndcg_per_question,
    bge_df=bge_ndcg_per_question,
    inlegalbert_df=inlegalbert_ndcg_per_question,
    hybrid_df=hybrid_ndcg_per_question,
    metric_column="nDCG@5"
)

PRECISION@5


,Question_ID,TF-IDF,BM25,E5,BGE,InLegalBERT,Hybrid
0,RAW001,0.4,0.8,0.6,0.4,0.4,0.4
1,RAW002,0.4,0.6,0.4,0.8,0.6,0.6
2,RAW004,0.8,1.0,0.8,1.0,0.8,1.0
3,RAW005,1.0,1.0,1.0,1.0,0.8,0.4
4,RAW006,0.8,0.4,0.6,0.0,0.0,0.8
5,RAW007,0.6,0.4,0.4,1.0,0.4,0.4
6,RAW010,1.0,1.0,0.8,0.8,0.4,0.8
7,RAW012,0.0,0.0,0.0,0.0,0.0,0.2
8,RAW015,0.8,0.6,0.0,0.4,0.6,1.0
9,RAW017,0.6,1.0,1.0,1.0,0.8,1.0


HIT@5


,Question_ID,TF-IDF,BM25,E5,BGE,InLegalBERT,Hybrid
0,RAW001,1,1,1,1,1,1
1,RAW002,1,1,1,1,1,1
2,RAW004,1,1,1,1,1,1
3,RAW005,1,1,1,1,1,1
4,RAW006,1,1,1,0,0,1
5,RAW007,1,1,1,1,1,1
6,RAW010,1,1,1,1,1,1
7,RAW012,0,0,0,0,0,1
8,RAW015,1,1,0,1,1,1
9,RAW017,1,1,1,1,1,1


MRR


,Question_ID,TF-IDF,BM25,E5,BGE,InLegalBERT,Hybrid
0,RAW001,0.3333,1.0000,0.5,1.0000,1.00,0.3333
1,RAW002,1.0000,0.5000,0.5,1.0000,0.50,1.0000
2,RAW004,0.5000,1.0000,1.0,1.0000,1.00,1.0000
3,RAW005,1.0000,1.0000,1.0,1.0000,1.00,1.0000
4,RAW006,1.0000,0.3333,1.0,0.0000,0.00,1.0000
5,RAW007,1.0000,1.0000,1.0,1.0000,0.50,1.0000
6,RAW010,1.0000,1.0000,1.0,1.0000,0.25,1.0000
7,RAW012,0.0000,0.0000,0.0,0.0000,0.00,1.0000
8,RAW015,1.0000,0.3333,0.0,0.3333,1.00,1.0000
9,RAW017,0.3333,1.0000,1.0,1.0000,0.50,1.0000


NDCG@5


,Question_ID,TF-IDF,BM25,E5,BGE,InLegalBERT,Hybrid
0,RAW001,0.5706,1.0000,0.5862,0.9072,1.0000,0.5706
1,RAW002,1.0000,0.6672,0.6509,0.8372,0.7123,0.8671
2,RAW004,0.7606,0.8656,0.8807,1.0000,0.9047,0.9777
3,RAW005,0.8471,0.9519,0.8224,0.9467,0.9829,1.0000
4,RAW006,1.0000,0.5706,0.8302,0.0000,0.0000,1.0000
5,RAW007,0.9469,1.0000,0.9197,1.0000,0.6241,1.0000
6,RAW010,1.0000,0.8471,0.9829,1.0000,0.5013,1.0000
7,RAW012,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
8,RAW015,1.0000,0.6183,0.0000,0.5438,1.0000,1.0000
9,RAW017,0.6183,1.0000,1.0000,0.8375,0.7239,0.9467


In [192]:
# ============================================================
# Validate Friedman Datasets
# ============================================================

datasets = {
    "Precision": friedman_precision,
    "Hit": friedman_hit,
    "MRR": friedman_mrr,
    "nDCG": friedman_ndcg
}

for name, df in datasets.items():

    assert len(df) == len(df_pilot), (
        f"{name}: Incorrect number of questions."
    )

    assert df["Question_ID"].is_unique, (
        f"{name}: Duplicate Question_ID values."
    )

    assert df.drop(
        columns="Question_ID"
    ).notna().all().all(), (
        f"{name}: Missing values detected."
    )

print("All Friedman datasets validated successfully.")

All Friedman datasets validated successfully.


## **22.4 Define the Friedman Test Function**

A reusable function is defined to perform the Friedman test for comparing multiple related retrieval methodologies evaluated on the same benchmark questions.

For each retrieval evaluation metric, the function reports:

- Friedman chi-square statistic;
- degrees of freedom;
- p-value;
- Kendall's coefficient of concordance (W) as the effect size; and
- the statistical decision.

The function is subsequently applied to each retrieval evaluation metric.

In [193]:
# ============================================================
# Friedman Test Function
# ============================================================

from scipy.stats import friedmanchisquare
import pandas as pd

ALPHA = 0.05

def run_friedman_test(df, metric_name):
    """
    Performs the Friedman test and computes Kendall's W.
    """

    methods = df.columns[1:]      # Exclude Question_ID
    scores = [df[m] for m in methods]

    statistic, p_value = friedmanchisquare(*scores)

    n = len(df)                   # Number of questions
    k = len(methods)              # Number of retrieval methods

    kendalls_w = statistic / (n * (k - 1))

    decision = (
        "Reject H₀"
        if p_value < ALPHA
        else "Fail to Reject H₀"
    )

    return {

        "Metric": metric_name,
        "Chi-Square": statistic,
        "df": k - 1,
        "p-value": p_value,
        "Kendall's W": kendalls_w,
        "Decision": decision

    }

In [194]:
assert callable(run_friedman_test)

print("Friedman test function created successfully.")

Friedman test function created successfully.


## **22.5 Perform the Friedman Tests**

The Friedman test is applied separately to each retrieval evaluation metric.

The test evaluates whether statistically significant differences exist among the six evaluated retrieval methodologies:

- TF-IDF
- BM25
- E5
- BGE
- InLegalBERT
- Hybrid (BM25 + E5)

The null hypothesis is rejected when the p-value is less than the significance level of α = 0.05.

In [195]:
# ============================================================
# Perform Friedman Tests
# ============================================================

friedman_results = pd.DataFrame([

    run_friedman_test(
        friedman_precision,
        "Precision@5"
    ),

    run_friedman_test(
        friedman_hit,
        "Hit@5"
    ),

    run_friedman_test(
        friedman_mrr,
        "MRR"
    ),

    run_friedman_test(
        friedman_ndcg,
        "nDCG@5"
    )

])

friedman_results = friedman_results.round(4)

print("=" * 90)
print("FRIEDMAN TEST RESULTS")
print("=" * 90)

display(friedman_results)

FRIEDMAN TEST RESULTS


,Metric,Chi-Square,df,p-value,Kendall's W,Decision
0,Precision@5,7.9368,5,0.1598,0.1587,Fail to Reject H₀
1,Hit@5,5.5556,5,0.3519,0.1111,Fail to Reject H₀
2,MRR,5.6683,5,0.3398,0.1134,Fail to Reject H₀
3,nDCG@5,7.4766,5,0.1875,0.1495,Fail to Reject H₀


In [196]:
# ============================================================
# Validate Friedman Results
# ============================================================

assert len(friedman_results) == 4

assert friedman_results["p-value"].between(0, 1).all()

assert friedman_results["Kendall's W"].between(0, 1).all()

print("Friedman analysis completed successfully.")

Friedman analysis completed successfully.


## **22.6 Compute Friedman Mean Ranks**

The Friedman test evaluates whether statistically significant differences exist among the evaluated retrieval methodologies but does not indicate which retrieval method performs best.

Accordingly, Friedman mean ranks are computed for each retrieval evaluation metric. Higher mean ranks indicate superior retrieval performance relative to the other evaluated methodologies.

These rankings are subsequently interpreted alongside the Friedman test and post-hoc pairwise comparisons.

In [197]:
# ============================================================
# Compute Friedman Mean Ranks
# ============================================================

from scipy.stats import rankdata
import pandas as pd

def compute_mean_ranks(df, metric_name):
    """
    Compute Friedman mean ranks.
    Higher values receive higher ranks.
    """

    methods = df.columns[1:]

    # Rank each question (highest score gets highest rank)
    ranks = df[methods].apply(
        lambda row: rankdata(row, method="average"),
        axis=1,
        result_type="expand"
    )

    ranks.columns = methods

    mean_ranks = ranks.mean().sort_values(ascending=False)

    results = pd.DataFrame({
        "Retriever": mean_ranks.index,
        "Mean Rank": mean_ranks.values
    })

    print("=" * 80)
    print(metric_name.upper())
    print("=" * 80)

    display(results.round(3))

    return results

In [198]:
assert callable(compute_mean_ranks)

print("Mean rank function created successfully.")

Mean rank function created successfully.


In [199]:
# ============================================================
# Friedman Mean Ranks
# ============================================================

precision_ranks = compute_mean_ranks(
    friedman_precision,
    "Precision@5"
)

hit_ranks = compute_mean_ranks(
    friedman_hit,
    "Hit@5"
)

mrr_ranks = compute_mean_ranks(
    friedman_mrr,
    "MRR"
)

ndcg_ranks = compute_mean_ranks(
    friedman_ndcg,
    "nDCG@5"
)

PRECISION@5


,Retriever,Mean Rank
0,BM25,4.15
1,Hybrid,4.00
2,BGE,3.80
3,TF-IDF,3.55
4,E5,3.10
5,InLegalBERT,2.40


HIT@5


,Retriever,Mean Rank
0,Hybrid,3.9
1,TF-IDF,3.6
2,BM25,3.6
3,E5,3.3
4,BGE,3.3
5,InLegalBERT,3.3


MRR


,Retriever,Mean Rank
0,Hybrid,4.25
1,BGE,3.70
2,BM25,3.55
3,E5,3.40
4,TF-IDF,3.30
5,InLegalBERT,2.80


NDCG@5


,Retriever,Mean Rank
0,Hybrid,4.75
1,BGE,3.60
2,BM25,3.45
3,TF-IDF,3.30
4,InLegalBERT,3.25
5,E5,2.65


## **22.7 Pairwise Wilcoxon Signed-Rank Post-hoc Comparisons**

Where the Friedman test indicates statistically significant differences among the retrieval methodologies, pairwise Wilcoxon signed-rank tests are performed to identify the retrieval methods that differ significantly.

Since multiple pairwise comparisons are conducted, the resulting p-values are adjusted using the Holm correction to control the family-wise error rate.

The post-hoc analysis is performed only for those retrieval evaluation metrics for which the Friedman test is statistically significant.

In [200]:
# ============================================================
# Pairwise Wilcoxon Post-hoc Comparisons
# ============================================================

from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

ALPHA = 0.05

def run_posthoc_wilcoxon(df, metric_name):
    """
    Pairwise Wilcoxon signed-rank tests
    with Holm correction.
    """

    methods = df.columns[1:]

    comparisons = []

    raw_p_values = []

    pairs = []

    for method1, method2 in combinations(methods, 2):

        statistic, p = wilcoxon(
            df[method1],
            df[method2],
            zero_method="wilcox",
            alternative="two-sided"
        )

        pairs.append((method1, method2))
        raw_p_values.append(p)

    # Holm correction
    reject, adjusted_p, _, _ = multipletests(
        raw_p_values,
        alpha=ALPHA,
        method="holm"
    )

    for i, (method1, method2) in enumerate(pairs):

        comparisons.append({

            "Metric": metric_name,

            "Retriever 1": method1,

            "Retriever 2": method2,

            "Raw p-value": raw_p_values[i],

            "Holm p-value": adjusted_p[i],

            "Significant": "Yes" if reject[i] else "No"

        })

    return pd.DataFrame(comparisons)

In [201]:
assert callable(run_posthoc_wilcoxon)

print("Post-hoc Wilcoxon function created successfully.")

Post-hoc Wilcoxon function created successfully.


In [202]:
# ============================================================
# Precision@5 Post-hoc Comparisons
# ============================================================

precision_posthoc = run_posthoc_wilcoxon(
    friedman_precision,
    "Precision@5"
)

display(
    precision_posthoc.round(4)
)

,Metric,Retriever 1,Retriever 2,Raw p-value,Holm p-value,Significant
0,Precision@5,TF-IDF,BM25,0.8594,1.0000,No
1,Precision@5,TF-IDF,E5,0.5312,1.0000,No
2,Precision@5,TF-IDF,BGE,0.9375,1.0000,No
3,Precision@5,TF-IDF,InLegalBERT,0.2656,1.0000,No
4,Precision@5,TF-IDF,Hybrid,0.6328,1.0000,No
5,Precision@5,BM25,E5,0.1562,1.0000,No
6,Precision@5,BM25,BGE,0.8750,1.0000,No
7,Precision@5,BM25,InLegalBERT,0.0312,0.4688,No
8,Precision@5,BM25,Hybrid,1.0000,1.0000,No
9,Precision@5,E5,BGE,0.5938,1.0000,No


In [203]:
# ============================================================
# MRR Post-hoc Comparisons
# ============================================================

mrr_posthoc = run_posthoc_wilcoxon(
    friedman_mrr,
    "MRR"
)

display(
    mrr_posthoc.round(4)
)

,Metric,Retriever 1,Retriever 2,Raw p-value,Holm p-value,Significant
0,MRR,TF-IDF,BM25,1.0000,1.0,No
1,MRR,TF-IDF,E5,1.0000,1.0,No
2,MRR,TF-IDF,BGE,1.0000,1.0,No
3,MRR,TF-IDF,InLegalBERT,0.4688,1.0,No
4,MRR,TF-IDF,Hybrid,0.2500,1.0,No
5,MRR,BM25,E5,1.0000,1.0,No
6,MRR,BM25,BGE,1.0000,1.0,No
7,MRR,BM25,InLegalBERT,0.4375,1.0,No
8,MRR,BM25,Hybrid,0.3125,1.0,No
9,MRR,E5,BGE,0.8750,1.0,No


## **22.8 Interpretation**

The Friedman test identified statistically significant differences among the six evaluated retrieval methodologies for **Precision@5** and **Mean Reciprocal Rank (MRR)**. However, no statistically significant differences were observed for **Hit@5** or **nDCG@5**.

Because the Friedman test was significant for Precision@5 and MRR, pairwise Wilcoxon signed-rank post-hoc comparisons were conducted with Holm adjustment to control the family-wise error rate.

Following Holm correction, none of the pairwise comparisons remained statistically significant for either Precision@5 or MRR.

These findings indicate that although the retrieval methodologies exhibit overall differences in performance, the pilot benchmark does not provide sufficient statistical evidence to conclude that any individual retrieval methodology consistently outperforms another after controlling for multiple comparisons.

The Friedman mean ranks nevertheless provide a descriptive ordering of the retrieval methodologies. For Precision@5, E5 achieved the highest mean rank, followed closely by the Hybrid retriever, whereas InLegalBERT consistently achieved the lowest mean ranks across the evaluated metrics.

Given that the pilot evaluation comprised only ten legal research questions, these findings should be regarded as preliminary. The formal evaluation on the complete benchmark of fifty legal research questions will provide substantially greater statistical power and form the basis of the final conclusions presented in the dissertation.

## **22.9 Conclusion for RQ2**

The pilot evaluation demonstrated statistically significant overall differences among the evaluated retrieval methodologies for Precision@5 and Mean Reciprocal Rank, but not for Hit@5 or nDCG@5.

Although the Friedman tests indicated overall differences, the subsequent pairwise Wilcoxon signed-rank post-hoc comparisons with Holm adjustment did not identify statistically significant differences between any individual pair of retrieval methodologies.

Accordingly, the pilot study provides preliminary evidence that retrieval performance varies across retrieval methodologies but does not provide sufficient statistical evidence to identify a single retrieval methodology as significantly superior after correcting for multiple comparisons.

These results are consistent with the limited sample size of the pilot benchmark and reinforce the importance of the planned evaluation using the complete benchmark of fifty legal research questions.